# VN QUANT RESEARCH ENGINE v2 + RSI/DG

**Baseline:** `Tool_CK_Claude_v1_baseline.ipynb` (untouched)  
**Spec:** MASTER SPECIFICATION — VN QUANT RESEARCH ENGINE v2 + RSI/DG  
**Philosophy:** Indicator → Feature → Historical Observation → Forward Outcome → Statistical Validation → Factor → Signal → Probability → Risk-adjusted Decision

---

### Implementation Status
| Phase | Description | Status |
|-------|-------------|--------|
| 1 | Correctness Audit | ✅ Done (`reports/correctness_report.md`) |
| 2 | Data Architecture | ✅ In this notebook |
| 3 | Research Dataset | 🔄 Skeleton + working builder |
| 4 | Triple Barrier + Empirical | 🔄 Skeleton |
| 5–12 | Factors → Walk-forward → Ablation | 📋 Scaffolded |

**Rule:** Never add a feature without a hypothesis and a plan to measure incremental OOS alpha.

In [1]:
# ================================
# SETUP ENVIRONMENT FOR GOOGLE COLAB
# ================================

!pip install -q --upgrade --force-reinstall \
    "numpy==2.2.6" \
    "pandas==2.3.3" \
    "pandas-ta==0.4.71b0" \
    "vnstock"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.2/110.2 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.3/80.3 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.0/125.0 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 17.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 39.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.9/284.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import numpy as np
import pandas as pd
import pandas_ta as ta
import vnstock

print("NumPy       :", np.__version__)
print("Pandas      :", pd.__version__)
print("Pandas-TA   :", ta.version)
print("vnstock     :", getattr(vnstock, "__version__", "unknown"))

NumPy       : 2.1.3
Pandas      : 2.3.3
Pandas-TA   : 0.4.71b0
vnstock     : unknown


In [3]:
!pip install scikit-learn -q

In [4]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — ENVIRONMENT / IMPORTS                                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import os
import sys
import warnings
import time
import json
import hashlib
from pathlib import Path
from datetime import datetime, timedelta, date
from dataclasses import dataclass, field, asdict
from typing import Optional, List, Dict, Tuple, Any, Union
from collections import defaultdict

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

warnings.filterwarnings("ignore")

# Versioning (Spec §73)
FEATURE_VERSION = "2.0.0"
MODEL_VERSION   = "none_yet"
CONFIG_VERSION  = "research_v1"
ENGINE_VERSION  = "2.0.0-phase2"

print(f"VN QUANT RESEARCH ENGINE v{ENGINE_VERSION}")
print(f"FEATURE={FEATURE_VERSION} | CONFIG={CONFIG_VERSION} | MODEL={MODEL_VERSION}")

VN QUANT RESEARCH ENGINE v2.0.0-phase2
FEATURE=2.0.0 | CONFIG=research_v1 | MODEL=none_yet


In [5]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — SECRETS / CONFIGURATION                                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# NEVER hard-code API keys. Use environment variable or Colab Secrets.
#
# Colab example:
#   from google.colab import userdata
#   os.environ["VNSTOCK_API_KEY"] = userdata.get("VNSTOCK_API_KEY")
#
# Local:
#   export VNSTOCK_API_KEY=your_key_here

def get_api_key() -> Optional[str]:
    key = os.environ.get("VNSTOCK_API_KEY")
    if key is None:
        # Fallback for Colab Secrets (safe)
        try:
            from google.colab import userdata
            key = userdata.get("VNSTOCK_API_KEY")
        except Exception:
            pass
    if key is None:
        print("⚠️  VNSTOCK_API_KEY not found. Set via env or Colab Secrets.")
        print("   Community mode (rate-limited) will be used if vnstock allows.")
    return key


def register_vnstock_if_needed():
    """Register only if key is present. Never print the key."""
    key = get_api_key()
    if key:
        try:
            from vnstock import register_user
            register_user(key)
            print("✓ vnstock API key registered (value hidden)")
        except Exception as e:
            print(f"⚠️  register_user failed: {type(e).__name__}")
    else:
        print("→ Running without explicit API key registration")


# Paths (Spec §75)
ROOT = Path("/home/workdir/artifacts") if Path("/home/workdir/artifacts").exists() else Path(".")
DATA_RAW       = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
DATA_FEATURES  = ROOT / "data" / "features"
DATA_LABELS    = ROOT / "data" / "labels"
MODELS_DIR     = ROOT / "models"
RESULTS_DIR    = ROOT / "results"
REPORTS_DIR    = ROOT / "reports"

for p in [DATA_RAW, DATA_PROCESSED, DATA_FEATURES, DATA_LABELS, MODELS_DIR, RESULTS_DIR, REPORTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

print("Data directories ready.")

Data directories ready.


In [6]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 3 — RESEARCH CONFIG                                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝

@dataclass
class ResearchConfig:
    """Central configuration for the research engine. All magic numbers live here."""

    # ── Versions ────────────────────────────────────────────────────────────
    feature_version: str = FEATURE_VERSION
    model_version:   str = MODEL_VERSION
    config_version:  str = CONFIG_VERSION

    # ── Universe / Data ─────────────────────────────────────────────────────
    sources:            List[str] = field(default_factory=lambda: ["VCI", "KBS"])
    delay_sec:          float = 1.5
    min_history_bars:   int   = 60          # minimum bars to compute features
    lookback_features:  int   = 300         # bars needed for indicators
    lookback_long:      int   = 600

    # ── Liquidity (percentile-based preferred; absolute as fallback) ───────
    min_avg_vol_20d:    int   = 50_000
    min_avg_val_20d:    float = 10e6        # 10 tỷ VND
    use_liquidity_percentile: bool = True
    min_adv_percentile: float = 20.0       # bottom 20% liquidity filtered

    # ── Triple Barrier defaults (will be validated empirically) ─────────────
    tp_pct:             float = 0.06       # +6%
    sl_pct:             float = 0.03       # -3%
    max_holding_sessions: int = 10

    # ── Sample size / confidence ────────────────────────────────────────────
    min_sample_low:     int   = 30
    min_sample_medium:  int   = 100

    # ── Risk ────────────────────────────────────────────────────────────────
    capital:            float = 100_000_000
    risk_per_trade:     float = 0.0075     # 0.75% NAV (conservative before calibration)
    max_position_pct:   float = 0.15
    max_sector_pct:     float = 0.35

    # ── Costs (VN realistic) ────────────────────────────────────────────────
    fee_buy:            float = 0.0015     # 0.15%
    fee_sell:           float = 0.0015
    tax_sell:           float = 0.001      # 0.1% TNCN
    slippage_base_bps:  float = 5.0        # 5 bps base

    # ── Execution model ─────────────────────────────────────────────────────
    # Spec default: Signal @ T close → Entry @ T+1 open
    execution_model:    str   = "next_open"  # next_open | next_vwap | next_close | same_close

    # ── Indicator params (kept from legacy for feature parity) ──────────────
    ichi_tenkan:        int   = 9
    ichi_kijun:         int   = 26
    ichi_senkou_b:      int   = 52
    dmi_period:         int   = 14
    aroon_period:       int   = 25
    bb_period:          int   = 20
    bb_std:             float = 2.0
    rsi_period:         int   = 14

    # ── RSI divergence + Ichimoku Trịnh Phát / Dao Găm ─────────────────────
    # Pivot parameters: a pivot is only usable after `right` bars confirm it.
    rsi_div_left:       int   = 3
    rsi_div_right:      int   = 3
    rsi_div_max_bars:   int   = 60
    rsi_div_active_bars:int   = 8
    rsi_div_min_rsi:    float = 2.0

    # Dao Găm levels: midpoint(Highest High, Lowest Low) over 65 / 129 bars.
    dg_near_pct:        float = 0.015
    dg_break_pct:       float = 0.010
    dg_history_days:    int   = 5200
    dg_require_history_months: int = 129

    # ── Random seed for reproducibility ────────────────────────────────────
    random_seed:        int   = 42


CFG = ResearchConfig()
print("ResearchConfig loaded.")
print(f"  Execution model : {CFG.execution_model}")
print(f"  Triple-barrier  : TP={CFG.tp_pct:.0%} / SL={CFG.sl_pct:.0%} / H={CFG.max_holding_sessions} sessions")

ResearchConfig loaded.
  Execution model : next_open
  Triple-barrier  : TP=6% / SL=3% / H=10 sessions


In [7]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — TRADING CALENDAR                                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Spec §6: All horizons must mean N trading sessions, not calendar days.
# Do NOT approximate with days_held += 1 on a calendar index.

class TradingCalendar:
    """
    Vietnamese trading calendar helper.

    Design choices:
    - Prefer deriving sessions from actual OHLCV data (most reliable).
    - Optional static holiday list can be injected later.
    - Never invent synthetic sessions.
    """

    def __init__(self, sessions: Optional[pd.DatetimeIndex] = None):
        self._sessions = None
        if sessions is not None:
            self.set_sessions(sessions)

    def set_sessions(self, sessions: Union[pd.DatetimeIndex, pd.Series, list]):
        s = pd.DatetimeIndex(pd.to_datetime(sessions)).normalize().unique().sort_values()
        self._sessions = s

    def from_ohlcv(self, df: pd.DataFrame, time_col: str = "time"):
        """Derive trading sessions from real OHLCV (recommended)."""
        if time_col not in df.columns:
            raise ValueError(f"{time_col} not in DataFrame")
        self.set_sessions(df[time_col])
        return self

    @property
    def sessions(self) -> pd.DatetimeIndex:
        if self._sessions is None:
            raise RuntimeError("TradingCalendar has no sessions. Call from_ohlcv() or set_sessions().")
        return self._sessions

    def is_trading_day(self, d) -> bool:
        d = pd.Timestamp(d).normalize()
        return d in self.sessions

    def next_session(self, d, n: int = 1) -> Optional[pd.Timestamp]:
        """Return the n-th next trading session after d (exclusive)."""
        d = pd.Timestamp(d).normalize()
        idx = self.sessions.searchsorted(d, side="right")
        target = idx + (n - 1)
        if target >= len(self.sessions):
            return None
        return self.sessions[target]

    def prev_session(self, d, n: int = 1) -> Optional[pd.Timestamp]:
        d = pd.Timestamp(d).normalize()
        idx = self.sessions.searchsorted(d, side="left") - n
        if idx < 0:
            return None
        return self.sessions[idx]

    def session_offset(self, d, n: int) -> Optional[pd.Timestamp]:
        """Offset by n trading sessions (positive = future)."""
        if n >= 0:
            return self.next_session(d, n) if n > 0 else pd.Timestamp(d).normalize()
        return self.prev_session(d, -n)

    def sessions_between(self, start, end) -> pd.DatetimeIndex:
        start = pd.Timestamp(start).normalize()
        end   = pd.Timestamp(end).normalize()
        return self.sessions[(self.sessions >= start) & (self.sessions <= end)]

    def n_sessions(self, start, end) -> int:
        return len(self.sessions_between(start, end))

    def align_to_session(self, d, method: str = "ffill") -> Optional[pd.Timestamp]:
        """Snap a date to the nearest trading session."""
        d = pd.Timestamp(d).normalize()
        if d in self.sessions:
            return d
        if method == "ffill":
            return self.prev_session(d, 1)
        if method == "bfill":
            return self.next_session(d, 1)
        return None


print("TradingCalendar defined.")

TradingCalendar defined.


In [8]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — MARKET DATA STORE                                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Spec §9: fetch / clean / cache / validate / historical date-range retrieval.
# Spec §7: Do NOT blindly forward-fill OHLC across non-trading days.
# Prefer actual trading sessions only.

class MarketDataStore:
    """
    Central data access layer.

    Responsibilities:
    - Fetch OHLCV from vnstock (v3/v4 fallback)
    - Clean without inventing synthetic candles
    - Cache to parquet for research reuse
    - Retrieve by symbol + start_date + end_date
    - Provide benchmark (VNINDEX) and universe helpers
    """

    def __init__(self, cfg: ResearchConfig = CFG):
        self.cfg = cfg
        self._memory_cache: Dict[str, pd.DataFrame] = {}
        self.calendar = TradingCalendar()

    # ── Public API ──────────────────────────────────────────────────────────

    def get(self, symbol: str, start: str, end: str,
            use_cache: bool = True, force_refresh: bool = False) -> Optional[pd.DataFrame]:
        """
        Retrieve OHLCV for [start, end] inclusive (trading sessions only).

        Returns DataFrame with columns:
            time, open, high, low, close, volume
        Sorted ascending, no synthetic rows.
        """
        start_ts = pd.Timestamp(start).normalize()
        end_ts   = pd.Timestamp(end).normalize()

        cache_key = f"{symbol}_{start_ts.date()}_{end_ts.date()}"
        parquet_path = DATA_RAW / f"{symbol}.parquet"

        # 1. Memory cache
        if use_cache and not force_refresh and cache_key in self._memory_cache:
            return self._memory_cache[cache_key].copy()

        # 2. Disk cache (full history) then slice
        df_full = None
        if use_cache and not force_refresh and parquet_path.exists():
            try:
                df_full = pd.read_parquet(parquet_path)
                df_full["time"] = pd.to_datetime(df_full["time"])
            except Exception:
                df_full = None

        # 3. Fetch if needed
        if df_full is None or force_refresh:
            # Fetch with buffer so we can support long lookbacks later
            fetch_start = (start_ts - timedelta(days=900)).strftime("%Y-%m-%d")
            fetch_end   = (end_ts + timedelta(days=5)).strftime("%Y-%m-%d")
            df_full = self._fetch_raw(symbol, fetch_start, fetch_end)
            if df_full is not None and len(df_full) > 0:
                df_full = self._clean(df_full)
                if df_full is not None:
                    try:
                        df_full.to_parquet(parquet_path, index=False)
                    except Exception as e:
                        print(f"  ⚠️  Could not write parquet for {symbol}: {e}")

        if df_full is None or len(df_full) == 0:
            return None

        # 4. Slice to requested window (real sessions only)
        mask = (df_full["time"] >= start_ts) & (df_full["time"] <= end_ts)
        df = df_full.loc[mask].reset_index(drop=True)

        if len(df) == 0:
            return None

        self._memory_cache[cache_key] = df
        return df.copy()

    def get_as_of(self, symbol: str, as_of: str, lookback_bars: int = 300) -> Optional[pd.DataFrame]:
        """
        Return the last `lookback_bars` trading sessions ending on or before as_of.
        Critical for research: features at T must use only data ≤ T.
        """
        as_of_ts = pd.Timestamp(as_of).normalize()
        # Fetch a generous window then cut
        start = (as_of_ts - timedelta(days=int(lookback_bars * 2.2))).strftime("%Y-%m-%d")
        end   = as_of_ts.strftime("%Y-%m-%d")
        df = self.get(symbol, start, end)
        if df is None:
            return None
        df = df[df["time"] <= as_of_ts].tail(lookback_bars).reset_index(drop=True)
        return df if len(df) >= self.cfg.min_history_bars else None

    def get_benchmark(self, start: str, end: str) -> Optional[pd.DataFrame]:
        return self.get("VNINDEX", start, end)

    def update_calendar_from(self, symbol: str = "VNINDEX", start: str = "2018-01-01", end: str = None):
        if end is None:
            end = datetime.today().strftime("%Y-%m-%d")
        df = self.get(symbol, start, end)
        if df is not None:
            self.calendar.from_ohlcv(df)
            print(f"  ✓ Calendar loaded: {len(self.calendar.sessions)} sessions "
                  f"({self.calendar.sessions[0].date()} → {self.calendar.sessions[-1].date()})")
        return self.calendar

    # ── Internal ────────────────────────────────────────────────────────────


    def _fetch_raw(self, symbol: str, start: str, end: str):
        """
        Prefer modern vnstock.api.Quote to avoid deprecation warnings.
        Fallback chain: api.Quote → legacy Quote → legacy Vnstock().stock
        """
        # 1) New official API
        try:
            from vnstock.api.quote import Quote as ApiQuote
            for src in self.cfg.sources:
                try:
                    q = ApiQuote(symbol=symbol, source=src)
                    df = q.history(start=start, end=end, interval="1D")
                    if df is not None and len(df) > 0:
                        return df
                except Exception:
                    continue
                time.sleep(0.3)
        except ImportError:
            pass

        # 2) Legacy Quote (still works on many installs)
        for src in self.cfg.sources:
            try:
                df = self._v4(symbol, start, end, src)
                if df is not None and len(df) > 0:
                    return df
            except Exception:
                continue
            time.sleep(self.cfg.delay_sec * 0.5)

        # 3) Very legacy Vnstock().stock — last resort (triggers deprecation)
        for src in self.cfg.sources:
            try:
                df = self._v3(symbol, start, end, src)
                if df is not None and len(df) > 0:
                    return df
            except Exception:
                continue
            time.sleep(self.cfg.delay_sec)

        return None


    def _v4(self, sym, start, end, src):
        from vnstock import Quote
        q = Quote(symbol=sym, source=src)
        try:
            return q.history(start=start, end=end, interval="1D")
        except TypeError:
            return q.history(start_date=start, end_date=end, interval="1D")

    def _v3(self, sym, start, end, src):
        from vnstock import Vnstock
        stk = Vnstock().stock(symbol=sym, source=src)
        try:
            return stk.quote.history(start=start, end=end, interval="1D")
        except TypeError:
            return stk.quote.history(start_date=start, end_date=end, interval="1D")

    def _clean(self, df: pd.DataFrame) -> Optional[pd.DataFrame]:
        """
        Clean without inventing synthetic trading days.

        Rules (documented):
        1. Normalize column names.
        2. Keep only rows that have a valid close > 0.
        3. Do NOT ffill across calendar gaps — gaps are non-sessions.
        4. Drop exact duplicate dates (keep last).
        5. Basic OHLC integrity checks.
        6. Sort by time ascending.
        """
        df = df.copy()
        df.columns = [str(c).lower().strip() for c in df.columns]
        df = df.rename(columns={
            "tradingdate": "time", "date": "time",
            "o": "open", "h": "high", "l": "low", "c": "close", "v": "volume",
        })
        required = {"time", "open", "high", "low", "close", "volume"}
        if not required.issubset(set(df.columns)):
            return None

        for col in ["open", "high", "low", "close", "volume"]:
            df[col] = pd.to_numeric(df[col], errors="coerce")

        df["time"] = pd.to_datetime(df["time"]).dt.normalize()

        # Drop invalid prices
        df = df[df["close"] > 0].copy()

        # OHLC integrity (soft): fix obvious swaps, drop irreparable
        bad = (df["high"] < df[["open", "close"]].max(axis=1)) | \
              (df["low"]  > df[["open", "close"]].min(axis=1))
        if bad.any():
            # Conservative: drop bad rows rather than invent values
            df = df[~bad]

        # Deduplicate dates
        df = df.drop_duplicates(subset=["time"], keep="last")

        # Sort & reset
        df = df.sort_values("time").reset_index(drop=True)

        # Volume: leave NaN as NaN; do not force 0 (0 is meaningful only if real)
        # Callers that need volume stats should dropna or fill carefully.

        cols = ["time", "open", "high", "low", "close", "volume"]
        return df[cols]

print("MarketDataStore defined.")

MarketDataStore defined.


In [9]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — LOOK-AHEAD UNIT TEST (Spec §71)                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def test_no_lookahead(feature_fn, df: pd.DataFrame, t_idx: int, tol: float = 1e-9) -> bool:
    """
    Spec §71 Look-ahead test:

    1. Compute features using data up to t_idx (inclusive).
    2. Mutate all future rows (t_idx+1 onward) with garbage.
    3. Recompute features up to t_idx.
    4. Assert features are identical.

    feature_fn(df_slice) -> dict or pd.Series of features at the last bar.
    """
    if t_idx >= len(df) - 1:
        print("  ⚠️  t_idx too close to end; need future rows to mutate")
        return False

    df_clean = df.iloc[: t_idx + 1].copy()
    feat_clean = feature_fn(df_clean)

    df_mut = df.copy()
    # Mutate future OHLC + volume with extreme values
    future_idx = df_mut.index[t_idx + 1 :]
    for col in ["open", "high", "low", "close"]:
        if col in df_mut.columns:
            df_mut.loc[future_idx, col] = 999999.0
    if "volume" in df_mut.columns:
        df_mut.loc[future_idx, "volume"] = 999999999

    df_mut_slice = df_mut.iloc[: t_idx + 1].copy()
    feat_mut = feature_fn(df_mut_slice)

    # Compare
    if isinstance(feat_clean, dict):
        for k in feat_clean:
            v1, v2 = feat_clean[k], feat_mut.get(k)
            if v1 is None and v2 is None:
                continue
            if isinstance(v1, (int, float, np.floating, np.integer)) and isinstance(v2, (int, float, np.floating, np.integer)):
                if not np.isclose(float(v1), float(v2), atol=tol, equal_nan=True):
                    print(f"  ❌ LOOK-AHEAD BUG on feature '{k}': {v1} vs {v2}")
                    return False
            elif v1 != v2:
                print(f"  ❌ LOOK-AHEAD BUG on feature '{k}': {v1} vs {v2}")
                return False
    else:
        # assume Series / array
        if not np.allclose(feat_clean, feat_mut, atol=tol, equal_nan=True):
            print("  ❌ LOOK-AHEAD BUG detected in array features")
            return False

    print("  ✓ No look-ahead detected for tested features at t_idx")
    return True


print("Look-ahead test helper defined.")

Look-ahead test helper defined.


In [10]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — CORE FEATURE CALCULATORS (timestamp-contracted)              ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Every function here obeys:
#   feature_at(T)  ==  computed using information available through T only.
#
# We deliberately keep calculations pure (no external TA libs) for auditability.

def _safe_last(s, default=np.nan):
    if s is None or len(s) == 0:
        return default
    v = s.iloc[-1] if hasattr(s, "iloc") else s[-1]
    return default if pd.isna(v) else float(v)


def compute_basic_features(df: pd.DataFrame, cfg: ResearchConfig = CFG) -> dict:
    """
    Minimal, fast feature set for research dataset.
    Input df must end at the decision bar T (no future rows).
    """
    if df is None or len(df) < cfg.min_history_bars:
        return {}

    c = df["close"].astype(float)
    h = df["high"].astype(float)
    l = df["low"].astype(float)
    v = df["volume"].astype(float).fillna(0)
    o = df["open"].astype(float)

    out = {
        "close": _safe_last(c),
        "open":  _safe_last(o),
        "high":  _safe_last(h),
        "low":   _safe_last(l),
        "volume": int(_safe_last(v, 0)),
    }

    # Returns
    for n in (1, 3, 5, 10, 20):
        if len(c) > n:
            out[f"ret_{n}d"] = float(c.iloc[-1] / c.iloc[-n-1] - 1)
        else:
            out[f"ret_{n}d"] = np.nan

    # Volume ratio
    vol_ma = v.rolling(20).mean()
    out["volume_ratio"] = float(v.iloc[-1] / vol_ma.iloc[-1]) if vol_ma.iloc[-1] > 0 else np.nan
    out["avg_vol_20d"]  = float(vol_ma.iloc[-1]) if not pd.isna(vol_ma.iloc[-1]) else np.nan
    out["avg_val_20d"]  = float(vol_ma.iloc[-1] * c.rolling(20).mean().iloc[-1]) if not pd.isna(vol_ma.iloc[-1]) else np.nan

    # ATR%
    tr = pd.concat([h - l, (h - c.shift()).abs(), (l - c.shift()).abs()], axis=1).max(axis=1)
    atr = tr.rolling(14).mean()
    out["atr_pct"] = float(atr.iloc[-1] / c.iloc[-1] * 100) if c.iloc[-1] > 0 else np.nan
    out["atr_abs"] = float(atr.iloc[-1])

    # RSI
    delta = c.diff()
    gain = delta.clip(lower=0).rolling(cfg.rsi_period).mean()
    loss = (-delta.clip(upper=0)).rolling(cfg.rsi_period).mean()
    rs = gain / loss.replace(0, np.nan)
    out["rsi"] = float((100 - 100 / (1 + rs)).iloc[-1])

    # EMA / SMA
    ema20 = c.ewm(span=20, adjust=False).mean()
    ema50 = c.ewm(span=50, adjust=False).mean()
    out["ema20"] = float(ema20.iloc[-1])
    out["ema50"] = float(ema50.iloc[-1])
    out["ema_pct"] = float((c.iloc[-1] - ema20.iloc[-1]) / ema20.iloc[-1] * 100)

    if len(c) >= 50:
        sma50 = c.rolling(50).mean()
        out["sma50"] = float(sma50.iloc[-1])
        out["above_sma50"] = int(c.iloc[-1] > sma50.iloc[-1])
    else:
        out["sma50"] = np.nan
        out["above_sma50"] = 0

    if len(c) >= 200:
        sma200 = c.rolling(200).mean()
        out["sma200"] = float(sma200.iloc[-1])
        out["above_sma200"] = int(c.iloc[-1] > sma200.iloc[-1])
    else:
        out["sma200"] = np.nan
        out["above_sma200"] = 0

    # MACD hist (normalized)
    ema12 = c.ewm(span=12, adjust=False).mean()
    ema26 = c.ewm(span=26, adjust=False).mean()
    macd_line = ema12 - ema26
    signal = macd_line.ewm(span=9, adjust=False).mean()
    hist = macd_line - signal
    out["macd_hist_pct"] = float(hist.iloc[-1] / c.iloc[-1] * 100)

    # Bollinger
    sma20 = c.rolling(cfg.bb_period).mean()
    std20 = c.rolling(cfg.bb_period).std()
    upper = sma20 + cfg.bb_std * std20
    lower = sma20 - cfg.bb_std * std20
    bw = (upper - lower) / sma20
    out["bb_width"] = float(bw.iloc[-1])
    out["bb_pctb"]  = float((c.iloc[-1] - lower.iloc[-1]) / (upper.iloc[-1] - lower.iloc[-1] + 1e-12))

    # CMF
    mfv = ((c - l) - (h - c)) / (h - l + 1e-9) * v
    out["cmf"] = float((mfv.rolling(21).sum() / v.rolling(21).sum().replace(0, np.nan)).iloc[-1])

    # OBV slope proxy
    obv = (np.sign(c.diff().fillna(0)) * v).cumsum()
    if len(obv) >= 20:
        out["obv_rising"] = int(obv.iloc[-1] > obv.iloc[-20])
    else:
        out["obv_rising"] = 0

    # Stochastic
    l14 = l.rolling(14).min()
    h14 = h.rolling(14).max()
    sk = (c - l14) / (h14 - l14 + 1e-9) * 100
    out["stoch_k"] = float(sk.iloc[-1])

    return out


print("Core feature calculators defined.")

Core feature calculators defined.


In [11]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — TRIPLE BARRIER LABELER (Spec §13)                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

@dataclass
class BarrierResult:
    label: str              # TP_FIRST | SL_FIRST | TIMEOUT
    entry_price: float
    exit_price: float
    exit_idx: int           # relative to future bars (0 = first bar after entry)
    sessions_held: int
    ret: float
    mfe: float              # max favorable excursion
    mae: float              # max adverse excursion
    hit_tp: bool
    hit_sl: bool


class TripleBarrierLabeler:
    """
    Classic Lopez de Prado triple-barrier method adapted to VN market.

    Entry assumption (Spec §5):
        Signal at T close → Entry at T+1 open  (default)

    Barriers:
        Upper  = entry * (1 + tp_pct)
        Lower  = entry * (1 - sl_pct)
        Vertical = max_holding_sessions trading sessions

    Returns the barrier that is touched first (path-dependent).
    """

    def __init__(self, cfg: ResearchConfig = CFG):
        self.cfg = cfg

    def label_one(self,
                  entry_price: float,
                  future_bars: pd.DataFrame,
                  tp_pct: float = None,
                  sl_pct: float = None,
                  max_hold: int = None) -> Optional[BarrierResult]:
        """
        future_bars: DataFrame of sessions AFTER the signal bar,
                     first row = intended entry bar (T+1).
        """
        if future_bars is None or len(future_bars) == 0:
            return None

        tp_pct  = tp_pct  if tp_pct  is not None else self.cfg.tp_pct
        sl_pct  = sl_pct  if sl_pct  is not None else self.cfg.sl_pct
        max_hold = max_hold if max_hold is not None else self.cfg.max_holding_sessions

        upper = entry_price * (1 + tp_pct)
        lower = entry_price * (1 - sl_pct)

        mfe = 0.0
        mae = 0.0

        n = min(len(future_bars), max_hold)
        for i in range(n):
            row = future_bars.iloc[i]
            hi, lo, cl = float(row["high"]), float(row["low"]), float(row["close"])

            # Path-dependent MFE / MAE using intra-bar extremes
            fav = (hi - entry_price) / entry_price
            adv = (entry_price - lo) / entry_price
            mfe = max(mfe, fav)
            mae = max(mae, adv)

            hit_tp = hi >= upper
            hit_sl = lo <= lower

            if hit_tp and hit_sl:
                # Ambiguous bar: conservative → assume SL first (risk-first)
                return BarrierResult(
                    label="SL_FIRST", entry_price=entry_price, exit_price=lower,
                    exit_idx=i, sessions_held=i+1,
                    ret=(lower - entry_price)/entry_price,
                    mfe=mfe, mae=mae, hit_tp=True, hit_sl=True
                )
            if hit_sl:
                return BarrierResult(
                    label="SL_FIRST", entry_price=entry_price, exit_price=lower,
                    exit_idx=i, sessions_held=i+1,
                    ret=(lower - entry_price)/entry_price,
                    mfe=mfe, mae=mae, hit_tp=False, hit_sl=True
                )
            if hit_tp:
                return BarrierResult(
                    label="TP_FIRST", entry_price=entry_price, exit_price=upper,
                    exit_idx=i, sessions_held=i+1,
                    ret=(upper - entry_price)/entry_price,
                    mfe=mfe, mae=mae, hit_tp=True, hit_sl=False
                )

        # Timeout
        last_close = float(future_bars.iloc[n-1]["close"])
        return BarrierResult(
            label="TIMEOUT", entry_price=entry_price, exit_price=last_close,
            exit_idx=n-1, sessions_held=n,
            ret=(last_close - entry_price)/entry_price,
            mfe=mfe, mae=mae, hit_tp=False, hit_sl=False
        )


print("TripleBarrierLabeler defined.")

TripleBarrierLabeler defined.


In [12]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 9 — RESEARCH DATASET BUILDER (Spec §10–12)                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class ResearchDatasetBuilder:
    """
    Builds historical observation tables:

        for each (symbol, date):
            features using data ≤ date
            labels using data > date

    Strict timestamp contract enforced.
    """

    def __init__(self, store: MarketDataStore, cfg: ResearchConfig = CFG):
        self.store = store
        self.cfg = cfg
        self.labeler = TripleBarrierLabeler(cfg)

    def build_for_symbol(self,
                         symbol: str,
                         start: str,
                         end: str,
                         stride: int = 5,
                         feature_fn=compute_basic_features) -> pd.DataFrame:
        """
        stride: sample every N trading sessions (speed vs density trade-off).
        """
        # Need extra history before start for feature warm-up + future after end for labels
        fetch_start = (pd.Timestamp(start) - timedelta(days=int(self.cfg.lookback_features * 2.5))).strftime("%Y-%m-%d")
        fetch_end   = (pd.Timestamp(end) + timedelta(days=int(self.cfg.max_holding_sessions * 3))).strftime("%Y-%m-%d")

        df = self.store.get(symbol, fetch_start, fetch_end)
        if df is None or len(df) < self.cfg.min_history_bars + self.cfg.max_holding_sessions + 5:
            return pd.DataFrame()

        df = df.sort_values("time").reset_index(drop=True)
        times = df["time"]

        # Candidate decision dates inside [start, end]
        mask = (times >= pd.Timestamp(start)) & (times <= pd.Timestamp(end))
        decision_idx = np.where(mask)[0]
        decision_idx = decision_idx[::stride]  # stride sampling

        rows = []
        for i in decision_idx:
            # Features: only data up to i (inclusive)
            window = df.iloc[max(0, i - self.cfg.lookback_features + 1): i + 1].copy()
            if len(window) < self.cfg.min_history_bars:
                continue

            feats = feature_fn(window, self.cfg)
            if not feats:
                continue

            signal_date = times.iloc[i]

            # Entry = next bar open (T+1)
            if i + 1 >= len(df):
                continue
            entry_bar = df.iloc[i + 1]
            entry_price = float(entry_bar["open"])
            entry_date  = times.iloc[i + 1]

            # Future bars for labels (from entry bar onward)
            future = df.iloc[i + 1:].reset_index(drop=True)

            # Simple forward returns (close-to-close from entry close for simplicity)
            # More precise path-dependent stats come from TripleBarrier
            labels = {}
            for horizon in (3, 5, 10, 20):
                if len(future) > horizon:
                    # return from entry open to close of horizon bar
                    labels[f"future_return_{horizon}d"] = float(future.iloc[horizon]["close"] / entry_price - 1)
                    # max return / max drawdown inside horizon
                    window_f = future.iloc[: horizon + 1]
                    labels[f"future_max_return_{horizon}d"] = float(window_f["high"].max() / entry_price - 1)
                    labels[f"future_max_drawdown_{horizon}d"] = float(window_f["low"].min() / entry_price - 1)
                else:
                    labels[f"future_return_{horizon}d"] = np.nan
                    labels[f"future_max_return_{horizon}d"] = np.nan
                    labels[f"future_max_drawdown_{horizon}d"] = np.nan

            # Triple barrier
            br = self.labeler.label_one(entry_price, future)
            if br is not None:
                labels["tb_label"] = br.label
                labels["tb_ret"] = br.ret
                labels["tb_mfe"] = br.mfe
                labels["tb_mae"] = br.mae
                labels["tb_sessions_held"] = br.sessions_held
            else:
                labels["tb_label"] = None
                labels["tb_ret"] = np.nan
                labels["tb_mfe"] = np.nan
                labels["tb_mae"] = np.nan
                labels["tb_sessions_held"] = np.nan

            row = {
                "date": signal_date,
                "symbol": symbol,
                "entry_date": entry_date,
                "entry_price": entry_price,
                **feats,
                **labels,
            }
            rows.append(row)

        return pd.DataFrame(rows)

    def build_universe(self,
                       symbols: List[str],
                       start: str,
                       end: str,
                       stride: int = 5,
                       save: bool = True) -> pd.DataFrame:
        all_dfs = []
        for i, sym in enumerate(symbols):
            print(f"  [{i+1}/{len(symbols)}] {sym}...", end=" ")
            try:
                dfi = self.build_for_symbol(sym, start, end, stride=stride)
                print(f"{len(dfi)} rows")
                if len(dfi):
                    all_dfs.append(dfi)
            except Exception as e:
                print(f"FAIL {type(e).__name__}")
            time.sleep(self.cfg.delay_sec)

        if not all_dfs:
            return pd.DataFrame()

        out = pd.concat(all_dfs, ignore_index=True)
        out["feature_version"] = self.cfg.feature_version

        if save:
            path = DATA_FEATURES / f"features_{start}_{end}_s{stride}.parquet"
            out.to_parquet(path, index=False)
            print(f"\n✓ Saved {len(out)} rows → {path}")

        return out


print("ResearchDatasetBuilder defined.")

ResearchDatasetBuilder defined.


In [13]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 10 — EMPIRICAL PROBABILITY ENGINE (Spec §15–17)                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class EmpiricalProbabilityEngine:
    """
    Replaces heuristic wr_base + additive deltas.

    Given a historical research dataset, compute conditional frequencies:
        P(TP_FIRST | setup conditions)
        P(SL_FIRST | ...)
        P(TIMEOUT  | ...)

    Enforces minimum sample size (Spec §17).
    """

    def __init__(self, cfg: ResearchConfig = CFG):
        self.cfg = cfg

    def summarize(self, df: pd.DataFrame, condition_mask: pd.Series = None,
                  label_col: str = "tb_label") -> dict:
        if condition_mask is not None:
            sub = df.loc[condition_mask].copy()
        else:
            sub = df.copy()

        n = len(sub)
        if n == 0:
            return {"n": 0, "confidence": "NONE", "P_TP": np.nan, "P_SL": np.nan, "P_TIMEOUT": np.nan}

        vc = sub[label_col].value_counts(normalize=True)
        p_tp = float(vc.get("TP_FIRST", 0.0))
        p_sl = float(vc.get("SL_FIRST", 0.0))
        p_to = float(vc.get("TIMEOUT", 0.0))

        if n < self.cfg.min_sample_low:
            conf = "LOW"
        elif n < self.cfg.min_sample_medium:
            conf = "MEDIUM"
        else:
            conf = "HIGH"

        avg_ret = float(sub["tb_ret"].mean()) if "tb_ret" in sub.columns else np.nan
        avg_mfe = float(sub["tb_mfe"].mean()) if "tb_mfe" in sub.columns else np.nan
        avg_mae = float(sub["tb_mae"].mean()) if "tb_mae" in sub.columns else np.nan

        # Simple expected value under fixed R:R derived from barriers
        # EV ≈ P_TP * tp_pct - P_SL * sl_pct + P_TIMEOUT * avg_timeout_ret
        ev = p_tp * self.cfg.tp_pct - p_sl * self.cfg.sl_pct
        if p_to > 0 and not np.isnan(avg_ret):
            # crude adjustment
            pass

        return {
            "n": n,
            "confidence": conf,
            "P_TP": round(p_tp, 4),
            "P_SL": round(p_sl, 4),
            "P_TIMEOUT": round(p_to, 4),
            "avg_ret": round(avg_ret, 4) if not np.isnan(avg_ret) else np.nan,
            "avg_mfe": round(avg_mfe, 4) if not np.isnan(avg_mfe) else np.nan,
            "avg_mae": round(avg_mae, 4) if not np.isnan(avg_mae) else np.nan,
            "EV_simple": round(ev, 4),
        }

    def by_bucket(self, df: pd.DataFrame, feature: str, quantiles: int = 5,
                  label_col: str = "tb_label") -> pd.DataFrame:
        """Simple univariate analysis: feature quantiles vs outcome."""
        tmp = df[[feature, label_col, "tb_ret"]].dropna().copy()
        if len(tmp) < self.cfg.min_sample_low:
            return pd.DataFrame()
        tmp["bucket"] = pd.qcut(tmp[feature], quantiles, duplicates="drop")
        rows = []
        for b, g in tmp.groupby("bucket"):
            s = self.summarize(g, label_col=label_col)
            s["bucket"] = str(b)
            s["feature"] = feature
            rows.append(s)
        return pd.DataFrame(rows)


print("EmpiricalProbabilityEngine defined.")

EmpiricalProbabilityEngine defined.


In [14]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 11 — LEGACY COMPATIBILITY SHIM (Spec §77)                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Keep the old QuantEngine API working so existing notebooks / habits
# continue to function while the research stack is built beside it.
#
# Full legacy code lives in Tool_CK_Claude_v1_baseline.ipynb.
# Here we only provide a thin wrapper that warns and points to research mode.

class QuantEngineLegacyWarning:
    """Placeholder so `from v2 import QuantEngine` does not break immediately."""

    def __init__(self, *args, **kwargs):
        print("⚠️  You are using the LEGACY QuantEngine interface.")
        print("   For research-grade work use QuantResearchEngine / ResearchDatasetBuilder.")
        print("   Full legacy implementation remains executable in:")
        print("   → Tool_CK_Claude_v1_baseline.ipynb")

    def run(self, *args, **kwargs):
        raise NotImplementedError(
            "Legacy scoring engine not re-implemented in v2. "
            "Open Tool_CK_Claude_v1_baseline.ipynb for the original engine, "
            "or use the research pipeline (ResearchDatasetBuilder + EmpiricalProbabilityEngine)."
        )

    def detail(self, *args, **kwargs):
        return self.run()


# Alias for transitional compatibility
QuantEngine = QuantEngineLegacyWarning

print("Legacy compatibility shim installed.")

Legacy compatibility shim installed.


In [15]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 12 — RESEARCH RUN METADATA (Spec §72)                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

@dataclass
class ResearchRunMetadata:
    run_id: str
    timestamp: str
    feature_version: str
    model_version: str
    config_version: str
    data_start: str
    data_end: str
    universe: List[str]
    n_symbols: int
    execution_model: str
    tp_pct: float
    sl_pct: float
    max_holding_sessions: int
    fee_buy: float
    fee_sell: float
    tax_sell: float
    random_seed: int
    notes: str = ""

    def save(self, path: Path = None):
        if path is None:
            path = RESULTS_DIR / f"run_{self.run_id}.json"
        with open(path, "w") as f:
            json.dump(asdict(self), f, indent=2, default=str)
        print(f"  ✓ Metadata saved → {path}")
        return path


def make_run_id() -> str:
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    return f"{ts}_{hashlib.md5(ts.encode()).hexdigest()[:6]}"


print("ResearchRunMetadata defined.")

ResearchRunMetadata defined.


In [16]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 13 — QUICK DEMO / SANITY CHECK                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# This cell demonstrates the correct research workflow on a tiny universe.
# It does NOT claim alpha. It only verifies that:
#   - data can be fetched & cleaned without synthetic candles
#   - features at T ignore future
#   - labels are computed strictly after T
#   - empirical frequencies can be calculated

def demo_research_pipeline(symbols=None, start="2024-01-01", end="2024-06-30", stride=10):
    if symbols is None:
        symbols = ["FPT", "VNM", "HPG"]  # small liquid set for demo

    print("═" * 60)
    print("  DEMO RESEARCH PIPELINE (tiny universe — for correctness only)")
    print("═" * 60)

    register_vnstock_if_needed()

    store = MarketDataStore(CFG)
    store.update_calendar_from("VNINDEX", start="2020-01-01")

    # Look-ahead test on one symbol
    print("\n[1] Look-ahead test on FPT...")
    df_fpt = store.get("FPT", "2023-01-01", "2024-12-31")
    if df_fpt is not None and len(df_fpt) > 100:
        t_idx = len(df_fpt) // 2
        ok = test_no_lookahead(
            lambda d: compute_basic_features(d, CFG),
            df_fpt, t_idx
        )
        print(f"    Result: {'PASS' if ok else 'FAIL'}")
    else:
        print("    Skipped (insufficient data)")

    # Build research dataset
    print("\n[2] Building research dataset...")
    builder = ResearchDatasetBuilder(store, CFG)
    ds = builder.build_universe(symbols, start, end, stride=stride, save=True)

    if len(ds) == 0:
        print("  ❌ Empty dataset — check API / network / symbols")
        return None

    print(f"\n  Dataset shape: {ds.shape}")
    print(f"  Columns: {list(ds.columns)[:20]}...")

    # Empirical stats overall
    print("\n[3] Empirical triple-barrier frequencies (all observations):")
    eng = EmpiricalProbabilityEngine(CFG)
    summary = eng.summarize(ds)
    for k, v in summary.items():
        print(f"    {k:18s}: {v}")

    # Simple feature analysis
    if "rsi" in ds.columns and ds["rsi"].notna().sum() > 30:
        print("\n[4] RSI quantile vs outcome (univariate — illustrative only):")
        buck = eng.by_bucket(ds, "rsi", quantiles=4)
        if len(buck):
            print(buck[["bucket", "n", "confidence", "P_TP", "P_SL", "avg_ret"]].to_string(index=False))

    # Metadata
    meta = ResearchRunMetadata(
        run_id=make_run_id(),
        timestamp=datetime.now().isoformat(),
        feature_version=CFG.feature_version,
        model_version=CFG.model_version,
        config_version=CFG.config_version,
        data_start=start,
        data_end=end,
        universe=symbols,
        n_symbols=len(symbols),
        execution_model=CFG.execution_model,
        tp_pct=CFG.tp_pct,
        sl_pct=CFG.sl_pct,
        max_holding_sessions=CFG.max_holding_sessions,
        fee_buy=CFG.fee_buy,
        fee_sell=CFG.fee_sell,
        tax_sell=CFG.tax_sell,
        random_seed=CFG.random_seed,
        notes="Demo run — not for production decisions",
    )
    meta.save()

    print("\n✓ Demo complete. Research pipeline is operational.")
    return ds


print("Demo function defined. Call demo_research_pipeline() after installing vnstock.")

Demo function defined. Call demo_research_pipeline() after installing vnstock.


In [17]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 14 — FACTOR ENGINE (Spec §18–21)                                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Group raw indicators into interpretable factors (0–100 scale).
# Do NOT invent magic points. Normalize within available evidence.
# Factors are FEATURES for later probability models, not final scores.

from typing import Dict

def _clip01(x, lo=0.0, hi=1.0):
    return float(np.clip(x, lo, hi))

def _safe(v, default=np.nan):
    try:
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return default
        return float(v)
    except Exception:
        return default


class FactorEngine:
    """
    Converts raw features into 6 core factors:
      TREND, MOMENTUM, FLOW, RELATIVE_STRENGTH, VOLATILITY, STRUCTURE

    Each factor is roughly 0–100.
    Higher = more bullish / more favorable for long bias (except VOLATILITY
    which is contextual — high vol can help or hurt depending on setup).
    """

    def __init__(self, cfg: ResearchConfig = CFG):
        self.cfg = cfg

    def trend_factor(self, f: dict) -> float:
        """
        Components (equal-ish weight, no arbitrary +8 magic):
        - price vs SMA50 / SMA200
        - EMA alignment (close > ema20 > ema50 proxy via ema_pct + above_sma50)
        - simple slope proxies via ret_20d
        """
        score = 50.0  # neutral baseline
        above50  = _safe(f.get("above_sma50"), 0)
        above200 = _safe(f.get("above_sma200"), 0)
        ema_pct  = _safe(f.get("ema_pct"), 0)
        ret20    = _safe(f.get("ret_20d"), 0)

        score += 15 * above50
        score += 15 * above200
        # mild contribution from short-term position vs EMA20
        score += 10 * _clip01((ema_pct + 5) / 10)   # -5%..+5% → 0..1
        # medium-term drift
        score += 10 * _clip01((ret20 + 0.10) / 0.20)  # -10%..+10% → 0..1

        return float(np.clip(score, 0, 100))

    def momentum_factor(self, f: dict) -> float:
        """
        RSI, MACD hist, short returns, stoch.
        IMPORTANT: RSI>70 is NOT automatically bearish here.
        We keep it continuous; regime interaction comes later.
        """
        rsi   = _safe(f.get("rsi"), 50)
        macd  = _safe(f.get("macd_hist_pct"), 0)
        ret5  = _safe(f.get("ret_5d"), 0)
        stoch = _safe(f.get("stoch_k"), 50)

        # Map RSI 0–100 → contribution centered
        rsi_c = (rsi - 50) / 50          # -1..+1
        macd_c = np.tanh(macd * 50)      # squash
        ret_c  = np.tanh(ret5 * 20)
        st_c   = (stoch - 50) / 50

        raw = 0.30*rsi_c + 0.25*macd_c + 0.25*ret_c + 0.20*st_c
        return float(np.clip(50 + 40 * raw, 0, 100))

    def flow_factor(self, f: dict) -> float:
        """
        Volume ratio + CMF + OBV direction.
        High volume alone is NOT bullish — we need direction (CMF / close pos proxy).
        """
        vol_r = _safe(f.get("volume_ratio"), 1.0)
        cmf   = _safe(f.get("cmf"), 0.0)
        obv_r = _safe(f.get("obv_rising"), 0)

        # Volume expansion is context; pair with money flow sign
        vol_c = np.tanh((vol_r - 1.0) * 1.5)     # centered at 1.0x
        cmf_c = np.clip(cmf / 0.25, -1, 1)       # typical CMF range
        obv_c = 1.0 if obv_r else -0.3

        # If volume high but CMF negative → drag the score down
        if vol_r > 1.4 and cmf < -0.05:
            penalty = -15
        else:
            penalty = 0

        raw = 0.35*cmf_c + 0.30*vol_c + 0.35*obv_c
        return float(np.clip(50 + 35 * raw + penalty, 0, 100))

    def rs_factor(self, f: dict) -> float:
        """
        Relative strength vs market (if available).
        Falls back to medium-term own return if RS missing.
        """
        rs = f.get("rs_20d")
        if rs is None or (isinstance(rs, float) and np.isnan(rs)):
            # fallback: own ret_20d as weak proxy
            ret20 = _safe(f.get("ret_20d"), 0)
            return float(np.clip(50 + 200 * ret20, 0, 100))
        # rs_20d expected roughly in [-0.3, +0.3] typical
        return float(np.clip(50 + 150 * float(rs), 0, 100))

    def volatility_factor(self, f: dict) -> float:
        """
        ATR% and BB width.
        Mid-range volatility often better for swing setups than extreme quiet or chaos.
        Returns a 'tradability' style score (not pure bullishness).
        """
        atr = _safe(f.get("atr_pct"), 2.0)
        bbw = _safe(f.get("bb_width"), 0.05)

        # Prefer ATR roughly 1.5%–4% for VN swing
        if 1.5 <= atr <= 4.0:
            atr_s = 1.0
        elif atr < 1.5:
            atr_s = atr / 1.5
        else:
            atr_s = max(0.0, 1.0 - (atr - 4.0) / 4.0)

        bb_s = _clip01(1.0 - abs(bbw - 0.06) / 0.10)
        return float(np.clip(100 * (0.6*atr_s + 0.4*bb_s), 0, 100))

    def structure_factor(self, f: dict) -> float:
        """
        Lightweight structure proxy until full HH/HL engine exists.
        Uses distance to SMA + BB position.
        """
        above50 = _safe(f.get("above_sma50"), 0)
        above200 = _safe(f.get("above_sma200"), 0)
        pctb = _safe(f.get("bb_pctb"), 0.5)
        ema_pct = _safe(f.get("ema_pct"), 0)

        # Prefer holding above key MAs and not extended to extreme upper band
        score = 40.0
        score += 20 * above50
        score += 15 * above200
        # mild preference for mid-upper band in uptrend, avoid >0.95 extension
        if pctb > 0.95:
            score -= 15
        elif 0.5 <= pctb <= 0.85:
            score += 15
        elif pctb < 0.2:
            score -= 5
        # not too extended above EMA
        if ema_pct > 12:
            score -= 10
        elif 0 <= ema_pct <= 6:
            score += 10

        return float(np.clip(score, 0, 100))

    def compute(self, f: dict) -> dict:
        factors = {
            "factor_trend":     round(self.trend_factor(f), 2),
            "factor_momentum":  round(self.momentum_factor(f), 2),
            "factor_flow":      round(self.flow_factor(f), 2),
            "factor_rs":        round(self.rs_factor(f), 2),
            "factor_volatility":round(self.volatility_factor(f), 2),
            "factor_structure": round(self.structure_factor(f), 2),
        }
        # Simple composite (equal weight) — only for diagnostics, NOT trade decision
        vals = list(factors.values())
        factors["factor_composite"] = round(float(np.mean(vals)), 2)
        return factors


print("FactorEngine defined.")


FactorEngine defined.


In [18]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 15 — EXPANDED FEATURES + VSA STATE (no hard scores)               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Spec §22: Keep VSA state labels, but DO NOT assign 8.5 / 1.5 magic scores.
# Empirical forward outcomes will decide whether each state has alpha.

def compute_vsa_state(df: pd.DataFrame, lookback: int = 20) -> str:
    """
    Vectorized single-bar VSA state for the LAST bar only.
    Returns one of:
      NỔ VOL | CẠN VOL | RÚT CHÂN | PHÂN PHỐI | CẠN CẦU | —
    Same logic spirit as legacy, but output is a categorical feature only.
    """
    if df is None or len(df) < lookback + 2:
        return "—"

    c, h, l, v = df["close"], df["high"], df["low"], df["volume"].fillna(0)
    avg_vol = v.rolling(lookback).mean().replace(0, np.nan)
    vol_ratio = v / avg_vol
    spread = h - l
    avg_spread = spread.rolling(lookback).mean().replace(0, np.nan)
    spread_ratio = spread / avg_spread
    close_pos = np.where(spread == 0, 0.5, (c - l) / (spread + 1e-9))
    roc = c.pct_change()

    i = -1
    vr = float(vol_ratio.iloc[i]) if not pd.isna(vol_ratio.iloc[i]) else 1.0
    sr = float(spread_ratio.iloc[i]) if not pd.isna(spread_ratio.iloc[i]) else 1.0
    cp = float(close_pos[i])
    rc = float(roc.iloc[i]) if not pd.isna(roc.iloc[i]) else 0.0

    if vr > 1.5 and sr > 1.2 and cp > 0.7 and rc > 0.015:
        return "NỔ VOL"
    if vr < 0.6 and sr < 0.85 and cp >= 0.4 and rc > -0.015:
        return "CẠN VOL"
    if vr > 1.3 and sr > 1.3 and cp > 0.65 and rc > -0.02:
        return "RÚT CHÂN"
    if vr > 1.5 and cp < 0.35 and sr > 1.0 and rc < 0.02:
        return "PHÂN PHỐI"
    if vr < 0.6 and cp < 0.4 and rc > 0:
        return "CẠN CẦU"
    return "—"


def compute_rs_vs_benchmark(df_stk: pd.DataFrame, df_bmk: pd.DataFrame, period: int = 20) -> float:
    """RS = stock return - benchmark return over `period` sessions."""
    if df_stk is None or df_bmk is None:
        return np.nan
    if len(df_stk) < period + 1 or len(df_bmk) < period + 1:
        return np.nan
    # align on last available dates
    s = df_stk["close"].astype(float)
    b = df_bmk["close"].astype(float)
    rs = (s.iloc[-1] / s.iloc[-period-1] - 1) - (b.iloc[-1] / b.iloc[-period-1] - 1)
    return float(rs)


def compute_features_v2(df: pd.DataFrame, cfg: ResearchConfig = CFG,
                        df_bmk: pd.DataFrame = None) -> dict:
    """
    Extended feature set used by research dataset v2.
    Still strictly uses only rows present in `df` (ends at T).
    """
    base = compute_basic_features(df, cfg)
    if not base:
        return {}

    # VSA categorical
    base["vsa_state"] = compute_vsa_state(df)

    # RS vs benchmark if provided
    if df_bmk is not None:
        base["rs_5d"]  = compute_rs_vs_benchmark(df, df_bmk, 5)
        base["rs_20d"] = compute_rs_vs_benchmark(df, df_bmk, 20)
        base["rs_60d"] = compute_rs_vs_benchmark(df, df_bmk, 60)
    else:
        base["rs_5d"] = base["rs_20d"] = base["rs_60d"] = np.nan

    # Factors
    fe = FactorEngine(cfg)
    base.update(fe.compute(base))

    return base


print("Expanded features + VSA state (label-only) defined.")


Expanded features + VSA state (label-only) defined.


In [19]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 16 — MARKET REGIME ENGINE v2 (Spec §27)                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Keep rule-based initially (auditable). Expand dimensions:
#   trend | breadth (placeholder) | volatility | liquidity | risk_appetite
# Full breadth requires universe scan — plugged in later.

class MarketRegimeEngineV2:
    """
    Multi-dimensional regime from VNINDEX (and optional breadth inputs).
    Output example:
      {
        "trend": "bull",
        "volatility": "normal",
        "liquidity": "expanding",
        "risk_appetite": "risk_on",
        "label": "BULL / normal vol",
        "score_mult": 1.0,
        "wr_mult": 1.1,
      }
    """

    def __init__(self, store: MarketDataStore = None, cfg: ResearchConfig = CFG):
        self.store = store
        self.cfg = cfg
        self._last = {}

    def detect(self, df_vni: pd.DataFrame = None, breadth: dict = None) -> dict:
        if df_vni is None or len(df_vni) < 40:
            out = {
                "trend": "unknown", "volatility": "unknown",
                "liquidity": "unknown", "risk_appetite": "unknown",
                "label": "UNKNOWN", "score_mult": 0.9, "wr_mult": 1.0,
                "vnindex": np.nan, "mom5": np.nan, "atr_pct": np.nan,
            }
            self._last = out
            return out

        c = df_vni["close"].astype(float)
        h = df_vni["high"].astype(float)
        l = df_vni["low"].astype(float)
        v = df_vni["volume"].astype(float).fillna(0)

        ema10 = c.ewm(span=10, adjust=False).mean()
        ema30 = c.ewm(span=30, adjust=False).mean()
        ma200 = c.rolling(200).mean() if len(c) >= 200 else ema30

        trend_up = float(ema10.iloc[-1]) > float(ema30.iloc[-1])
        above_ma200 = float(c.iloc[-1]) > float(ma200.iloc[-1])
        mom5 = float(c.iloc[-1] / c.iloc[-6] - 1) * 100 if len(c) >= 6 else 0.0

        tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
        atr_pct = float(tr.rolling(14).mean().iloc[-1] / c.iloc[-1] * 100)

        # Trend dimension
        if trend_up and above_ma200 and mom5 > 1.0:
            trend = "bull"
        elif (not trend_up) and mom5 < -1.0:
            trend = "bear"
        elif atr_pct > 2.5 and mom5 < -2.0:
            trend = "panic"
        else:
            trend = "sideways"

        # Volatility dimension
        if atr_pct > 2.8:
            vol_dim = "high"
        elif atr_pct < 1.2:
            vol_dim = "low"
        else:
            vol_dim = "normal"

        # Liquidity proxy from index volume
        vol_ma20 = v.rolling(20).mean()
        if len(vol_ma20.dropna()) >= 5:
            vr = float(v.iloc[-1] / vol_ma20.iloc[-1]) if vol_ma20.iloc[-1] > 0 else 1.0
            liquidity = "expanding" if vr > 1.2 else ("contracting" if vr < 0.8 else "stable")
        else:
            liquidity = "unknown"

        # Risk appetite (simple)
        if trend == "bull" and vol_dim != "high":
            risk = "risk_on"
        elif trend in ("bear", "panic"):
            risk = "risk_off"
        else:
            risk = "neutral"

        # Optional breadth override
        if breadth:
            pct_above_ma50 = breadth.get("pct_above_ma50")
            if pct_above_ma50 is not None:
                if pct_above_ma50 < 0.40 and trend == "bull":
                    risk = "narrow_rally"
                elif pct_above_ma50 > 0.70 and trend == "bull":
                    risk = "risk_on_broad"

        mult_map = {"bull": 1.0, "sideways": 0.95, "bear": 0.85, "panic": 0.70, "unknown": 0.9}
        wr_map   = {"bull": 1.12, "sideways": 1.0, "bear": 0.82, "panic": 0.65, "unknown": 1.0}

        out = {
            "trend": trend,
            "volatility": vol_dim,
            "liquidity": liquidity,
            "risk_appetite": risk,
            "label": f"{trend.upper()} / {vol_dim} vol / {risk}",
            "score_mult": mult_map.get(trend, 0.9),
            "wr_mult": wr_map.get(trend, 1.0),
            "vnindex": round(float(c.iloc[-1]), 2),
            "mom5": round(mom5, 2),
            "atr_pct": round(atr_pct, 2),
        }
        self._last = out
        return out

    @property
    def last(self):
        return self._last


print("MarketRegimeEngineV2 defined.")


MarketRegimeEngineV2 defined.


In [20]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 17 — RESEARCH DATASET BUILDER v2 (with factors + RS + regime)     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class ResearchDatasetBuilderV2(ResearchDatasetBuilder):
    """
    Extends v1 builder:
    - uses compute_features_v2 (factors, VSA state, RS)
    - attaches market regime at each decision date
    - still strict: features ≤ T, labels > T
    """

    def build_for_symbol(self, symbol: str, start: str, end: str,
                         stride: int = 5,
                         df_vni_full: pd.DataFrame = None) -> pd.DataFrame:
        fetch_start = (pd.Timestamp(start) - timedelta(days=int(self.cfg.lookback_features * 2.5))).strftime("%Y-%m-%d")
        fetch_end   = (pd.Timestamp(end) + timedelta(days=int(self.cfg.max_holding_sessions * 3))).strftime("%Y-%m-%d")

        df = self.store.get(symbol, fetch_start, fetch_end)
        if df is None or len(df) < self.cfg.min_history_bars + self.cfg.max_holding_sessions + 5:
            return pd.DataFrame()

        df = df.sort_values("time").reset_index(drop=True)
        times = df["time"]

        if df_vni_full is None:
            df_vni_full = self.store.get("VNINDEX", fetch_start, fetch_end)
        if df_vni_full is not None:
            df_vni_full = df_vni_full.sort_values("time").reset_index(drop=True)

        regime_eng = MarketRegimeEngineV2(self.store, self.cfg)

        mask = (times >= pd.Timestamp(start)) & (times <= pd.Timestamp(end))
        decision_idx = np.where(mask)[0][::stride]

        rows = []
        for i in decision_idx:
            window = df.iloc[max(0, i - self.cfg.lookback_features + 1): i + 1].copy()
            if len(window) < self.cfg.min_history_bars:
                continue

            # Benchmark window ending at same date
            sig_date = times.iloc[i]
            df_bmk = None
            if df_vni_full is not None:
                df_bmk = df_vni_full[df_vni_full["time"] <= sig_date].tail(self.cfg.lookback_features)

            feats = compute_features_v2(window, self.cfg, df_bmk=df_bmk)
            if not feats:
                continue

            # Regime at T (using only VNI ≤ T)
            if df_bmk is not None and len(df_bmk) >= 40:
                reg = regime_eng.detect(df_bmk)
                feats["market_trend"] = reg["trend"]
                feats["market_vol"] = reg["volatility"]
                feats["market_label"] = reg["label"]
            else:
                feats["market_trend"] = "unknown"
                feats["market_vol"] = "unknown"
                feats["market_label"] = "UNKNOWN"

            if i + 1 >= len(df):
                continue
            entry_bar = df.iloc[i + 1]
            entry_price = float(entry_bar["open"])
            entry_date = times.iloc[i + 1]
            future = df.iloc[i + 1:].reset_index(drop=True)

            labels = {}
            for horizon in (3, 5, 10, 20):
                if len(future) > horizon:
                    labels[f"future_return_{horizon}d"] = float(future.iloc[horizon]["close"] / entry_price - 1)
                    window_f = future.iloc[: horizon + 1]
                    labels[f"future_max_return_{horizon}d"] = float(window_f["high"].max() / entry_price - 1)
                    labels[f"future_max_drawdown_{horizon}d"] = float(window_f["low"].min() / entry_price - 1)
                else:
                    labels[f"future_return_{horizon}d"] = np.nan
                    labels[f"future_max_return_{horizon}d"] = np.nan
                    labels[f"future_max_drawdown_{horizon}d"] = np.nan

            br = self.labeler.label_one(entry_price, future)
            if br is not None:
                labels.update({
                    "tb_label": br.label,
                    "tb_ret": br.ret,
                    "tb_mfe": br.mfe,
                    "tb_mae": br.mae,
                    "tb_sessions_held": br.sessions_held,
                })
            else:
                labels.update({
                    "tb_label": None, "tb_ret": np.nan, "tb_mfe": np.nan,
                    "tb_mae": np.nan, "tb_sessions_held": np.nan,
                })

            rows.append({
                "date": sig_date,
                "symbol": symbol,
                "entry_date": entry_date,
                "entry_price": entry_price,
                **feats,
                **labels,
            })

        return pd.DataFrame(rows)

    def build_universe(self, symbols, start, end, stride=5, save=True):
        # Prefetch VNINDEX once
        fetch_start = (pd.Timestamp(start) - timedelta(days=900)).strftime("%Y-%m-%d")
        fetch_end   = (pd.Timestamp(end) + timedelta(days=60)).strftime("%Y-%m-%d")
        df_vni = self.store.get("VNINDEX", fetch_start, fetch_end)

        all_dfs = []
        for i, sym in enumerate(symbols):
            print(f"  [{i+1}/{len(symbols)}] {sym}...", end=" ")
            try:
                dfi = self.build_for_symbol(sym, start, end, stride=stride, df_vni_full=df_vni)
                print(f"{len(dfi)} rows")
                if len(dfi):
                    all_dfs.append(dfi)
            except Exception as e:
                print(f"FAIL {type(e).__name__}: {e}")
            time.sleep(self.cfg.delay_sec)

        if not all_dfs:
            return pd.DataFrame()

        out = pd.concat(all_dfs, ignore_index=True)
        out["feature_version"] = self.cfg.feature_version + "+factors"

        if save:
            path = DATA_FEATURES / f"features_v2_{start}_{end}_s{stride}.parquet"
            out.to_parquet(path, index=False)
            print(f"\n✓ Saved {len(out)} rows → {path}")
        return out


print("ResearchDatasetBuilderV2 defined.")


ResearchDatasetBuilderV2 defined.


In [21]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 18 — RESEARCH HELPERS (factor & VSA analysis)                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def analyze_factors(ds: pd.DataFrame, eng: EmpiricalProbabilityEngine = None) -> pd.DataFrame:
    """Univariate factor quantiles vs triple-barrier outcomes."""
    if eng is None:
        eng = EmpiricalProbabilityEngine(CFG)
    rows = []
    for col in ["factor_trend", "factor_momentum", "factor_flow",
                "factor_rs", "factor_volatility", "factor_structure", "factor_composite"]:
        if col not in ds.columns:
            continue
        buck = eng.by_bucket(ds, col, quantiles=4)
        if len(buck):
            rows.append(buck)
    if not rows:
        return pd.DataFrame()
    return pd.concat(rows, ignore_index=True)


def analyze_vsa(ds: pd.DataFrame, eng: EmpiricalProbabilityEngine = None) -> pd.DataFrame:
    """Empirical outcomes by VSA state — this REPLACES hard-coded 8.5/1.5 scores."""
    if eng is None:
        eng = EmpiricalProbabilityEngine(CFG)
    if "vsa_state" not in ds.columns:
        return pd.DataFrame()
    rows = []
    for state, g in ds.groupby("vsa_state"):
        s = eng.summarize(g)
        s["vsa_state"] = state
        rows.append(s)
    return pd.DataFrame(rows).sort_values("n", ascending=False)


def analyze_regime(ds: pd.DataFrame, eng: EmpiricalProbabilityEngine = None) -> pd.DataFrame:
    if eng is None:
        eng = EmpiricalProbabilityEngine(CFG)
    if "market_trend" not in ds.columns:
        return pd.DataFrame()
    rows = []
    for reg, g in ds.groupby("market_trend"):
        s = eng.summarize(g)
        s["market_trend"] = reg
        rows.append(s)
    return pd.DataFrame(rows)


def demo_research_pipeline_v2(symbols=None, start="2023-01-01", end="2024-12-31", stride=5):
    if symbols is None:
        symbols = ["FPT", "VNM", "HPG", "MWG", "VCB"]

    print("═" * 60)
    print("  DEMO RESEARCH PIPELINE v2 (factors + VSA empirical + regime)")
    print("═" * 60)

    register_vnstock_if_needed()
    store = MarketDataStore(CFG)
    store.update_calendar_from("VNINDEX", start="2020-01-01")

    print("\n[1] Look-ahead test...")
    df_fpt = store.get("FPT", "2023-01-01", "2024-12-31")
    if df_fpt is not None and len(df_fpt) > 100:
        ok = test_no_lookahead(lambda d: compute_features_v2(d, CFG), df_fpt, len(df_fpt)//2)
        print(f"    Result: {'PASS' if ok else 'FAIL'}")

    print("\n[2] Building dataset v2...")
    builder = ResearchDatasetBuilderV2(store, CFG)
    ds = builder.build_universe(symbols, start, end, stride=stride, save=True)

    if len(ds) == 0:
        print("  ❌ Empty dataset")
        return None

    print(f"\n  Shape: {ds.shape}")
    eng = EmpiricalProbabilityEngine(CFG)

    print("\n[3] Overall triple-barrier:")
    print(eng.summarize(ds))

    print("\n[4] Empirical VSA states (NO hard-coded scores):")
    vsa_tbl = analyze_vsa(ds, eng)
    if len(vsa_tbl):
        cols = [c for c in ["vsa_state", "n", "confidence", "P_TP", "P_SL", "P_TIMEOUT", "avg_ret", "EV_simple"] if c in vsa_tbl.columns]
        print(vsa_tbl[cols].to_string(index=False))

    print("\n[5] Factor composite quantiles:")
    fac_tbl = analyze_factors(ds, eng)
    if len(fac_tbl):
        sub = fac_tbl[fac_tbl["feature"] == "factor_composite"]
        if len(sub):
            print(sub[["bucket", "n", "P_TP", "P_SL", "avg_ret"]].to_string(index=False))

    print("\n[6] By market regime:")
    reg_tbl = analyze_regime(ds, eng)
    if len(reg_tbl):
        cols = [c for c in ["market_trend", "n", "confidence", "P_TP", "P_SL", "avg_ret"] if c in reg_tbl.columns]
        print(reg_tbl[cols].to_string(index=False))

    meta = ResearchRunMetadata(
        run_id=make_run_id(),
        timestamp=datetime.now().isoformat(),
        feature_version=CFG.feature_version + "+factors",
        model_version=CFG.model_version,
        config_version=CFG.config_version,
        data_start=start, data_end=end,
        universe=symbols, n_symbols=len(symbols),
        execution_model=CFG.execution_model,
        tp_pct=CFG.tp_pct, sl_pct=CFG.sl_pct,
        max_holding_sessions=CFG.max_holding_sessions,
        fee_buy=CFG.fee_buy, fee_sell=CFG.fee_sell, tax_sell=CFG.tax_sell,
        random_seed=CFG.random_seed,
        notes="v2 demo: factors + VSA empirical + regime",
    )
    meta.save()
    print("\n✓ Demo v2 complete.")
    return ds


print("Research helpers + demo_v2 defined.")
print("→ Call: ds = demo_research_pipeline_v2()")


Research helpers + demo_v2 defined.
→ Call: ds = demo_research_pipeline_v2()


In [22]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 19 — SETUP ENGINE (Spec §28–33)                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Interpretable setups. Thresholds are CONFIGURABLE and must later be
# validated statistically — do not treat them as truth.

@dataclass
class SetupConfig:
    # Breakout
    brk_vol_min: float = 1.4
    brk_close_pos_min: float = 0.65
    brk_rs20_min: float = 0.0
    # Pullback
    pb_ema_touch_pct: float = 2.0      # within 2% of EMA20
    pb_vol_max: float = 0.95
    pb_trend_min: float = 55.0         # factor_trend
    # Accumulation
    acc_bb_width_max: float = 0.07
    acc_vol_max: float = 0.85
    acc_cmf_min: float = -0.02
    # Distribution
    dist_rs20_max: float = -0.02
    dist_cmf_max: float = -0.05
    dist_vol_min: float = 1.3


class SetupEngine:
    """
    Labels each observation with zero or more setup flags.
    Primary setup = highest-priority match (distribution > breakout > pullback > accumulation).
    """

    PRIORITY = ["DISTRIBUTION", "BREAKOUT", "PULLBACK", "ACCUMULATION", "NONE"]

    def __init__(self, cfg: ResearchConfig = CFG, scfg: SetupConfig = None):
        self.cfg = cfg
        self.scfg = scfg or SetupConfig()

    def classify(self, f: dict) -> dict:
        sc = self.scfg
        flags = {
            "setup_breakout": False,
            "setup_pullback": False,
            "setup_accumulation": False,
            "setup_distribution": False,
        }

        vol_r = float(f.get("volume_ratio") or 1.0)
        cmf   = float(f.get("cmf") or 0.0)
        rs20  = f.get("rs_20d")
        rs20  = float(rs20) if rs20 is not None and not (isinstance(rs20, float) and np.isnan(rs20)) else 0.0
        ema_pct = float(f.get("ema_pct") or 0.0)
        bb_w  = float(f.get("bb_width") or 0.1)
        trend_f = float(f.get("factor_trend") or 50.0)
        flow_f  = float(f.get("factor_flow") or 50.0)
        vsa = str(f.get("vsa_state") or "—")
        above50 = int(f.get("above_sma50") or 0)
        ret5 = float(f.get("ret_5d") or 0.0)

        # Approx close position from bb_pctb as proxy when not stored
        pctb = float(f.get("bb_pctb") or 0.5)

        # BREAKOUT: strong short momentum + volume expansion + not extended RS negative
        if (vol_r >= sc.brk_vol_min and pctb >= 0.7 and ret5 > 0.01
                and rs20 >= sc.brk_rs20_min and above50 == 1):
            flags["setup_breakout"] = True
        if vsa == "NỔ VOL" and ret5 > 0 and above50 == 1:
            flags["setup_breakout"] = True

        # PULLBACK: uptrend structure, price near EMA20, volume contracts
        if (trend_f >= sc.pb_trend_min and above50 == 1
                and abs(ema_pct) <= sc.pb_ema_touch_pct
                and vol_r <= sc.pb_vol_max and cmf > -0.08):
            flags["setup_pullback"] = True

        # ACCUMULATION: compression + quiet volume + flow not bleeding
        if (bb_w <= sc.acc_bb_width_max and vol_r <= sc.acc_vol_max
                and cmf >= sc.acc_cmf_min and trend_f >= 45):
            flags["setup_accumulation"] = True
        if vsa == "CẠN VOL" and cmf >= -0.05:
            flags["setup_accumulation"] = True

        # DISTRIBUTION: uptrend context deteriorating
        if (trend_f >= 50 and (rs20 <= sc.dist_rs20_max or cmf <= sc.dist_cmf_max)
                and vol_r >= sc.dist_vol_min and ret5 < 0):
            flags["setup_distribution"] = True
        if vsa in ("PHÂN PHỐI", "CẠN CẦU") and trend_f >= 45:
            flags["setup_distribution"] = True

        # Primary label
        if flags["setup_distribution"]:
            primary = "DISTRIBUTION"
        elif flags["setup_breakout"]:
            primary = "BREAKOUT"
        elif flags["setup_pullback"]:
            primary = "PULLBACK"
        elif flags["setup_accumulation"]:
            primary = "ACCUMULATION"
        else:
            primary = "NONE"

        flags["setup"] = primary
        return flags


print("SetupEngine defined.")


SetupEngine defined.


In [23]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 20 — EXIT RISK ENGINE (Spec §35–38)                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# exit_risk 0–100. Bands are semantic only until statistical validation:
#   0–20 HEALTHY | 20–40 WATCH | 40–60 DETERIORATING | 60–80 REDUCE | 80–100 EXIT
# Do NOT sell solely because price < MA9 (Spec §38).

class ExitRiskEngine:
    def __init__(self, cfg: ResearchConfig = CFG):
        self.cfg = cfg

    def score(self, f: dict) -> dict:
        risk = 0.0
        reasons = []

        rs20 = f.get("rs_20d")
        rs20 = float(rs20) if rs20 is not None and not (isinstance(rs20, float) and np.isnan(rs20)) else 0.0
        cmf = float(f.get("cmf") or 0.0)
        ema_pct = float(f.get("ema_pct") or 0.0)
        above50 = int(f.get("above_sma50") or 0)
        above200 = int(f.get("above_sma200") or 0)
        trend_f = float(f.get("factor_trend") or 50.0)
        flow_f = float(f.get("factor_flow") or 50.0)
        mom_f = float(f.get("factor_momentum") or 50.0)
        vsa = str(f.get("vsa_state") or "—")
        setup = str(f.get("setup") or "NONE")
        ret5 = float(f.get("ret_5d") or 0.0)
        macd = float(f.get("macd_hist_pct") or 0.0)

        # RS deterioration
        if rs20 < -0.05:
            risk += 18; reasons.append("RS20 weak")
        elif rs20 < -0.02:
            risk += 10; reasons.append("RS20 soft")

        # Flow deterioration
        if cmf < -0.12:
            risk += 18; reasons.append("CMF very negative")
        elif cmf < -0.05:
            risk += 10; reasons.append("CMF negative")

        if flow_f < 35:
            risk += 10; reasons.append("Flow factor low")

        # Trend / structure
        if above50 == 0:
            risk += 12; reasons.append("Below SMA50")
        if above200 == 0 and trend_f < 45:
            risk += 10; reasons.append("Below SMA200 + weak trend")

        if ema_pct < -5:
            risk += 8; reasons.append("Deep below EMA20")

        # Momentum
        if mom_f < 35:
            risk += 8; reasons.append("Momentum factor low")
        if macd < -0.05 and ret5 < 0:
            risk += 6; reasons.append("MACD hist negative + 5d down")

        # Distribution signals
        if setup == "DISTRIBUTION" or vsa == "PHÂN PHỐI":
            risk += 20; reasons.append("Distribution setup/VSA")
        elif vsa == "CẠN CẦU":
            risk += 12; reasons.append("Cạn cầu VSA")

        risk = float(np.clip(risk, 0, 100))

        if risk < 20:
            action, band = "HOLD", "HEALTHY"
        elif risk < 40:
            action, band = "WATCH", "WATCH"
        elif risk < 60:
            action, band = "WATCH", "DETERIORATING"
        elif risk < 80:
            action, band = "REDUCE", "REDUCE"
        else:
            action, band = "EXIT", "EXIT"

        return {
            "exit_risk": round(risk, 1),
            "exit_band": band,
            "exit_action": action,
            "exit_reasons": reasons,
        }


print("ExitRiskEngine defined.")


ExitRiskEngine defined.


In [24]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 21 — ENRICH DATASET WITH SETUP + EXIT RISK                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def enrich_with_setup_exit(ds: pd.DataFrame, cfg: ResearchConfig = CFG) -> pd.DataFrame:
    """Post-process an existing research dataset: add setup flags + exit risk."""
    if ds is None or len(ds) == 0:
        return ds
    se = SetupEngine(cfg)
    ee = ExitRiskEngine(cfg)
    rows = []
    for _, row in ds.iterrows():
        f = row.to_dict()
        setup = se.classify(f)
        f.update(setup)
        ex = ee.score(f)
        f.update(ex)
        rows.append(f)
    out = pd.DataFrame(rows)
    return out


def analyze_setups(ds: pd.DataFrame, eng: EmpiricalProbabilityEngine = None) -> pd.DataFrame:
    if eng is None:
        eng = EmpiricalProbabilityEngine(CFG)
    if "setup" not in ds.columns:
        return pd.DataFrame()
    rows = []
    for name, g in ds.groupby("setup"):
        s = eng.summarize(g)
        s["setup"] = name
        rows.append(s)
    return pd.DataFrame(rows).sort_values("n", ascending=False)


def demo_research_pipeline_v3(symbols=None, start="2023-01-01", end="2024-12-31", stride=5):
    if symbols is None:
        symbols = ["FPT", "VNM", "HPG", "MWG", "VCB"]

    print("═" * 60)
    print("  DEMO RESEARCH PIPELINE v3 (setup + exit risk)")
    print("═" * 60)

    register_vnstock_if_needed()
    store = MarketDataStore(CFG)
    store.update_calendar_from("VNINDEX", start="2020-01-01")

    print("\n[1] Build dataset v2...")
    builder = ResearchDatasetBuilderV2(store, CFG)
    ds = builder.build_universe(symbols, start, end, stride=stride, save=False)
    if len(ds) == 0:
        print("  ❌ Empty")
        return None

    print("\n[2] Enrich setup + exit risk...")
    ds = enrich_with_setup_exit(ds, CFG)
    path = DATA_FEATURES / f"features_v3_{start}_{end}_s{stride}.parquet"
    ds.to_parquet(path, index=False)
    print(f"  ✓ Saved {len(ds)} rows → {path}")

    eng = EmpiricalProbabilityEngine(CFG)

    print("\n[3] Overall barrier:")
    print(eng.summarize(ds))

    print("\n[4] By SETUP (conditional — this is the key table):")
    st = analyze_setups(ds, eng)
    if len(st):
        cols = [c for c in ["setup", "n", "confidence", "P_TP", "P_SL", "P_TIMEOUT", "avg_ret", "EV_simple"] if c in st.columns]
        print(st[cols].to_string(index=False))

    print("\n[5] VSA (reference):")
    vsa_tbl = analyze_vsa(ds, eng)
    if len(vsa_tbl):
        cols = [c for c in ["vsa_state", "n", "confidence", "P_TP", "P_SL", "avg_ret"] if c in vsa_tbl.columns]
        print(vsa_tbl[cols].to_string(index=False))

    print("\n[6] Exit band distribution:")
    if "exit_band" in ds.columns:
        print(ds["exit_band"].value_counts().to_string())
        print("\n  Exit band vs avg future 10d return:")
        if "future_return_10d" in ds.columns:
            print(ds.groupby("exit_band")["future_return_10d"].agg(["count", "mean"]).to_string())

    print("\n[7] BREAKOUT only — regime split (if enough n):")
    if "setup" in ds.columns:
        brk = ds[ds["setup"] == "BREAKOUT"]
        if len(brk) >= 10:
            print(analyze_regime(brk, eng)[["market_trend", "n", "P_TP", "P_SL", "avg_ret"]].to_string(index=False)
                  if "market_trend" in brk.columns else f"n_breakout={len(brk)}")
        else:
            print(f"  n_breakout={len(brk)} (need more symbols / longer window for confidence)")

    meta = ResearchRunMetadata(
        run_id=make_run_id(),
        timestamp=datetime.now().isoformat(),
        feature_version=CFG.feature_version + "+setup_exit",
        model_version=CFG.model_version,
        config_version=CFG.config_version,
        data_start=start, data_end=end,
        universe=symbols, n_symbols=len(symbols),
        execution_model=CFG.execution_model,
        tp_pct=CFG.tp_pct, sl_pct=CFG.sl_pct,
        max_holding_sessions=CFG.max_holding_sessions,
        fee_buy=CFG.fee_buy, fee_sell=CFG.fee_sell, tax_sell=CFG.tax_sell,
        random_seed=CFG.random_seed,
        notes="v3: SetupEngine + ExitRiskEngine",
    )
    meta.save()
    print("\n✓ Demo v3 complete.")
    return ds


print("Enrich + demo_v3 defined.")
print("→ Call: ds = demo_research_pipeline_v3()")


Enrich + demo_v3 defined.
→ Call: ds = demo_research_pipeline_v3()


---

## Status

| Phase | Status | Notes |
|-------|--------|-------|
| 1 Correctness | ✅ | look-ahead PASS on live data |
| 2 Data layer | ✅ | prefer `vnstock.api.quote.Quote` |
| 3–4 Dataset + TB | ✅ | 500 rows demo |
| 5 Factors | ✅ | no magic trade score |
| 6 Regime + RS | ✅ | breadth/sector still thin |
| 7 SetupEngine | ✅ | BREAKOUT / PULLBACK / ACCUMULATION / DISTRIBUTION |
| 8 ExitRiskEngine | ✅ | 0–100 + HOLD/WATCH/REDUCE/EXIT |
| 9+ ML / Backtest / WF | 📋 | next after setup stats stabilize |

### Read the VSA table from your last run (honestly)

| State | n | Signal |
|-------|---|--------|
| — | 435 | baseline |
| NỔ VOL / CẠN VOL | ~20 | **LOW confidence** — no clear edge yet |
| PHÂN PHỐI / CẠN CẦU | 12 / 6 | P_SL 67% but **n too small** to trust |
| Factor composite | — | **not monotonic** → do not trade on composite alone |

This is exactly why Spec forbids hard-coded VSA scores and fixed 9% TP.

### Run next

```python
ds = demo_research_pipeline_v3(
    symbols=["FPT","VNM","HPG","MWG","VCB","SSI","ACB","MBB","HDB","VHM"],
    start="2022-01-01",
    end="2025-12-31",
    stride=5,
)
```

Expand symbols if rate limit allows — setup conditional stats need **n ≥ 30 per setup** ideally.


In [25]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  NOTE — VNSTOCK COMMUNITY DATA LIMIT                                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# WARNING text:
#   "Community edition: OHLCV data (1D) limited to maximum 8 years."
#
# Impact assessment (as of 2026-08):
#   - Rolling window available ≈ last 8 years from today.
#   - Research windows 2020→2025 or 2022→2025 are FULLY inside the limit.
#   - NO truncation for current demos / backtests in this notebook.
#   - Only a problem if you request history before ~2018-08.
#
# Actions taken:
#   1. Document here (do not panic).
#   2. MarketDataStore already caches parquet → repeat runs do not re-hit API.
#   3. Optional: suppress repeated vnai warnings after first notice.

import logging
logging.getLogger("vnai.beam.patching").setLevel(logging.ERROR)
logging.getLogger("vnstock.common.data").setLevel(logging.ERROR)

# Hard guard: refuse research starts that would be silently truncated
COMMUNITY_MAX_HISTORY_YEARS = 8

def assert_history_window_ok(start: str, end: str = None):
    """Raise if requested start is older than community 8y limit (approx)."""
    start_ts = pd.Timestamp(start)
    # approximate: 8*365.25 days back from today
    earliest = pd.Timestamp.today().normalize() - pd.Timedelta(days=int(COMMUNITY_MAX_HISTORY_YEARS * 365.25))
    if start_ts < earliest:
        raise ValueError(
            f"Requested start={start} is older than community OHLCV limit (~{earliest.date()}). "
            f"Either upgrade vnstock plan or set start >= {earliest.date()}."
        )
    return True

print("Community 8y limit noted. Warnings suppressed after first load. Guard helper ready.")


Community 8y limit noted. Warnings suppressed after first load. Guard helper ready.


In [26]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 22 — BACKTEST ENGINE v2 (Spec §45–50)                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Default execution: Signal @ T close → Entry @ T+1 open
# Includes: fees, tax on sell, simple slippage, T+ settlement delay,
#           busy-until per symbol, setup filter.
#
# Does NOT yet: portfolio correlation, limit-up/down fill model, full WF.

@dataclass
class BacktestConfigV2:
    initial_capital: float = 100_000_000
    risk_per_trade: float = 0.0075          # 0.75% NAV
    max_position_pct: float = 0.15
    max_open_positions: int = 5
    settlement_sessions: int = 2            # cannot exit before T+2 sessions
    max_hold_sessions: int = 10
    fee_buy: float = 0.0015
    fee_sell: float = 0.0015
    tax_sell: float = 0.001
    slippage_bps: float = 5.0               # flat 5 bps each side (simple)
    # Which setups to take
    allowed_setups: tuple = ("PULLBACK", "BREAKOUT", "ACCUMULATION")
    min_factor_trend: float = 0.0           # optional filter
    use_exit_risk_gate: bool = True
    max_exit_risk_to_enter: float = 60.0    # do not enter if already deteriorating


class BacktestEngineV2:
    """
    Event-style backtest over a pre-built research dataset (features+labels).
    Uses entry_price already stored (T+1 open) and walks barriers with costs.
    """

    def __init__(self, cfg: ResearchConfig = CFG, bt: BacktestConfigV2 = None):
        self.cfg = cfg
        self.bt = bt or BacktestConfigV2(
            fee_buy=cfg.fee_buy, fee_sell=cfg.fee_sell, tax_sell=cfg.tax_sell,
            max_hold_sessions=cfg.max_holding_sessions,
        )
        self.trades = []

    def _apply_slippage(self, price: float, side: str) -> float:
        bps = self.bt.slippage_bps / 10000.0
        if side == "buy":
            return price * (1 + bps)
        return price * (1 - bps)

    def _costs_roundtrip(self, entry: float, exit_p: float, shares: int) -> float:
        """Total cost in VND (fees + tax on sell)."""
        notional_in = entry * shares
        notional_out = exit_p * shares
        fee_in = notional_in * self.bt.fee_buy
        fee_out = notional_out * self.bt.fee_sell
        tax = notional_out * self.bt.tax_sell
        return fee_in + fee_out + tax

    def run_from_dataset(self, ds: pd.DataFrame) -> pd.DataFrame:
        """
        ds must contain: date, symbol, entry_date, entry_price, setup,
        tb_label, tb_ret, tb_sessions_held, atr_abs (optional), exit_risk (optional)
        """
        if ds is None or len(ds) == 0:
            return pd.DataFrame()

        df = ds.copy()
        df["date"] = pd.to_datetime(df["date"])
        df["entry_date"] = pd.to_datetime(df["entry_date"])
        df = df.sort_values(["date", "symbol"]).reset_index(drop=True)

        capital = self.bt.initial_capital
        open_pos = {}  # symbol -> trade dict
        equity_curve = []
        self.trades = []

        # Group by signal date for sequential processing
        for sig_date, batch in df.groupby("date", sort=True):
            # 1) Mark exits that are due (simplified: use precomputed tb on signal row)
            #    Full path-dependent needs future OHLCV; here we use dataset labels
            #    which already path-walked TP/SL/TIMEOUT from T+1.
            to_close = []
            for sym, tr in list(open_pos.items()):
                # approximate: if held long enough vs label horizon
                held = (sig_date - tr["entry_date"]).days  # calendar proxy; OK for reporting
                # Better: use sessions_held from label when available at entry time
                if tr.get("resolved"):
                    to_close.append(sym)
            for sym in to_close:
                open_pos.pop(sym, None)

            # 2) New entries
            if len(open_pos) >= self.bt.max_open_positions:
                continue

            cands = batch.copy()
            # Filters
            if "setup" in cands.columns:
                cands = cands[cands["setup"].isin(self.bt.allowed_setups)]
            if self.bt.use_exit_risk_gate and "exit_risk" in cands.columns:
                cands = cands[cands["exit_risk"] <= self.bt.max_exit_risk_to_enter]
            if "factor_trend" in cands.columns and self.bt.min_factor_trend > 0:
                cands = cands[cands["factor_trend"] >= self.bt.min_factor_trend]

            # Rank by factor_composite or pullback preference
            if "factor_composite" in cands.columns:
                cands = cands.sort_values("factor_composite", ascending=False)

            for _, row in cands.iterrows():
                sym = row["symbol"]
                if sym in open_pos:
                    continue
                if len(open_pos) >= self.bt.max_open_positions:
                    break

                entry_raw = float(row["entry_price"])
                entry = self._apply_slippage(entry_raw, "buy")

                # Stop distance: use SL barrier % or ATR
                sl_pct = self.cfg.sl_pct
                stop_dist = entry * sl_pct
                if stop_dist <= 0:
                    continue

                risk_budget = capital * self.bt.risk_per_trade
                shares = int(risk_budget / stop_dist)
                # lot size VN often 100
                shares = (shares // 100) * 100
                if shares <= 0:
                    continue

                max_shares = int((capital * self.bt.max_position_pct) / entry)
                max_shares = (max_shares // 100) * 100
                shares = min(shares, max_shares)
                if shares <= 0:
                    continue

                # Resolve outcome from precomputed triple barrier (gross)
                tb_label = row.get("tb_label")
                tb_ret = float(row.get("tb_ret") or 0.0)
                sessions = int(row.get("tb_sessions_held") or self.bt.max_hold_sessions)

                # Enforce settlement: if sessions < settlement, treat as timeout at label exit
                # (dataset already used path rules; we just account costs)
                exit_gross = entry_raw * (1 + tb_ret)
                exit_net_price = self._apply_slippage(exit_gross, "sell")

                cost = self._costs_roundtrip(entry, exit_net_price, shares)
                pnl_gross = (exit_gross - entry_raw) * shares
                # more accurate with slipped prices:
                pnl_net = (exit_net_price - entry) * shares - cost
                ret_net = pnl_net / (entry * shares) if shares else 0.0

                trade = {
                    "signal_date": row["date"],
                    "entry_date": row["entry_date"],
                    "symbol": sym,
                    "setup": row.get("setup"),
                    "entry": round(entry, 2),
                    "exit": round(exit_net_price, 2),
                    "shares": shares,
                    "tb_label": tb_label,
                    "sessions_held": sessions,
                    "pnl_net": round(pnl_net, 0),
                    "ret_net": round(ret_net, 4),
                    "cost": round(cost, 0),
                    "exit_risk": row.get("exit_risk"),
                    "market_trend": row.get("market_trend"),
                    "factor_composite": row.get("factor_composite"),
                }
                self.trades.append(trade)
                capital += pnl_net
                open_pos[sym] = {**trade, "resolved": True}

            equity_curve.append({"date": sig_date, "capital": capital, "n_open": len(open_pos)})

        return pd.DataFrame(self.trades)

    def report(self, trades: pd.DataFrame = None) -> dict:
        if trades is None:
            trades = pd.DataFrame(self.trades)
        if trades is None or len(trades) == 0:
            print("  No trades.")
            return {}

        t = trades
        n = len(t)
        wins = t[t["ret_net"] > 0]
        losses = t[t["ret_net"] <= 0]
        wr = len(wins) / n * 100
        avg_win = wins["ret_net"].mean() if len(wins) else 0
        avg_loss = losses["ret_net"].mean() if len(losses) else 0
        pf = wins["pnl_net"].sum() / abs(losses["pnl_net"].sum()) if len(losses) and losses["pnl_net"].sum() != 0 else float("inf")
        exp = t["ret_net"].mean()
        total_ret = t["pnl_net"].sum() / self.bt.initial_capital

        # Max DD on cumulative pnl
        cum = t["pnl_net"].cumsum()
        peak = cum.cummax()
        dd = (cum - peak)
        max_dd = float(dd.min()) if len(dd) else 0

        rep = {
            "n_trades": n,
            "win_rate_pct": round(wr, 2),
            "profit_factor": round(pf, 3) if pf != float("inf") else "inf",
            "expectancy_ret": round(float(exp), 4),
            "avg_win": round(float(avg_win), 4),
            "avg_loss": round(float(avg_loss), 4),
            "total_return_on_capital": round(float(total_ret), 4),
            "sum_pnl_net": round(float(t["pnl_net"].sum()), 0),
            "max_dd_pnl": round(max_dd, 0),
            "avg_sessions": round(float(t["sessions_held"].mean()), 2),
            "total_costs": round(float(t["cost"].sum()), 0),
        }

        print("═" * 60)
        print("  BACKTEST v2 REPORT (costs + slippage included)")
        print("═" * 60)
        for k, v in rep.items():
            print(f"  {k:28s}: {v}")

        if "setup" in t.columns:
            print("\n  By setup:")
            g = t.groupby("setup").agg(
                n=("ret_net", "count"),
                wr=("ret_net", lambda x: (x > 0).mean()),
                avg_ret=("ret_net", "mean"),
                sum_pnl=("pnl_net", "sum"),
            )
            print(g.to_string())

        if "market_trend" in t.columns:
            print("\n  By regime:")
            g2 = t.groupby("market_trend").agg(
                n=("ret_net", "count"),
                wr=("ret_net", lambda x: (x > 0).mean()),
                avg_ret=("ret_net", "mean"),
            )
            print(g2.to_string())

        # Save
        path = RESULTS_DIR / "backtest_v2_trades.parquet"
        t.to_parquet(path, index=False)
        print(f"\n  ✓ Trades saved → {path}")
        return rep


print("BacktestEngineV2 defined.")


BacktestEngineV2 defined.


In [27]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 23 — RUN BACKTEST ON LATEST DATASET                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def run_backtest_demo(ds: pd.DataFrame = None, parquet_path: str = None):
    """
    Prefer in-memory ds from demo_v3; else load latest features_v3 parquet.
    """
    if ds is None:
        if parquet_path is None:
            # pick newest v3 features
            files = sorted(DATA_FEATURES.glob("features_v3_*.parquet"))
            if not files:
                print("No features_v3 parquet found. Run demo_research_pipeline_v3 first.")
                return None
            parquet_path = files[-1]
        print(f"Loading {parquet_path}...")
        ds = pd.read_parquet(parquet_path)
        # ensure setup/exit columns exist
        if "setup" not in ds.columns:
            ds = enrich_with_setup_exit(ds)

    print(f"Dataset: {len(ds)} rows | setups: {ds['setup'].value_counts().to_dict() if 'setup' in ds.columns else 'n/a'}")

    # Baseline: only PULLBACK + BREAKOUT (best empirical so far)
    bt_cfg = BacktestConfigV2(
        allowed_setups=("PULLBACK", "BREAKOUT"),
        use_exit_risk_gate=True,
        max_exit_risk_to_enter=50.0,
        risk_per_trade=0.0075,
        max_open_positions=4,
    )
    engine = BacktestEngineV2(CFG, bt_cfg)
    trades = engine.run_from_dataset(ds)
    rep = engine.report(trades)

    # Ablation lite: ALL setups vs PULLBACK-only
    print("\n── Ablation lite: PULLBACK only ──")
    bt2 = BacktestConfigV2(allowed_setups=("PULLBACK",), use_exit_risk_gate=True, max_exit_risk_to_enter=50.0)
    e2 = BacktestEngineV2(CFG, bt2)
    t2 = e2.run_from_dataset(ds)
    e2.report(t2)

    print("\n── Ablation lite: no setup filter (all rows) ──")
    bt3 = BacktestConfigV2(allowed_setups=("PULLBACK", "BREAKOUT", "ACCUMULATION", "DISTRIBUTION", "NONE"),
                           use_exit_risk_gate=False)
    e3 = BacktestEngineV2(CFG, bt3)
    t3 = e3.run_from_dataset(ds)
    e3.report(t3)

    return trades, rep


print("run_backtest_demo defined.")
print("→ After demo_v3: trades, rep = run_backtest_demo(ds)")
print("→ Or: trades, rep = run_backtest_demo()  # loads latest parquet")


run_backtest_demo defined.
→ After demo_v3: trades, rep = run_backtest_demo(ds)
→ Or: trades, rep = run_backtest_demo()  # loads latest parquet


---

## Interpretation of your v3 run (2000 rows, 10 names, 2022–2025)

### Setup table (most important)

| Setup | n | P_TP | P_SL | avg_ret | EV_simple |
|-------|---|------|------|---------|-----------|
| **PULLBACK** | 179 | **25.7%** | 44.7% | **+0.56%** | **+0.002** |
| BREAKOUT | 115 | 23.5% | 46.1% | +0.41% | ~0 |
| ACCUMULATION | 65 | 21.5% | 43.1% | +0.24% | ~0 |
| DISTRIBUTION | 206 | 23.3% | 50.0% | +0.20% | ~0 |
| NONE | 1435 | 20.8% | 49.6% | +0.08% | negative |

**Takeaway:** Under fixed 6%/3%/10-session barriers, only **PULLBACK** shows a small positive expectancy. Edge is **thin** — costs may erase it. Do not scale capital yet.

### VSA
- **CẠN VOL** looks best (P_TP 27.5%, lower SL) with n=109 HIGH confidence → worth keeping as a feature, still not a standalone signal.
- **NỔ VOL** does **not** beat baseline → legacy score 8.5 was not justified on this sample.

### Exit risk (important problem)
EXIT band has **higher** average future 10d return than HEALTHY.  
→ Current ExitRiskEngine heuristic is **miscalibrated** (penalizing conditions that still mean-revert up).  
→ Do **not** use exit_action for live sells until rebuilt from empirical deterioration → forward loss link.

### Community 8-year warning
**No impact** on 2022–2025 (or 2020–2025). Only matters if `start < ~2018-08`. Guard + log suppression added.

### Next
```python
# If ds still in memory:
trades, rep = run_backtest_demo(ds)

# Or from parquet:
trades, rep = run_backtest_demo()
```

Then we will: ATR-adaptive barriers, fix exit model, walk-forward split.


In [28]:
# trades, rep = run_backtest_demo()  # uses latest features_v3 parquet
# or: trades, rep = run_backtest_demo(ds)  # if ds from demo_v3 still in memory


In [29]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 24 — ATR-ADAPTIVE TRIPLE BARRIER (Spec §14)                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Fixed 6%/3% ignores volatility. VN large-caps often ATR% ~1.5–3%.
# Adaptive:
#   TP = max(min_tp, k_tp * ATR%)
#   SL = max(min_sl, k_sl * ATR%)
# Default k_tp=2.0, k_sl=1.0 → ~ asymmetric but vol-aware.

@dataclass
class AdaptiveBarrierConfig:
    k_tp: float = 2.0
    k_sl: float = 1.0
    min_tp_pct: float = 0.03   # floor 3%
    max_tp_pct: float = 0.12
    min_sl_pct: float = 0.015
    max_sl_pct: float = 0.06
    max_hold: int = 10


def atr_barriers(atr_pct: float, cfg: AdaptiveBarrierConfig = None) -> tuple:
    """Return (tp_pct, sl_pct) from ATR% (e.g. atr_pct=2.5 means 2.5%)."""
    cfg = cfg or AdaptiveBarrierConfig()
    if atr_pct is None or (isinstance(atr_pct, float) and np.isnan(atr_pct)) or atr_pct <= 0:
        atr_pct = 2.0
    tp = float(np.clip(cfg.k_tp * atr_pct / 100.0, cfg.min_tp_pct, cfg.max_tp_pct))
    sl = float(np.clip(cfg.k_sl * atr_pct / 100.0, cfg.min_sl_pct, cfg.max_sl_pct))
    return tp, sl


class AdaptiveTripleBarrierLabeler(TripleBarrierLabeler):
    def label_adaptive(self, entry_price: float, future_bars: pd.DataFrame,
                       atr_pct: float, ab_cfg: AdaptiveBarrierConfig = None):
        ab_cfg = ab_cfg or AdaptiveBarrierConfig()
        tp, sl = atr_barriers(atr_pct, ab_cfg)
        return self.label_one(entry_price, future_bars, tp_pct=tp, sl_pct=sl,
                              max_hold=ab_cfg.max_hold), tp, sl


def relabel_dataset_atr(ds: pd.DataFrame, store: MarketDataStore,
                        ab_cfg: AdaptiveBarrierConfig = None,
                        symbols: list = None) -> pd.DataFrame:
    """
    Re-compute triple-barrier labels with ATR-adaptive TP/SL.
    Needs future OHLCV from store for each (symbol, entry_date).
    """
    ab_cfg = ab_cfg or AdaptiveBarrierConfig()
    labeler = AdaptiveTripleBarrierLabeler(CFG)
    rows = []
    symbols = symbols or sorted(ds["symbol"].unique())
    cache = {}

    for sym in symbols:
        sub = ds[ds["symbol"] == sym].copy()
        if len(sub) == 0:
            continue
        # fetch once
        d0 = pd.to_datetime(sub["date"]).min() - pd.Timedelta(days=30)
        d1 = pd.to_datetime(sub["entry_date"]).max() + pd.Timedelta(days=40)
        df = store.get(sym, d0.strftime("%Y-%m-%d"), d1.strftime("%Y-%m-%d"))
        if df is None:
            print(f"  skip {sym}: no data")
            continue
        df = df.sort_values("time").reset_index(drop=True)
        df["time"] = pd.to_datetime(df["time"])
        cache[sym] = df

        for _, r in sub.iterrows():
            entry_date = pd.Timestamp(r["entry_date"]).normalize()
            entry_price = float(r["entry_price"])
            atr_pct = float(r["atr_pct"]) if pd.notna(r.get("atr_pct")) else 2.0
            fut = df[df["time"] >= entry_date].reset_index(drop=True)
            if len(fut) < 2:
                continue
            br, tp, sl = labeler.label_adaptive(entry_price, fut, atr_pct, ab_cfg)
            if br is None:
                continue
            rr = dict(r)
            rr["tb_label"] = br.label
            rr["tb_ret"] = br.ret
            rr["tb_mfe"] = br.mfe
            rr["tb_mae"] = br.mae
            rr["tb_sessions_held"] = br.sessions_held
            rr["tp_pct_used"] = tp
            rr["sl_pct_used"] = sl
            rows.append(rr)

    out = pd.DataFrame(rows)
    print(f"  Relabeled {len(out)} rows with ATR barriers")
    return out


print("ATR-adaptive barrier defined.")


ATR-adaptive barrier defined.


In [30]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 25 — WALK-FORWARD VALIDATION (Spec §53–54)                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

@dataclass
class WalkForwardFold:
    train_start: str
    train_end: str
    test_start: str
    test_end: str


def make_walk_forward_folds(
    start="2022-01-01", end="2025-12-31",
    train_years=2, test_years=1, step_years=1
) -> list:
    """
    Expanding/rolling style annual folds.
    Example:
      train 2022-2023 → test 2024
      train 2023-2024 → test 2025
    """
    folds = []
    y0 = pd.Timestamp(start).year
    y1 = pd.Timestamp(end).year
    t = y0
    while t + train_years <= y1:
        tr_s = f"{t}-01-01"
        tr_e = f"{t + train_years - 1}-12-31"
        te_s = f"{t + train_years}-01-01"
        te_e = f"{t + train_years + test_years - 1}-12-31"
        if pd.Timestamp(te_s) > pd.Timestamp(end):
            break
        if pd.Timestamp(te_e) > pd.Timestamp(end):
            te_e = end
        folds.append(WalkForwardFold(tr_s, tr_e, te_s, te_e))
        t += step_years
    return folds


def run_walk_forward(ds: pd.DataFrame, setups=("PULLBACK",),
                     bt_base: BacktestConfigV2 = None) -> pd.DataFrame:
    """
    For each fold: filter test period, run backtest, collect OOS metrics.
    (Train period reserved for future model fit; rule-based setups use test only.)
    """
    folds = make_walk_forward_folds()
    bt_base = bt_base or BacktestConfigV2(
        allowed_setups=setups,
        use_exit_risk_gate=True,
        max_exit_risk_to_enter=50.0,
        risk_per_trade=0.0075,
        max_open_positions=4,
    )
    results = []
    ds = ds.copy()
    ds["date"] = pd.to_datetime(ds["date"])

    print("═" * 60)
    print("  WALK-FORWARD OOS")
    print("═" * 60)

    for i, fold in enumerate(folds):
        te = ds[(ds["date"] >= fold.test_start) & (ds["date"] <= fold.test_end)]
        if "setup" in te.columns:
            te = te[te["setup"].isin(setups)]
        print(f"\n  Fold {i+1}: TEST {fold.test_start} → {fold.test_end} | n={len(te)}")
        if len(te) < 5:
            print("    skip (too few)")
            continue
        eng = BacktestEngineV2(CFG, bt_base)
        trades = eng.run_from_dataset(te)
        if len(trades) == 0:
            print("    no trades")
            continue
        rep = eng.report(trades)
        rep["fold"] = i + 1
        rep["test_start"] = fold.test_start
        rep["test_end"] = fold.test_end
        rep["n_rows"] = len(te)
        results.append(rep)

    if not results:
        return pd.DataFrame()
    out = pd.DataFrame(results)
    print("\n══ WALK-FORWARD SUMMARY ══")
    cols = [c for c in ["fold", "test_start", "test_end", "n_trades", "win_rate_pct",
                        "profit_factor", "expectancy_ret", "total_return_on_capital",
                        "sum_pnl_net", "max_dd_pnl"] if c in out.columns]
    print(out[cols].to_string(index=False))
    path = RESULTS_DIR / "walk_forward_oos.csv"
    out.to_csv(path, index=False)
    print(f"\n  ✓ Saved → {path}")
    return out


print("Walk-forward defined.")
print("Folds:", [(f.train_start, f.train_end, f.test_start, f.test_end)
                 for f in make_walk_forward_folds()])


Walk-forward defined.
Folds: [('2022-01-01', '2023-12-31', '2024-01-01', '2024-12-31'), ('2023-01-01', '2024-12-31', '2025-01-01', '2025-12-31')]


In [31]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 26 — LOGISTIC PROBABILITY BASELINE (Spec §39–41)                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Target: P(TP_FIRST) under current barrier labels.
# Interpretable baseline only — no tree models until this is calibrated.

try:
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import brier_score_loss, log_loss
    HAS_SK = True
except (ImportError, AttributeError): # Catch both ImportError and AttributeError
    HAS_SK = False
    print("⚠️ scikit-learn not found or incompatible. Attempting to install...")
    try:
        import subprocess, sys
        # Uninstall existing (potentially incompatible) scikit-learn and its dependencies
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "scikit-learn", "-y", "-q"])
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "scipy", "-y", "-q"])
        subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "numpy", "-y", "-q"])
        # Reinstall scikit-learn to get compatible versions
        subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn", "-q"])
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler
        from sklearn.metrics import brier_score_loss, log_loss
        HAS_SK = True
        print("✓ scikit-learn installed successfully.")
    except Exception as e:
        print(f"❌ Failed to install scikit-learn: {e}")
        HAS_SK = False


FEATURE_COLS = [
    "factor_trend", "factor_momentum", "factor_flow", "factor_rs",
    "factor_volatility", "factor_structure",
    "rsi", "cmf", "volume_ratio", "atr_pct", "ema_pct",
    "rs_20d", "bb_pctb", "ret_5d",
]


def _prep_xy(ds: pd.DataFrame, feature_cols=None):
    feature_cols = feature_cols or FEATURE_COLS
    cols = [c for c in feature_cols if c in ds.columns]
    d = ds.dropna(subset=cols + ["tb_label"]).copy()
    d = d[d["tb_label"].isin(["TP_FIRST", "SL_FIRST", "TIMEOUT"])]
    y = (d["tb_label"] == "TP_FIRST").astype(int).values
    X = d[cols].astype(float).values
    return d, X, y, cols


def fit_logistic_oos(ds: pd.DataFrame, setups=("PULLBACK",)) -> dict:
    if not HAS_SK:
        return {}
    d0 = ds.copy()
    d0["date"] = pd.to_datetime(d0["date"])
    if "setup" in d0.columns:
        d0 = d0[d0["setup"].isin(setups)]

    folds = make_walk_forward_folds()
    all_rows = []
    print("═" * 60)
    print(f"  LOGISTIC OOS — setups={setups}")
    print("═" * 60)

    for i, fold in enumerate(folds):
        tr = d0[(d0["date"] >= fold.train_start) & (d0["date"] <= fold.train_end)]
        te = d0[(d0["date"] >= fold.test_start) & (d0["date"] <= fold.test_end)]
        tr, Xtr, ytr, cols = _prep_xy(tr)
        te, Xte, yte, _ = _prep_xy(te)
        if len(tr) < 40 or len(te) < 10 or ytr.sum() < 5:
            print(f"  Fold {i+1}: skip (train={len(tr)}, test={len(te)}, pos={ytr.sum() if len(tr) else 0})")
            continue

        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr)
        Xte_s = scaler.transform(Xte)
        clf = LogisticRegression(max_iter=500, C=0.5, class_weight="balanced")
        clf.fit(Xtr_s, ytr)
        proba = clf.predict_proba(Xte_s)[:, 1]

        # calibration buckets
        te = te.copy()
        te["p_tp"] = proba
        te["y_tp"] = yte

        brier = brier_score_loss(yte, proba)
        # observed frequency by predicted quintile
        try:
            te["pq"] = pd.qcut(te["p_tp"], 4, duplicates="drop")
            cal = te.groupby("pq").agg(n=("y_tp", "count"), pred=("p_tp", "mean"), obs=("y_tp", "mean"))
        except Exception:
            cal = pd.DataFrame()

        print(f"\n  Fold {i+1}: TEST {fold.test_start}→{fold.test_end}")
        print(f"    n_test={len(te)}  base_rate={yte.mean():.3f}  mean_pred={proba.mean():.3f}  brier={brier:.4f}")
        if len(cal):
            print(cal.to_string())

        # top vs bottom predicted
        hi = te[te["p_tp"] >= te["p_tp"].quantile(0.7)]
        lo = te[te["p_tp"] <= te["p_tp"].quantile(0.3)]
        print(f"    TOP30% pred: n={len(hi)} obs_TP={hi['y_tp'].mean():.3f} avg_tb_ret={hi['tb_ret'].mean():.4f}")
        print(f"    BOT30% pred: n={len(lo)} obs_TP={lo['y_tp'].mean():.3f} avg_tb_ret={lo['tb_ret'].mean():.4f}")

        # coefficients (last fold kept)
        coef = pd.Series(clf.coef_[0], index=cols).sort_values(key=abs, ascending=False)
        print("    Top coefficients:")
        print(coef.head(10).to_string())

        all_rows.append({
            "fold": i+1, "n_test": len(te), "base_rate": float(yte.mean()),
            "mean_pred": float(proba.mean()), "brier": float(brier),
            "top_obs": float(hi["y_tp"].mean()) if len(hi) else np.nan,
            "bot_obs": float(lo["y_tp"].mean()) if len(lo) else np.nan,
        })

    return {"folds": pd.DataFrame(all_rows), "last_coef": coef if all_rows else None}


print("Logistic baseline defined.")

⚠️ scikit-learn not found or incompatible. Attempting to install...
❌ Failed to install scikit-learn: Command '['/usr/bin/python3', '-m', 'pip', 'install', 'scikit-learn', '-q']' returned non-zero exit status 1.
Logistic baseline defined.


In [32]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 27 — ONE-SHOT NEXT PHASE PIPELINE                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def run_phase_next(ds: pd.DataFrame = None):
    """
    1) Load ds if needed
    2) Relabel with ATR barriers
    3) Empirical setup stats under ATR barriers
    4) Backtest PULLBACK under ATR labels
    5) Walk-forward OOS
    6) Logistic OOS
    """
    register_vnstock_if_needed()
    store = MarketDataStore(CFG)

    if ds is None:
        files = sorted(DATA_FEATURES.glob("features_v3_*.parquet"))
        if not files:
            print("No features_v3 — run demo_v3 first")
            return None
        ds = pd.read_parquet(files[-1])
        if "setup" not in ds.columns:
            ds = enrich_with_setup_exit(ds)
        print(f"Loaded {files[-1]} → {len(ds)} rows")

    # Guard community window
    try:
        assert_history_window_ok("2022-01-01")
    except Exception as e:
        print(e)

    print("\n[A] Relabel ATR-adaptive barriers...")
    ds_atr = relabel_dataset_atr(ds, store, AdaptiveBarrierConfig(
        k_tp=2.0, k_sl=1.0, min_tp_pct=0.03, max_tp_pct=0.10,
        min_sl_pct=0.015, max_sl_pct=0.05, max_hold=10,
    ))
    if len(ds_atr) == 0:
        print("ATR relabel failed")
        return None
    path = DATA_FEATURES / "features_v3_atr.parquet"
    ds_atr.to_parquet(path, index=False)
    print(f"  Saved → {path}")

    eng = EmpiricalProbabilityEngine(CFG)
    print("\n[B] Setup stats under ATR barriers:")
    st = analyze_setups(ds_atr, eng)
    if len(st):
        cols = [c for c in ["setup","n","confidence","P_TP","P_SL","P_TIMEOUT","avg_ret","EV_simple"] if c in st.columns]
        print(st[cols].to_string(index=False))

    print("\n[C] Backtest PULLBACK (ATR labels + costs):")
    bt_cfg = BacktestConfigV2(
        allowed_setups=("PULLBACK",),
        use_exit_risk_gate=True,
        max_exit_risk_to_enter=50.0,
        risk_per_trade=0.0075,
        max_open_positions=4,
    )
    # Override barrier ret already in ds_atr
    be = BacktestEngineV2(CFG, bt_cfg)
    trades = be.run_from_dataset(ds_atr[ds_atr["setup"]=="PULLBACK"] if "setup" in ds_atr.columns else ds_atr)
    rep = be.report(trades)

    print("\n[D] Walk-forward OOS (PULLBACK):")
    wf = run_walk_forward(ds_atr, setups=("PULLBACK",), bt_base=bt_cfg)

    print("\n[E] Logistic P(TP) OOS (PULLBACK):")
    logit = fit_logistic_oos(ds_atr, setups=("PULLBACK",))

    meta = ResearchRunMetadata(
        run_id=make_run_id(),
        timestamp=datetime.now().isoformat(),
        feature_version=CFG.feature_version + "+atr_wf_logit",
        model_version="logistic_v1",
        config_version=CFG.config_version,
        data_start="2022-01-01", data_end="2025-12-31",
        universe=sorted(ds_atr["symbol"].unique().tolist()),
        n_symbols=ds_atr["symbol"].nunique(),
        execution_model=CFG.execution_model,
        tp_pct=-1.0,  # adaptive
        sl_pct=-1.0,
        max_holding_sessions=10,
        fee_buy=CFG.fee_buy, fee_sell=CFG.fee_sell, tax_sell=CFG.tax_sell,
        random_seed=CFG.random_seed,
        notes="ATR barriers + WF + logistic baseline",
    )
    meta.save()
    print("\n✓ Phase next complete.")
    return {"ds_atr": ds_atr, "trades": trades, "wf": wf, "logit": logit, "rep": rep}


print("run_phase_next() ready.")
print("→ result = run_phase_next()          # loads parquet")
print("→ result = run_phase_next(ds)        # if ds in memory")


run_phase_next() ready.
→ result = run_phase_next()          # loads parquet
→ result = run_phase_next(ds)        # if ds in memory


---

## Backtest v2 — honest read

| Config | n | WR | PF | Expectancy | On capital | Verdict |
|--------|---|----|----|------------|------------|---------|
| PULLBACK only | 179 | 44.7% | **1.03** | +0.06% | **+1.27%** | Thin edge after costs |
| PULLBACK+BREAKOUT | 290 | 44.5% | 0.99 | ~0 | -0.46% | Flat |
| All setups | 1000 | 40.5% | 0.80 | -0.33% | **-40%** | Destroyed by NONE |

**PULLBACK is the only candidate.** Edge is small; costs (~10.8M VND) ≈ same order as gross profit. Max DD >> net profit → **not deployable** yet.

Regime quirk: bear-period pullbacks look better (mean-reversion). Sideways hurts.

### What this phase does (Spec-aligned)

1. **§14** ATR-adaptive TP/SL (replace fixed 6%/3%)
2. **§53** Walk-forward OOS folds
3. **§39–41** Logistic regression baseline for P(TP_FIRST) + crude calibration
4. Re-backtest PULLBACK under ATR labels

### Run

```python
result = run_phase_next()   # or run_phase_next(ds)
```

Paste:
- Setup stats under ATR barriers
- Backtest PULLBACK ATR report
- Walk-forward summary table
- Logistic fold TOP30% vs BOT30%

Expect: ATR barriers may improve PF if fixed 6% TP was unrealistic for low-ATR names; if still PF≈1, edge is not robust.


In [33]:
# result = run_phase_next()  # full ATR + WF + logistic pipeline


In [34]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 28 — BARRIER SENSITIVITY + TIGHT SETUP FILTER                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Findings so far (honest):
#   - Fixed 6/3: PULLBACK marginal → negative when window includes 2025-26
#   - ATR k_tp=2, k_sl=1: SL too tight → P_SL ~58%, PF < 1
#   - WF OOS both folds lose
# Next: sensitivity grid + PULLBACK ∩ CẠN VOL (best VSA empirically)

def barrier_sensitivity(ds: pd.DataFrame, store: MarketDataStore,
                        setup_filter="PULLBACK",
                        grid=None) -> pd.DataFrame:
    """
    Small grid search on ATR multipliers — report empirical stats only.
    NOT for picking the max Sharpe on full sample as "the" strategy
    (Spec §57–58: look for stable region, not single optimum).
    """
    if grid is None:
        grid = [
            AdaptiveBarrierConfig(k_tp=1.5, k_sl=1.0, min_tp_pct=0.025, min_sl_pct=0.02, max_hold=10),
            AdaptiveBarrierConfig(k_tp=2.0, k_sl=1.0, min_tp_pct=0.03,  min_sl_pct=0.02, max_hold=10),
            AdaptiveBarrierConfig(k_tp=2.0, k_sl=1.5, min_tp_pct=0.03,  min_sl_pct=0.025, max_hold=10),
            AdaptiveBarrierConfig(k_tp=2.5, k_sl=1.5, min_tp_pct=0.035, min_sl_pct=0.025, max_hold=12),
            AdaptiveBarrierConfig(k_tp=3.0, k_sl=1.5, min_tp_pct=0.04,  min_sl_pct=0.025, max_hold=15),
            # fixed reference
        ]

    eng = EmpiricalProbabilityEngine(CFG)
    rows = []
    base = ds.copy()
    if setup_filter and "setup" in base.columns:
        base = base[base["setup"] == setup_filter]

    print(f"Sensitivity on setup={setup_filter} | n_base={len(base)}")
    for i, ab in enumerate(grid):
        print(f"  [{i+1}/{len(grid)}] k_tp={ab.k_tp} k_sl={ab.k_sl} hold={ab.max_hold}...", end=" ")
        rel = relabel_dataset_atr(base, store, ab, symbols=sorted(base["symbol"].unique()))
        if len(rel) == 0:
            print("empty")
            continue
        s = eng.summarize(rel)
        s["k_tp"] = ab.k_tp
        s["k_sl"] = ab.k_sl
        s["max_hold"] = ab.max_hold
        s["min_tp"] = ab.min_tp_pct
        s["min_sl"] = ab.min_sl_pct
        # rough cost-aware EV: assume 0.4% roundtrip cost drag
        s["EV_after_cost"] = (s.get("EV_simple") or 0) - 0.004
        rows.append(s)
        print(f"P_TP={s['P_TP']:.3f} P_SL={s['P_SL']:.3f} EV={s['EV_simple']:.4f}")

    out = pd.DataFrame(rows)
    if len(out):
        cols = ["k_tp","k_sl","max_hold","n","P_TP","P_SL","P_TIMEOUT","avg_ret","EV_simple","EV_after_cost"]
        cols = [c for c in cols if c in out.columns]
        print("\n══ SENSITIVITY TABLE ══")
        print(out[cols].to_string(index=False))
        path = RESULTS_DIR / "barrier_sensitivity.csv"
        out.to_csv(path, index=False)
        print(f"Saved → {path}")
    return out


def filter_pullback_canvol(ds: pd.DataFrame) -> pd.DataFrame:
    """Tight setup: PULLBACK + CẠN VOL VSA (best empirical VSA)."""
    d = ds.copy()
    if "setup" not in d.columns or "vsa_state" not in d.columns:
        return d.iloc[0:0]
    return d[(d["setup"] == "PULLBACK") & (d["vsa_state"] == "CẠN VOL")]


def filter_pullback_quality(ds: pd.DataFrame) -> pd.DataFrame:
    """PULLBACK + flow not weak + not high exit_risk."""
    d = ds.copy()
    m = d["setup"] == "PULLBACK"
    if "factor_flow" in d.columns:
        m &= d["factor_flow"] >= 45
    if "exit_risk" in d.columns:
        m &= d["exit_risk"] <= 40
    if "cmf" in d.columns:
        m &= d["cmf"] >= -0.05
    return d[m]


def run_tight_setup_tests(ds: pd.DataFrame = None):
    register_vnstock_if_needed()
    store = MarketDataStore(CFG)
    if ds is None:
        files = sorted(DATA_FEATURES.glob("features_v3_*.parquet"))
        # prefer non-atr original for relabel flexibility
        files = [f for f in files if "atr" not in f.name] or files
        ds = pd.read_parquet(files[-1])
        if "setup" not in ds.columns:
            ds = enrich_with_setup_exit(ds)
        print(f"Loaded {files[-1]} → {len(ds)} rows")

    # 1) Barrier sensitivity on PULLBACK
    print("\n[1] Barrier sensitivity (PULLBACK)...")
    sens = barrier_sensitivity(ds, store, "PULLBACK")

    # Pick a stabler config: prefer EV_after_cost closest to max among mid grid
    # Default recommendation if grid run: k_tp=2.5, k_sl=1.5, hold=12
    ab_rec = AdaptiveBarrierConfig(k_tp=2.5, k_sl=1.5, min_tp_pct=0.035,
                                   min_sl_pct=0.025, max_tp_pct=0.12, max_sl_pct=0.06,
                                   max_hold=12)

    print("\n[2] Relabel recommended barriers k_tp=2.5 k_sl=1.5 hold=12...")
    ds_r = relabel_dataset_atr(ds, store, ab_rec)
    ds_r.to_parquet(DATA_FEATURES / "features_v3_atr_rec.parquet", index=False)

    eng = EmpiricalProbabilityEngine(CFG)
    print("\n[3] Setup stats (recommended barriers):")
    st = analyze_setups(ds_r, eng)
    if len(st):
        print(st[[c for c in ["setup","n","P_TP","P_SL","avg_ret","EV_simple"] if c in st.columns]].to_string(index=False))

    # Tight filters
    for name, fn in [
        ("PULLBACK raw", lambda d: d[d["setup"]=="PULLBACK"]),
        ("PULLBACK ∩ CẠN VOL", filter_pullback_canvol),
        ("PULLBACK quality", filter_pullback_quality),
    ]:
        sub = fn(ds_r)
        print(f"\n[4] {name}: n={len(sub)}")
        if len(sub) < 15:
            print("  too few — skip backtest")
            continue
        print("  Empirical:", eng.summarize(sub))
        bt = BacktestConfigV2(
            allowed_setups=tuple(sub["setup"].unique()) if "setup" in sub.columns else ("PULLBACK",),
            use_exit_risk_gate=False,  # already filtered
            risk_per_trade=0.005,
            max_open_positions=3,
        )
        # Force only these rows by temporarily marking others none — simpler: pass sub directly
        be = BacktestEngineV2(CFG, bt)
        # BacktestEngine filters by setup name; for CẠN VOL we pass only sub rows
        tr = be.run_from_dataset(sub)
        be.report(tr)

    # WF on recommended barriers + PULLBACK quality
    print("\n[5] Walk-forward PULLBACK quality @ recommended barriers...")
    pq = filter_pullback_quality(ds_r)
    if len(pq) >= 30:
        # tag all as PULLBACK for engine filter
        pq = pq.copy()
        pq["setup"] = "PULLBACK"
        wf = run_walk_forward(pq, setups=("PULLBACK",), bt_base=BacktestConfigV2(
            allowed_setups=("PULLBACK",), use_exit_risk_gate=False,
            risk_per_trade=0.005, max_open_positions=3,
        ))
    else:
        print(f"  n={len(pq)} insufficient for WF")
        wf = None

    # Logistic with clearer skip reasons
    print("\n[6] Logistic retry...")
    if HAS_SK:
        fit_logistic_oos(ds_r, setups=("PULLBACK",))
    else:
        print("  sklearn missing — pip install scikit-learn")

    return {"sens": sens, "ds_r": ds_r, "wf": wf}


print("run_tight_setup_tests() ready.")
print("→ out = run_tight_setup_tests()  # or run_tight_setup_tests(ds)")


run_tight_setup_tests() ready.
→ out = run_tight_setup_tests()  # or run_tight_setup_tests(ds)


---

## Research verdict so far (Spec §82)

> System must think in probability distributions, not "stock will go up."

| Evidence | Result |
|----------|--------|
| Fixed 6/3 PULLBACK (2022–2024) | Tiny +EV, disappeared with 2025–26 |
| ATR k_sl=1 | **Worse** — noise stops |
| Walk-forward 2024 & 2025 | **Both OOS negative** |
| NONE / broad universe | Large losses after costs |
| CẠN VOL VSA | Best univariate hint, still modest |
| ExitRisk heuristic | **Inverted** vs future returns |

**Current PULLBACK definition + cost model does NOT clear the bar for a deployable edge.**

### This cell does
1. Barrier **sensitivity grid** (stable region, not cherry-pick)
2. Recommended wider SL (`k_sl=1.5`, min SL 2.5%)
3. Tighter filters: `PULLBACK ∩ CẠN VOL`, flow/CMF quality
4. Re-backtest + WF on quality subset
5. Logistic again with sklearn check

### Run
```python
out = run_tight_setup_tests()
# or out = run_tight_setup_tests(ds)
```

If after this PF still < 1.1 and WF still red → **stop adding indicators**; either:
- change hypothesis (mean-reversion bear-only, longer hold, sector filter), or
- accept no edge on this 10-name liquid universe under realistic costs.


In [35]:
# out = run_tight_setup_tests()  # barrier sensitivity + tight filters + WF


In [36]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 29 — UNIVERSE (user watchlist)                                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

VN100 = ['FPT', 'VCB', 'HPG', 'TCB', 'ACB', 'BID', 'MWG', 'SSI', 'VND', 'CTD', 'VHM', 'VIC', 'VRE', 'PNJ', 'MSN',
                   'GAS', 'PLX', 'SAB', 'STB', 'SHB', 'LPB', 'VPB', 'MBB', 'HDB', 'TPB', 'OCB', 'SSB', 'VIB', 'CTG', 'POW',
                   'GVR', 'DGC', 'DPM', 'BSR', 'PVD', 'PVT', 'VCG', 'VSC', 'HSG', 'NKG', 'BCM', 'KDH', 'DXG', 'NLG', 'TCH',
                   'DIG', 'PDR', 'KBC', 'CRE', 'VPI', 'HDG', 'AGG', 'HT1', 'AAA', 'DBC', 'NVL', 'VHC', 'MSH', 'CII', 'VGI',
                   'ANV', 'SMC', 'HAG', 'DLG', 'BFC', 'PTB', 'TLH', 'PHR', 'DCM', 'LAS', 'VIX', 'MSR', 'GEL', 'HUT', 'GEG',
                   'GEX', 'FTS', 'HCM', 'CTS', 'MBS', 'VCI', 'ORS', 'BSI', 'BAF', 'BMP', 'BVH', 'BWE', 'CMG', 'CTR', 'DGW',
                   'DSE', 'DXS', 'EIB', 'EVF', 'FRT', 'GEE', 'GMD', 'HDC', 'HHV', 'IMP', 'KDC', 'KOS', 'MSB', 'NAB', 'NT2',
                   'PAN', 'PC1', 'REE', 'SBT', 'SCS', 'SIP', 'SJS', 'SZC', 'VGC', 'VJC', 'VNM', 'VPL', 'CEO', 'SHS', 'PVS',
                   'MST', 'VC3', 'IDC', 'VTZ', 'VFS', 'PVC', 'C69', 'TNG', 'VGS', 'KSF', 'PVB', 'BVS', 'CTP', 'FID', 'VTV',
                   'PLC', 'OCH', 'PVI', 'NTP', 'IPA', 'DXP', 'VIW', 'TVN', 'OIL', 'ACV', 'HNG', 'BVB', 'FOX', 'DRI', 'DDV',
                   'AAS', 'VEA', 'TV1', 'ABB', 'VGT', 'QTP', 'MZG', 'PHP', 'SBS', 'STH', 'F88', 'HNM', 'FOC', 'DSH', 'BMS',
                   'QNS', 'TOS', 'XMC', 'BNA', 'VC7', 'VC2', 'DTD', 'UNI', 'NBC', 'NAG', 'CAP', 'PSI', 'VGP', 'PPT', 'PCH',
                   'NVB', 'L40', 'APS', 'VHE', 'API', 'PVG', 'IDJ', 'LDP', 'CMS', 'DST', 'AAV', 'AMS', 'BIG', 'VTD', 'G36',
                   'DSH', 'DDB', 'CLI', 'PIV', 'CLX', 'TTG', 'DVN', 'PXL', 'MPC', 'ABW', 'PGB', 'VBB', 'NCG', 'ALC', 'KCB',
                   'GCF', 'KLB', 'VAB', 'C32', 'CIG', 'HSL', 'RYG', 'VTO', 'HII', 'TCM', 'PPC', 'ELC', 'TCI', 'SHI', 'DAH',
                   'LDG', 'HQC', 'HHP', 'SCR', 'LCG', 'QCG', 'TTH', 'NRC', 'TVC', 'TIG', 'SVN', 'DL1', 'KIP', 'CTX', 'APF',
                   'SBB', 'VNP', 'HVN', 'TLG']

# Liquid core for fast daily scan / research expansion (rate-limit friendly)
LIQUID_CORE = [
    'FPT','VCB','HPG','TCB','ACB','BID','MWG','SSI','VHM','VIC','VRE','PNJ','MSN',
    'GAS','PLX','STB','SHB','VPB','MBB','HDB','TPB','CTG','POW','GVR','DGC',
    'VNM','VJC','REE','BMP','HCM','VCI','DGW','FRT','PC1','KDH','NLG','DXG',
]

print(f"VN100 watchlist: {len(VN100)} symbols | LIQUID_CORE: {len(LIQUID_CORE)}")


VN100 watchlist: 244 symbols | LIQUID_CORE: 37


In [37]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 30 — ALPHA RANKER + EXIT RANKER + DAILY SCANNER (Spec §59–62)     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Practical layer aligned with user goal:
#   BUY  → stocks with favorable upward continuation context
#   EXIT → stocks with deteriorating trend / distribution risk
#
# IMPORTANT: rankings are CONTEXT SCORES informed by research so far.
# They are NOT a claim of proven OOS alpha. Research showed thin/no edge
# after costs on naive pullback — scanner surfaces candidates for human review.

class AlphaRanker:
    """
    Cross-sectional rank for BUY candidates.
    Higher score = relatively better long context today.
    """

    def score_row(self, f: dict, regime: dict = None) -> dict:
        trend = float(f.get("factor_trend") or 50)
        mom   = float(f.get("factor_momentum") or 50)
        flow  = float(f.get("factor_flow") or 50)
        rs    = float(f.get("factor_rs") or 50)
        vola  = float(f.get("factor_volatility") or 50)
        struct= float(f.get("factor_structure") or 50)
        setup = str(f.get("setup") or "NONE")
        vsa   = str(f.get("vsa_state") or "—")
        cmf   = float(f.get("cmf") or 0)
        above50 = int(f.get("above_sma50") or 0)
        atr_pct = float(f.get("atr_pct") or 2)

        # Base: equal-ish factor blend (diagnostic composite, not magic)
        alpha = 0.25*trend + 0.15*mom + 0.25*flow + 0.20*rs + 0.10*struct + 0.05*vola

        # Setup / VSA tilts from empirical direction (modest, not hard score 8.5)
        if setup == "PULLBACK" and above50:
            alpha += 4
        if setup == "BREAKOUT" and flow >= 55:
            alpha += 3
        if setup == "ACCUMULATION":
            alpha += 2
        if vsa == "CẠN VOL":
            alpha += 3
        if vsa == "NỔ VOL" and cmf > 0:
            alpha += 2
        if setup == "DISTRIBUTION" or vsa == "PHÂN PHỐI":
            alpha -= 8
        if vsa == "CẠN CẦU":
            alpha -= 4

        # Liquidity / tradability
        avg_val = float(f.get("avg_val_20d") or 0)
        if avg_val < 5e6:
            alpha -= 10
        elif avg_val < 10e6:
            alpha -= 3

        # Regime context
        if regime:
            if regime.get("trend") == "bear":
                alpha -= 5  # fewer longs in confirmed bear
            elif regime.get("trend") == "bull":
                alpha += 2

        alpha = float(np.clip(alpha, 0, 100))

        # Soft probability proxy (NOT calibrated model) — map score to 0.15–0.40
        p_proxy = 0.15 + 0.25 * (alpha / 100.0)

        return {
            "alpha_score": round(alpha, 2),
            "p_proxy": round(p_proxy, 3),
            "setup": setup,
            "vsa_state": vsa,
            "factor_trend": round(trend, 1),
            "factor_flow": round(flow, 1),
            "factor_rs": round(rs, 1),
            "factor_momentum": round(mom, 1),
        }


class ExitRanker:
    """Wrap ExitRiskEngine + rank for portfolio monitoring."""

    def __init__(self):
        self.engine = ExitRiskEngine(CFG)

    def score_row(self, f: dict) -> dict:
        ex = self.engine.score(f)
        return {
            "exit_risk": ex["exit_risk"],
            "exit_band": ex["exit_band"],
            "exit_action": ex["exit_action"],
            "exit_reasons": "; ".join(ex["exit_reasons"][:5]),
        }


class DailyScanner:
    """
    End-of-day style scan:
      for each symbol → features as of latest bar → rank BUY / EXIT
    """

    def __init__(self, store: MarketDataStore = None, cfg: ResearchConfig = CFG):
        self.store = store or MarketDataStore(cfg)
        self.cfg = cfg
        self.alpha = AlphaRanker()
        self.exit = ExitRanker()
        self.setup_eng = SetupEngine(cfg)
        self.regime_eng = MarketRegimeEngineV2(self.store, cfg)

    def _features_one(self, symbol: str, df_vni: pd.DataFrame) -> Optional[dict]:
        df = self.store.get_as_of(symbol, datetime.today().strftime("%Y-%m-%d"),
                                  lookback_bars=self.cfg.lookback_features)
        if df is None or len(df) < self.cfg.min_history_bars:
            return None
        df_bmk = None
        if df_vni is not None:
            last = df["time"].iloc[-1]
            df_bmk = df_vni[df_vni["time"] <= last].tail(self.cfg.lookback_features)
        feats = compute_features_v2(df, self.cfg, df_bmk=df_bmk)
        if not feats:
            return None
        # attach setup + exit
        setup = self.setup_eng.classify(feats)
        feats.update(setup)
        feats.update(self.exit.score_row(feats))
        feats["symbol"] = symbol
        feats["asof"] = df["time"].iloc[-1]
        feats["close"] = float(df["close"].iloc[-1])
        return feats

    def scan(self, symbols: List[str], top_n: int = 300,
             min_avg_val: float = 8e6) -> dict:
        print("═" * 60)
        print(f"  DAILY SCANNER | {len(symbols)} symbols | {datetime.now():%Y-%m-%d %H:%M}")
        print("═" * 60)

        # Benchmark + regime once
        end = datetime.today().strftime("%Y-%m-%d")
        start = (datetime.today() - timedelta(days=400)).strftime("%Y-%m-%d")
        df_vni = self.store.get("VNINDEX", start, end)
        regime = self.regime_eng.detect(df_vni.tail(60) if df_vni is not None else None)
        print(f"  Market: {regime.get('label')} | VNI={regime.get('vnindex')} mom5={regime.get('mom5')}%")

        rows = []
        for i, sym in enumerate(symbols):
            print(f"  [{i+1}/{len(symbols)}] {sym}...", end="\r")
            try:
                f = self._features_one(sym, df_vni)
                if f is None:
                    continue
                if float(f.get("avg_val_20d") or 0) < min_avg_val:
                    continue
                a = self.alpha.score_row(f, regime)
                f.update(a)
                rows.append(f)
            except Exception as e:
                continue
            time.sleep(self.cfg.delay_sec * 0.4)

        if not rows:
            print("\n  No rows")
            return {"regime": regime, "buy": pd.DataFrame(), "exit": pd.DataFrame()}

        df = pd.DataFrame(rows)
        df["alpha_pct"] = df["alpha_score"].rank(pct=True) * 100

        buy = df.sort_values("alpha_score", ascending=False).head(top_n)
        # EXIT monitor: high exit risk among names that still look "uptrend-ish"
        exit_mon = df[df["exit_risk"] >= 40].sort_values("exit_risk", ascending=False).head(top_n)

        print(f"\n\n  MARKET STATUS")
        print(f"  {'─'*50}")
        print(f"  Regime     : {regime.get('label')}")
        print(f"  Breadth    : (full breadth needs full-universe MA scan — partial)")
        print(f"  Liquidity  : {regime.get('liquidity')}")
        print(f"  Risk app.  : {regime.get('risk_appetite')}")

        print(f"\n  TOP BUY ALPHA (review — not auto-trade)")
        print(f"  {'─'*50}")
        cols_b = [c for c in ["symbol","close","setup","vsa_state","alpha_score","alpha_pct",
                              "p_proxy","factor_trend","factor_flow","factor_rs","exit_risk"] if c in buy.columns]
        if len(buy):
            print(buy[cols_b].to_string(index=False))
        else:
            print("  (empty)")

        print(f"\n  EXIT MONITOR (deterioration)")
        print(f"  {'─'*50}")
        cols_e = [c for c in ["symbol","close","setup","exit_risk","exit_band","exit_action",
                              "exit_reasons","factor_trend","factor_flow","factor_rs"] if c in exit_mon.columns]
        if len(exit_mon):
            print(exit_mon[cols_e].to_string(index=False))
        else:
            print("  (none flagged)")

        # persist
        ts = datetime.now().strftime("%Y%m%d_%H%M")
        buy_path = RESULTS_DIR / f"scan_buy_{ts}.csv"
        exit_path = RESULTS_DIR / f"scan_exit_{ts}.csv"
        buy.to_csv(buy_path, index=False)
        exit_mon.to_csv(exit_path, index=False)
        print(f"\n  ✓ Saved {buy_path.name} | {exit_path.name}")

        return {"regime": regime, "buy": buy, "exit": exit_mon, "all": df}


print("DailyScanner / AlphaRanker / ExitRanker defined.")


DailyScanner / AlphaRanker / ExitRanker defined.


In [38]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 31 — RUN DAILY SCAN                                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def run_daily_scan(symbols=None, top_n=15, full=False):
    """
    full=False → LIQUID_CORE (~40 names, ~few minutes, community-friendly)
    full=True  → entire VN100 list (slow under 60 req/min)
    """
    register_vnstock_if_needed()
    if symbols is None:
        symbols = VN100 if full else LIQUID_CORE
    # dedupe preserve order
    seen = set()
    symbols = [s for s in symbols if not (s in seen or seen.add(s))]

    scanner = DailyScanner(MarketDataStore(CFG), CFG)
    return scanner.scan(symbols, top_n=top_n)


print("run_daily_scan() ready")
print("→ out = run_daily_scan()           # LIQUID_CORE fast")
print("→ out = run_daily_scan(full=True)  # full VN100 (slow)")
print("→ out = run_daily_scan(symbols=['FPT','HPG',...])")


run_daily_scan() ready
→ out = run_daily_scan()           # LIQUID_CORE fast
→ out = run_daily_scan(full=True)  # full VN100 (slow)
→ out = run_daily_scan(symbols=['FPT','HPG',...])


---

## Research status (honest) + what scanner is

### What the research **rejected** (so far)
- Naive “score high → buy” from legacy v5.5 style
- Fixed 9% TP / fixed 6–3 barriers as universal rules  
- Hard VSA scores (8.5 / 1.5)
- PULLBACK as a **standalone cost-aware profitable system** on 10 liquid names (OOS mixed/negative)
- ExitRisk heuristic is **miscalibrated** (EXIT band had higher forward returns — treat reasons as flags, not auto-sell)

### What still has **directional** value (features, not a finished system)
| Feature | Note |
|---------|------|
| CẠN VOL | Best VSA bucket empirically |
| factor_flow + CMF | Separates distribution-ish context |
| PULLBACK quality | Slightly better than raw PULLBACK |
| Regime split | Bear pullbacks ≠ bull pullbacks |
| Cross-sectional rank | Better than absolute magic score |

### Daily Scanner (this phase)
Implements Spec §59–62 in **practical** form:

```
MARKET STATUS
TOP BUY ALPHA   ← rank by factors + setup/VSA tilts + liquidity
EXIT MONITOR    ← deterioration flags for names you hold
```

**Use as decision support**, not autopilot. Edge after fees was not proven OOS.

### Recommended workflow
1. `out = run_daily_scan()` each session (core liquid)
2. Human filter: news, foreign room, ceiling/floor, portfolio concentration
3. Expand research later: more names, sector engine, calibrated logistic (install sklearn), bear-only book

### Run
```python
out = run_daily_scan()                 # ~40 liquid names
# out = run_daily_scan(full=True)      # full VN100 — slow on community API
```


In [39]:
# out = run_daily_scan()  # daily BUY alpha + EXIT monitor on LIQUID_CORE


In [40]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 32 — LEVEL BUILDER (Entry / SL / TP)  Spec §14, §42–43            ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Targets derived from ATR + structure — NOT fixed 9%.
# Default geometry (configurable):
#   SL  = entry - 1.2 * ATR
#   TP1 = entry + 2.0 * ATR   (partial)
#   TP2 = entry + 3.5 * ATR
# Soft structure anchors: EMA20, SMA50 when they improve R:R.

@dataclass
class LevelConfig:
    sl_atr_mult: float = 1.2
    tp1_atr_mult: float = 2.0
    tp2_atr_mult: float = 3.5
    min_sl_pct: float = 0.02
    max_sl_pct: float = 0.07
    min_tp1_pct: float = 0.03
    entry_pullback_atr: float = 0.35   # limit entry below close
    partial_tp_pct: float = 0.30       # take 30% at TP1


class LevelBuilder:
    def __init__(self, lcfg: LevelConfig = None, cfg: ResearchConfig = CFG):
        self.lcfg = lcfg or LevelConfig()
        self.cfg = cfg

    def build(self, f: dict, mode: str = "new") -> dict:
        """
        mode:
          new  → suggest limit entry (slight pullback) for fresh buys
          hold → use last close as reference (already in position)
        """
        close = float(f.get("close") or f.get("entry_price") or 0)
        atr_abs = float(f.get("atr_abs") or 0)
        atr_pct = float(f.get("atr_pct") or 2.0)
        ema20 = f.get("ema20")
        sma50 = f.get("sma50")
        setup = str(f.get("setup") or "NONE")

        if close <= 0:
            return {}

        if atr_abs <= 0:
            atr_abs = close * atr_pct / 100.0

        # --- Entry ---
        if mode == "new":
            # Prefer buying dip toward EMA20 if close is extended
            entry = close
            if ema20 and float(ema20) < close:
                pull = min(close - float(ema20), self.lcfg.entry_pullback_atr * atr_abs)
                entry = close - max(pull, 0)
            else:
                entry = close - self.lcfg.entry_pullback_atr * atr_abs * 0.5
            entry = max(entry, close * 0.97)  # never suggest >3% below close as default
            entry_note = "Limit gần giá / EMA20"
            if setup == "PULLBACK":
                entry_note = "Pullback — ưu tiên limit về hỗ trợ ngắn"
            elif setup == "BREAKOUT":
                entry = close  # breakout: less chase-discount
                entry_note = "Breakout — entry gần close (không mua đuổi sâu)"
        else:
            entry = close
            entry_note = "Đang cầm — ref = close"

        # --- Stop ---
        sl_dist = max(self.lcfg.sl_atr_mult * atr_abs, self.lcfg.min_sl_pct * entry)
        sl_dist = min(sl_dist, self.lcfg.max_sl_pct * entry)
        # structure: if SMA50 below and closer than ATR stop, use slightly under SMA50
        sl = entry - sl_dist
        if sma50 and float(sma50) < entry:
            struct_sl = float(sma50) * 0.995
            if struct_sl < entry and (entry - struct_sl) <= sl_dist * 1.35:
                sl = min(sl, struct_sl)
        sl_pct = (entry - sl) / entry

        # --- Targets ---
        tp1 = entry + max(self.lcfg.tp1_atr_mult * atr_abs, self.lcfg.min_tp1_pct * entry)
        tp2 = entry + self.lcfg.tp2_atr_mult * atr_abs
        # R:R
        risk = entry - sl
        rr1 = (tp1 - entry) / risk if risk > 0 else 0
        rr2 = (tp2 - entry) / risk if risk > 0 else 0

        return {
            "ref_close": round(close, 2),
            "entry_suggest": round(entry, 2),
            "entry_note": entry_note,
            "sl": round(sl, 2),
            "sl_pct": round(sl_pct * 100, 2),
            "tp1": round(tp1, 2),
            "tp1_pct": round((tp1 / entry - 1) * 100, 2),
            "tp2": round(tp2, 2),
            "tp2_pct": round((tp2 / entry - 1) * 100, 2),
            "rr_tp1": round(rr1, 2),
            "rr_tp2": round(rr2, 2),
            "partial_tp_pct": self.lcfg.partial_tp_pct,
            "atr_abs": round(atr_abs, 2),
        }


print("LevelBuilder defined.")


LevelBuilder defined.


In [41]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 33 — RECOMMENDATION ENGINE (New vs Holder)                        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Two audiences (user request):
#   1) Chưa có mã  → NEW_BUY / WAIT / AVOID
#   2) Đang cầm mã → HOLD / SCALE_IN / REDUCE / EXIT
#
# Actions are RULE-BASED decision support from factors + levels + exit_risk.
# Not a guarantee of profit (research OOS was mixed).

class RecommendationEngine:
    def __init__(self, cfg: ResearchConfig = CFG):
        self.cfg = cfg
        self.levels = LevelBuilder()
        self.alpha = AlphaRanker()
        self.exit_r = ExitRanker()

    def for_new_buyer(self, f: dict, regime: dict = None) -> dict:
        a = self.alpha.score_row(f, regime)
        ex = self.exit_r.score_row(f)
        lv = self.levels.build(f, mode="new")
        score = a["alpha_score"]
        setup = a.get("setup", "NONE")
        exit_risk = ex["exit_risk"]

        reasons = []
        if setup in ("PULLBACK", "BREAKOUT", "ACCUMULATION"):
            reasons.append(f"Setup {setup}")
        if a.get("vsa_state") == "CẠN VOL":
            reasons.append("VSA CẠN VOL")
        if float(f.get("factor_flow") or 0) >= 60:
            reasons.append("Flow mạnh")
        if float(f.get("factor_trend") or 0) >= 70:
            reasons.append("Trend factor cao")
        if float(f.get("above_sma50") or 0) == 1:
            reasons.append("Trên SMA50")
        if exit_risk >= 50:
            reasons.append("Exit risk cao — không mở mới")

        # Decision
        if exit_risk >= 55 or setup == "DISTRIBUTION":
            action, conf = "AVOID", "HIGH"
        elif score >= 72 and exit_risk < 35 and lv.get("rr_tp1", 0) >= 1.3:
            action, conf = "NEW_BUY", "MEDIUM"
        elif score >= 65 and exit_risk < 45:
            action, conf = "WAIT_DIP", "MEDIUM"  # watchlist, buy on suggested entry
        else:
            action, conf = "WAIT", "LOW"

        # Size hint (fractional risk) — only if NEW_BUY / WAIT_DIP
        size_hint = "—"
        if action in ("NEW_BUY", "WAIT_DIP") and lv.get("sl_pct", 0) > 0:
            risk_pct = self.cfg.risk_per_trade * 100
            size_hint = f"Risk {risk_pct:.2f}% NAV / SL {lv['sl_pct']:.1f}% → size ≈ {risk_pct/lv['sl_pct']*100:.0f}% NAV (cap {self.cfg.max_position_pct*100:.0f}%)"

        return {
            "audience": "NEW",
            "action": action,
            "confidence": conf,
            "reasons": "; ".join(reasons) if reasons else "Yếu tố trung tính",
            "size_hint": size_hint,
            **a, **{k: ex[k] for k in ("exit_risk", "exit_band")},
            **lv,
        }

    def for_holder(self, f: dict, cost_basis: float = None, regime: dict = None) -> dict:
        """cost_basis optional — if provided, show floating P&L vs suggest levels."""
        a = self.alpha.score_row(f, regime)
        ex = self.exit_r.score_row(f)   # dict kết quả exit
        lv = self.levels.build(f, mode="hold")
        close = float(f.get("close") or 0)
        exit_risk = ex["exit_risk"]
        trend = float(f.get("factor_trend") or 50)
        flow = float(f.get("factor_flow") or 50)
        setup = str(f.get("setup") or "NONE")

        _er = ex.get("exit_reasons")
        if isinstance(_er, list):
            reasons = [str(x) for x in _er if x]
        elif isinstance(_er, str) and _er:
            reasons = list(filter(None, _er.split("; ")))
        else:
            reasons = []

        if exit_risk >= 80 or (setup == "DISTRIBUTION" and flow < 40):
            action, conf = "EXIT", "HIGH"
        elif exit_risk >= 60:
            action, conf = "REDUCE", "MEDIUM"
        elif exit_risk >= 40 and flow < 45:
            action, conf = "WATCH", "MEDIUM"
        elif trend >= 70 and flow >= 55 and exit_risk < 30:
            action, conf = "HOLD_OR_SCALE_IN", "MEDIUM"
            reasons.append("Trend+Flow còn hỗ trợ")
        else:
            action, conf = "HOLD", "MEDIUM"
            if not reasons:
                reasons.append("Chưa có tín hiệu hỏng rõ")

        pnl = None
        if cost_basis and cost_basis > 0 and close > 0:
            pnl = round((close / cost_basis - 1) * 100, 2)

        # Trail SL suggestion for holders: max(structural SL, entry-based)
        trail_sl = lv.get("sl")
        if cost_basis and cost_basis > 0:
            # never suggest SL above basis in a way that locks tiny loss incorrectly —
            # just report both
            pass

        return {
            "audience": "HOLDER",
            "action": action,
            "confidence": conf,
            "reasons": "; ".join(reasons[:6]),
            "cost_basis": cost_basis,
            "unrealized_pnl_pct": pnl,
            "trail_sl": trail_sl,
            "tp1_hold": lv.get("tp1"),
            "tp2_hold": lv.get("tp2"),
            **a,
            **{k: ex[k] for k in ("exit_risk", "exit_band", "exit_action")},
            "ref_close": lv.get("ref_close"),
            "atr_abs": lv.get("atr_abs"),
        }


print("RecommendationEngine defined.")


RecommendationEngine defined.


In [42]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 34 — ENHANCED DAILY SCAN (levels + NEW vs HOLDER)                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class DailyScannerV2(DailyScanner):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.rec = RecommendationEngine(self.cfg)

    def scan(self, symbols: List[str], top_n: int = 15,
             min_avg_val: float = 8e6,
             portfolio: dict = None) -> dict:
        """
        portfolio: optional {symbol: cost_basis_price}
                   → produces HOLDER recommendations for those names
        """
        portfolio = portfolio or {}
        print("═" * 60)
        print(f"  DAILY SCANNER v2 | {len(symbols)} symbols | {datetime.now():%Y-%m-%d %H:%M}")
        print("═" * 60)

        end = datetime.today().strftime("%Y-%m-%d")
        start = (datetime.today() - timedelta(days=400)).strftime("%Y-%m-%d")
        df_vni = self.store.get("VNINDEX", start, end)
        regime = self.regime_eng.detect(df_vni.tail(60) if df_vni is not None else None)
        print(f"  Market: {regime.get('label')} | VNI={regime.get('vnindex')} mom5={regime.get('mom5')}%")

        rows = []
        for i, sym in enumerate(symbols):
            print(f"  [{i+1}/{len(symbols)}] {sym}...", end="\r")
            try:
                f = self._features_one(sym, df_vni)
                if f is None:
                    continue
                if float(f.get("avg_val_20d") or 0) < min_avg_val and sym not in portfolio:
                    continue
                rows.append(f)
            except Exception:
                continue
            time.sleep(self.cfg.delay_sec * 0.35)

        if not rows:
            print("\n  No data")
            return {"regime": regime}

        df = pd.DataFrame(rows)

        # NEW buyer table
        new_recs = []
        for _, r in df.iterrows():
            rec = self.rec.for_new_buyer(r.to_dict(), regime)
            rec["symbol"] = r["symbol"]
            rec["close"] = r["close"]
            new_recs.append(rec)
        new_df = pd.DataFrame(new_recs)
        new_df["alpha_pct"] = new_df["alpha_score"].rank(pct=True) * 100

        # buy_zone = new_df[new_df["action"].isin(["NEW_BUY", "WAIT_DIP"])].sort_values("alpha_score", ascending=False)
        buy_zone = new_df.sort_values("alpha_score", ascending=False)
        avoid_zone = new_df[new_df["action"] == "AVOID"].sort_values("exit_risk", ascending=False)

        # HOLDER table
        holder_rows = []
        hold_symbols = list(portfolio.keys()) if portfolio else list(
            df[df["exit_risk"] >= 35]["symbol"].head(300)
        )
        # always evaluate portfolio names; else show high exit-risk as monitor
        watch_set = set(hold_symbols) | set(portfolio.keys())
        for _, r in df.iterrows():
            sym = r["symbol"]
            if portfolio and sym not in portfolio and r.get("exit_risk", 0) < 40:
                continue
            if not portfolio and r.get("exit_risk", 0) < 40:
                continue
            basis = portfolio.get(sym)
            h = self.rec.for_holder(r.to_dict(), cost_basis=basis, regime=regime)
            h["symbol"] = sym
            h["close"] = r["close"]
            holder_rows.append(h)
        holder_df = pd.DataFrame(holder_rows)
        if len(holder_df):
            holder_df = holder_df.sort_values("exit_risk", ascending=False)

        # ---- Print dashboard ----
        print(f"\n\n  MARKET STATUS")
        print(f"  {'─'*56}")
        print(f"  Regime      : {regime.get('label')}")
        print(f"  Liquidity   : {regime.get('liquidity')} | Risk: {regime.get('risk_appetite')}")
        print(f"  Note        : Khuyến nghị = decision support, KHÔNG phải lệnh chắc thắng")

        print(f"\n  ══ CHƯA CẦM MÃ — Ứng viên MUA MỚI / CHỜ DIP ══")
        print(f"  {'─'*56}")
        cols_new = [c for c in [
            "symbol", "close", "action", "setup", "vsa_state", "alpha_score",
            "entry_suggest", "sl", "tp1", "tp2", "rr_tp1", "sl_pct", "tp1_pct",
            "exit_risk", "reasons"
        ] if c in buy_zone.columns]
        if len(buy_zone):
            print(buy_zone[cols_new].head(300).to_string(index=False))
        else:
            print("  (không có NEW_BUY/WAIT_DIP đạt ngưỡng hôm nay)")

        if len(avoid_zone):
            print(f"\n  ── AVOID (không mở mới) ──")
            acols = [c for c in ["symbol", "close", "action", "setup", "exit_risk", "reasons"] if c in avoid_zone.columns]
            print(avoid_zone[acols].head(50).to_string(index=False))

        print(f"\n  ══ ĐANG CẦM MÃ — HOLD / REDUCE / EXIT ══")
        print(f"  {'─'*56}")
        if len(holder_df):
            hcols = [c for c in [
                "symbol", "close", "action", "exit_risk", "exit_band",
                "trail_sl", "tp1_hold", "tp2_hold", "unrealized_pnl_pct", "reasons"
            ] if c in holder_df.columns]
            print(holder_df[hcols].to_string(index=False))
        else:
            print("  (không flag holder — truyền portfolio={{'FPT': 95.0, ...}} để soi danh mục)")

        # Detail cards for top 3 new
        print(f"\n  ══ CHI TIẾT TOP ỨNG VIÊN MUA ══")
        for _, r in buy_zone.head(20).iterrows():
            print(f"\n  [{r['symbol']}] {r['action']} | close={r['close']}")
            print(f"    Entry gợi ý : {r.get('entry_suggest')}  ({r.get('entry_note')})")
            print(f"    SL          : {r.get('sl')}  ({r.get('sl_pct')}%)")
            print(f"    TP1 / TP2   : {r.get('tp1')} ({r.get('tp1_pct')}%)  /  {r.get('tp2')} ({r.get('tp2_pct')}%)")
            print(f"    R:R TP1     : {r.get('rr_tp1')}  | partial TP1: {int(100*(r.get('partial_tp_pct') or 0.3))}%")
            print(f"    Size hint   : {r.get('size_hint')}")
            print(f"    Lý do       : {r.get('reasons')}")

        ts = datetime.now().strftime("%Y%m%d_%H%M")
        buy_zone.to_csv(RESULTS_DIR / f"rec_new_{ts}.csv", index=False)
        if len(holder_df):
            holder_df.to_csv(RESULTS_DIR / f"rec_holder_{ts}.csv", index=False)
        new_df.to_csv(RESULTS_DIR / f"rec_all_new_{ts}.csv", index=False)
        print(f"\n  ✓ CSV saved in results/")

        return {
            "regime": regime,
            "new": buy_zone,
            "avoid": avoid_zone,
            "holder": holder_df,
            "all_new": new_df,
        }


def run_daily_scan_v2(symbols=None, portfolio=None, full=False, top_n=15):
    register_vnstock_if_needed()
    if symbols is None:
        symbols = VN100 if full else LIQUID_CORE
    seen = set()
    symbols = [s for s in symbols if not (s in seen or seen.add(s))]
    # ensure portfolio symbols are scanned
    if portfolio:
        symbols = list(dict.fromkeys(list(symbols) + list(portfolio.keys())))
    sc = DailyScannerV2(MarketDataStore(CFG), CFG)
    return sc.scan(symbols, top_n=top_n, portfolio=portfolio or {})


print("DailyScannerV2 ready.")
print("→ out = run_daily_scan_v2()")
print("→ out = run_daily_scan_v2(portfolio={'HDB': 24.5, 'VNM': 58.0, 'PNJ': 40.0})")


DailyScannerV2 ready.
→ out = run_daily_scan_v2()
→ out = run_daily_scan_v2(portfolio={'HDB': 24.5, 'VNM': 58.0, 'PNJ': 40.0})


---

## Recommendation layer

### Hai đối tượng

| Audience | Actions |
|----------|---------|
| **Chưa cầm** | `NEW_BUY` · `WAIT_DIP` · `WAIT` · `AVOID` |
| **Đang cầm** | `HOLD_OR_SCALE_IN` · `HOLD` · `WATCH` · `REDUCE` · `EXIT` |

### Giá gợi ý (ATR-based, Spec §14)
- **Entry** (mới): limit gần close / EMA20 (pullback), breakout ít chiết khấu hơn  
- **SL**: ~1.2×ATR, sàn 2%–7%, có neo SMA50 khi hợp lý  
- **TP1 / TP2**: ~2.0× / 3.5× ATR — partial 30% tại TP1  
- **Size hint**: risk 0.75% NAV / khoảng SL (cap max position)

### Cách chạy
```python
# Chỉ quét ứng viên mua mới
out = run_daily_scan_v2()

# Kèm danh mục đang cầm (giá vốn)
out = run_daily_scan_v2(portfolio={
    "HDB": 24.5,
    "VNM": 58.0,
    "PNJ": 40.0,
    "SSI": 26.0,
})
```

`portfolio` values = **giá vốn trung bình**. Hệ thống sẽ ra HOLD/REDUCE/EXIT + trail SL + TP còn lại.

### Nhắc research
OOS chưa chứng minh edge sau phí → dùng mức giá như **khung quản trị rủi ro**, không như tín hiệu chắc thắng.


In [43]:
# out = run_daily_scan_v2()
# out = run_daily_scan_v2(portfolio={'HDB': 24.5, 'VNM': 58.0, 'PNJ': 40.0})


In [44]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 35 — SECTOR ENGINE (Spec §25)                                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Static VN sector map for liquid universe (no paid fundamental API required).
# Sector return / RS computed as equal-weight average of member closes.
# Stocks without mapping → sector="UNKNOWN" (no fake strength).

# ================================================================
# SECTOR MAP
# Primary Business / Dominant Stock Driver
#
# Rule:
#   - Each ticker MUST belong to exactly ONE sector.
#   - Classification is based on the primary business / dominant
#     stock-price driver, NOT every business activity.
#   - Designed for:
#       * Sector rotation
#       * Relative strength
#       * Regime Engine
#       * Momentum / breadth analysis
#       * Stock scoring
# ================================================================

SECTOR_MAP = {

    # ============================================================
    # BANKS
    # ============================================================
    "VCB": "Bank",
    "TCB": "Bank",
    "ACB": "Bank",
    "BID": "Bank",
    "MBB": "Bank",
    "HDB": "Bank",
    "VPB": "Bank",
    "CTG": "Bank",
    "STB": "Bank",
    "TPB": "Bank",
    "SHB": "Bank",
    "OCB": "Bank",
    "VIB": "Bank",
    "SSB": "Bank",
    "MSB": "Bank",
    "EIB": "Bank",
    "LPB": "Bank",
    "NAB": "Bank",
    "BVB": "Bank",
    "ABB": "Bank",
    "PGB": "Bank",
    "VBB": "Bank",
    "NCG": "Bank",
    "KLB": "Bank",
    "VAB": "Bank",
    "NVB": "Bank",
    "EVF": "Bank",

    # ============================================================
    # SECURITIES
    # ============================================================
    "SSI": "Securities",
    "VND": "Securities",
    "VCI": "Securities",
    "HCM": "Securities",
    "MBS": "Securities",
    "CTS": "Securities",
    "FTS": "Securities",
    "BSI": "Securities",
    "ORS": "Securities",
    "SHS": "Securities",
    "VIX": "Securities",
    "AAS": "Securities",
    "DSE": "Securities",
    "SBS": "Securities",
    "BMS": "Securities",
    "PSI": "Securities",
    "APS": "Securities",
    "VFS": "Securities",
    "TCI": "Securities",
    "TVC": "Securities",
    "ABW": "Securities",

    # ============================================================
    # FINANCIAL SERVICES
    # ============================================================
    "F88": "FinancialServices",
    "ALC": "FinancialServices",
    "IPA": "FinancialServices",

    # ============================================================
    # REAL ESTATE
    # ============================================================
    "VHM": "RealEstate",
    "VIC": "RealEstate",
    "VRE": "RealEstate",
    "NVL": "RealEstate",
    "DXG": "RealEstate",
    "KDH": "RealEstate",
    "NLG": "RealEstate",
    "PDR": "RealEstate",
    "DIG": "RealEstate",
    "HDG": "RealEstate",
    "BCM": "RealEstate",
    "KBC": "RealEstate",
    "CEO": "RealEstate",
    "AGG": "RealEstate",
    "TCH": "RealEstate",
    "CRE": "RealEstate",
    "VPI": "RealEstate",
    "HDC": "RealEstate",
    "DXS": "RealEstate",
    "SJS": "RealEstate",
    "KSF": "RealEstate",
    "FID": "RealEstate",
    "VC3": "RealEstate",
    "VC7": "RealEstate",
    "VC2": "RealEstate",
    "DTD": "RealEstate",
    "UNI": "RealEstate",
    "L40": "RealEstate",
    "API": "RealEstate",
    "IDJ": "RealEstate",
    "AAV": "RealEstate",
    "BIG": "RealEstate",
    "G36": "RealEstate",
    "LDG": "RealEstate",
    "HQC": "RealEstate",
    "SCR": "RealEstate",
    "QCG": "RealEstate",
    "NRC": "RealEstate",
    "TIG": "RealEstate",
    "DL1": "RealEstate",
    "CTP": "RealEstate",

    # ============================================================
    # CONSTRUCTION / ENGINEERING
    # ============================================================
    "CTD": "Construction",
    "VCG": "Construction",
    "CII": "Construction",
    "HHV": "Construction",
    "LCG": "Construction",
    "AMS": "Construction",
    "C69": "Construction",
    "MST": "Construction",
    "CMS": "Construction",
    "CTX": "Construction",
    "TTH": "Construction",
    "TTG": "Construction",
    "PIV": "Construction",
    "TV1": "Construction",

    # ============================================================
    # MATERIALS
    # Steel / Chemicals / Fertilizer / Cement / Mining /
    # Building Materials / Rubber
    # ============================================================
    "HPG": "Materials",
    "HSG": "Materials",
    "NKG": "Materials",
    "GVR": "Materials",
    "DGC": "Materials",
    "DCM": "Materials",
    "DPM": "Materials",
    "AAA": "Materials",
    "HT1": "Materials",
    "SMC": "Materials",
    "TLH": "Materials",
    "BFC": "Materials",
    "LAS": "Materials",
    "MSR": "Materials",
    "VGS": "Materials",
    "PVB": "Materials",
    "PLC": "Materials",
    "NTP": "Materials",
    "VGC": "Materials",
    "XMC": "Materials",
    "PCH": "Materials",
    "NBC": "Materials",
    "CAP": "Materials",
    "KCB": "Materials",
    "C32": "Materials",
    "HSL": "Materials",
    "RYG": "Materials",
    "HII": "Materials",
    "SHI": "Materials",
    "HHP": "Materials",
    "VNP": "Materials",
    "DDV": "Materials",
    "VTZ": "Materials",

    # ============================================================
    # ENERGY
    # Oil / Gas / Refining / Oil Services
    # ============================================================
    "GAS": "Energy",
    "PLX": "Energy",
    "BSR": "Energy",
    "PVD": "Energy",
    "PVT": "Energy",
    "OIL": "Energy",
    "PVS": "Energy",
    "PVC": "Energy",
    "PVG": "Energy",
    "PPT": "Energy",
    "PXL": "Energy",

    # ============================================================
    # UTILITIES
    # Electricity / Water
    # ============================================================
    "POW": "Utilities",
    "REE": "Utilities",
    "BWE": "Utilities",
    "GEG": "Utilities",
    "NT2": "Utilities",
    "QTP": "Utilities",
    "PPC": "Utilities",
    "VIW": "Utilities",

    # ============================================================
    # TECHNOLOGY / TELECOM
    # ============================================================
    "FPT": "Technology",
    "CMG": "Technology",
    "ELC": "Technology",
    "FOX": "Technology",
    "VGI": "Technology",
    "CTR": "Technology",

    # ============================================================
    # RETAIL
    # ============================================================
    "MWG": "Retail",
    "FRT": "Retail",
    "DGW": "Retail",
    "PNJ": "Retail",
    "KOS": "Retail",

    # ============================================================
    # CONSUMER / FOOD & BEVERAGE
    # ============================================================
    "MSN": "Consumer",
    "VNM": "Consumer",
    "SAB": "Consumer",
    "QNS": "Consumer",
    "BMP": "Consumer",
    "KDC": "Consumer",
    "SBT": "Consumer",
    "BNA": "Consumer",
    "HNM": "Consumer",
    "FOC": "Consumer",
    "OCH": "Consumer",
    "SBB": "Consumer",

    # ============================================================
    # AGRICULTURE / LIVESTOCK / SEAFOOD
    # ============================================================
    "DBC": "Agriculture",
    "BAF": "Agriculture",
    "HAG": "Agriculture",
    "HNG": "Agriculture",
    "DRI": "Agriculture",
    "ANV": "Agriculture",
    "VHC": "Agriculture",
    "MPC": "Agriculture",
    "PAN": "Agriculture",
    "APF": "Agriculture",
    "GCF": "Agriculture",
    "MZG": "Agriculture",

    # ============================================================
    # TEXTILE / GARMENTS
    # ============================================================
    "MSH": "Textile",
    "TNG": "Textile",
    "TCM": "Textile",
    "VGT": "Textile",

    # ============================================================
    # INDUSTRIAL / MANUFACTURING
    # ============================================================
    "PC1": "Industrial",
    "GEX": "Industrial",
    "GEE": "Industrial",
    "VEA": "Industrial",
    "KIP": "Industrial",
    "HUT": "Industrial",
    "DLG": "Industrial",
    "TVN": "Industrial",

    # ============================================================
    # TRANSPORT / LOGISTICS / PORTS / AVIATION
    # ============================================================
    "GMD": "Transport",
    "VSC": "Transport",
    "ACV": "Transport",
    "HVN": "Transport",
    "SCS": "Transport",
    "PHP": "Transport",
    "DXP": "Transport",
    "VGP": "Transport",
    "VTO": "Transport",
    "TOS": "Transport",

    # ============================================================
    # HEALTHCARE / PHARMACEUTICALS
    # ============================================================
    "IMP": "Healthcare",
    "DVN": "Healthcare",
    "LDP": "Healthcare",

    # ============================================================
    # INSURANCE
    # ============================================================
    "BVH": "Insurance",
    "BMI": "Insurance",
    "PVI": "Insurance",

    # ============================================================
    # TOURISM / HOSPITALITY
    # ============================================================
    "DAH": "Tourism",
    "DST": "Tourism",
    "VTD": "Tourism",

}


class SectorEngine:
    """
    Build sector panels from member OHLCV already in MarketDataStore cache / fetch.
    """

    def __init__(self, store: MarketDataStore, cfg: ResearchConfig = CFG,
                 sector_map: dict = None):
        self.store = store
        self.cfg = cfg
        self.sector_map = sector_map or SECTOR_MAP
        self._sector_returns = {}  # sector -> {5d,20d,60d, rs_vs_market}
        self._member_cache = {}

    def sector_of(self, symbol: str) -> str:
        return self.sector_map.get(symbol.upper(), "UNKNOWN")

    def members(self, sector: str) -> list:
        return [s for s, sec in self.sector_map.items() if sec == sector]

    def compute_sector_stats(self, as_of: str, universe: list = None,
                             df_vni: pd.DataFrame = None) -> pd.DataFrame:
        """
        Equal-weight sector returns ending at as_of (trading sessions).
        Returns DataFrame indexed by sector.
        """
        as_of_ts = pd.Timestamp(as_of).normalize()
        universe = universe or list(self.sector_map.keys())
        # group symbols
        by_sec = defaultdict(list)
        for s in universe:
            by_sec[self.sector_of(s)].append(s)

        # market return ref
        mkt_ret = {}
        if df_vni is not None and len(df_vni) >= 60:
            v = df_vni[df_vni["time"] <= as_of_ts]["close"].astype(float)
            for n in (5, 20, 60):
                mkt_ret[n] = float(v.iloc[-1] / v.iloc[-n-1] - 1) if len(v) > n else np.nan

        rows = []
        for sec, syms in by_sec.items():
            if sec == "UNKNOWN":
                continue
            rets = {5: [], 20: [], 60: []}
            for sym in syms:
                df = self.store.get_as_of(sym, as_of, lookback_bars=120)
                if df is None or len(df) < 25:
                    continue
                c = df["close"].astype(float)
                for n in (5, 20, 60):
                    if len(c) > n:
                        rets[n].append(float(c.iloc[-1] / c.iloc[-n-1] - 1))
            if not rets[20]:
                continue
            r5 = float(np.nanmean(rets[5])) if rets[5] else np.nan
            r20 = float(np.nanmean(rets[20])) if rets[20] else np.nan
            r60 = float(np.nanmean(rets[60])) if rets[60] else np.nan
            rs20 = r20 - mkt_ret.get(20, 0) if not np.isnan(r20) and 20 in mkt_ret else np.nan
            # strength score 0-100
            strength = 50.0
            if not np.isnan(r20):
                strength += float(np.clip(r20 * 200, -25, 25))
            if not np.isnan(rs20):
                strength += float(np.clip(rs20 * 150, -15, 15))
            strength = float(np.clip(strength, 0, 100))
            rows.append({
                "sector": sec,
                "n_members_used": len(rets[20]),
                "sector_ret_5d": round(r5, 4) if not np.isnan(r5) else np.nan,
                "sector_ret_20d": round(r20, 4) if not np.isnan(r20) else np.nan,
                "sector_ret_60d": round(r60, 4) if not np.isnan(r60) else np.nan,
                "sector_rs_20d": round(rs20, 4) if not np.isnan(rs20) else np.nan,
                "sector_strength": round(strength, 1),
            })
        out = pd.DataFrame(rows)
        self._sector_returns = {r["sector"]: r for r in rows}
        return out

    def attach_to_features(self, f: dict, symbol: str) -> dict:
        sec = self.sector_of(symbol)
        f = dict(f)
        f["sector"] = sec
        info = self._sector_returns.get(sec, {})
        f["sector_ret_20d"] = info.get("sector_ret_20d", np.nan)
        f["sector_rs_20d"] = info.get("sector_rs_20d", np.nan)
        f["sector_strength"] = info.get("sector_strength", 50.0)
        # conviction tilt: stock RS vs sector
        rs20 = f.get("rs_20d")
        if rs20 is not None and not (isinstance(rs20, float) and np.isnan(rs20)) and not np.isnan(f["sector_ret_20d"] if f["sector_ret_20d"] is not None else np.nan):
            # stock own ret proxy already in rs vs market; keep sector_strength as context
            pass
        return f


print(f"SectorEngine defined | mapped symbols: {len(SECTOR_MAP)}")


SectorEngine defined | mapped symbols: 224


In [45]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 36 — MARKET BREADTH ENGINE (Spec §26)                             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class MarketBreadthEngine:
    """
    Breadth from a scanned universe (not full HOSE/HNX unless provided).
    Document: breadth is relative to the scan universe → label as universe-breadth.
    """

    def __init__(self, store: MarketDataStore, cfg: ResearchConfig = CFG):
        self.store = store
        self.cfg = cfg
        self._last = {}

    def compute(self, symbols: list, as_of: str = None) -> dict:
        as_of = as_of or datetime.today().strftime("%Y-%m-%d")
        n = 0
        above20 = above50 = above200 = 0
        up = down = 0
        for sym in symbols:
            df = self.store.get_as_of(sym, as_of, lookback_bars=220)
            if df is None or len(df) < 30:
                continue
            c = df["close"].astype(float)
            n += 1
            last = float(c.iloc[-1])
            if len(c) >= 20 and last > float(c.tail(20).mean()):
                above20 += 1
            if len(c) >= 50 and last > float(c.tail(50).mean()):
                above50 += 1
            if len(c) >= 200 and last > float(c.tail(200).mean()):
                above200 += 1
            if len(c) >= 2:
                if last > float(c.iloc[-2]):
                    up += 1
                elif last < float(c.iloc[-2]):
                    down += 1

        if n == 0:
            out = {
                "n_universe": 0,
                "pct_above_ma20": np.nan,
                "pct_above_ma50": np.nan,
                "pct_above_ma200": np.nan,
                "advance_decline": np.nan,
                "breadth_label": "UNKNOWN",
            }
            self._last = out
            return out

        p20 = above20 / n
        p50 = above50 / n
        p200 = above200 / n
        ad = up / max(down, 1)

        if p50 >= 0.65 and p20 >= 0.60:
            label = "HEALTHY"
        elif p50 >= 0.45:
            label = "MIXED"
        elif p50 >= 0.30:
            label = "WEAK"
        else:
            label = "POOR"

        out = {
            "n_universe": n,
            "pct_above_ma20": round(p20 * 100, 1),
            "pct_above_ma50": round(p50 * 100, 1),
            "pct_above_ma200": round(p200 * 100, 1),
            "advance_decline": round(ad, 2),
            "breadth_label": label,
        }
        self._last = out
        return out

    @property
    def last(self):
        return self._last


print("MarketBreadthEngine defined.")


MarketBreadthEngine defined.


In [46]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 37 — EXIT RISK v2 (sector/breadth aware) + light calibration hook ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Spec §37: bands must eventually be validated. Prior scan showed EXIT band
# had higher forward returns (miscalibration). v2:
#   - softer thresholds
#   - sector weakness adds risk only if stock also weak
#   - do NOT auto-EXIT on single MA breach

class ExitRiskEngineV2(ExitRiskEngine):
    def score(self, f: dict, breadth: dict = None) -> dict:
        base = super().score(f)   # BẮT BUỘC có dòng này
        risk = float(base["exit_risk"])

        _er = base.get("exit_reasons")
        if isinstance(_er, list):
            reasons = [str(x) for x in _er if x]
        elif isinstance(_er, str) and _er:
            reasons = list(filter(None, _er.split("; ")))
        else:
            reasons = []

        sector_strength = float(f.get("sector_strength") or 50)
        stock_rs = f.get("rs_20d")
        stock_rs = float(stock_rs) if stock_rs is not None and not (isinstance(stock_rs, float) and np.isnan(stock_rs)) else 0.0

        # Sector drag only if sector weak AND stock not outperforming hard
        if sector_strength < 40 and stock_rs < 0.02:
            risk += 8
            reasons.append("Sector yếu + RS không vượt")
        elif sector_strength < 35:
            risk += 4
            reasons.append("Sector rất yếu")

        if breadth and breadth.get("breadth_label") in ("WEAK", "POOR"):
            if float(f.get("factor_trend") or 50) < 55:
                risk += 6
                reasons.append(f"Breadth {breadth.get('breadth_label')}")

        # Soften: cap extreme from single-factor pile-up unless distribution
        setup = str(f.get("setup") or "NONE")
        vsa = str(f.get("vsa_state") or "—")
        if setup != "DISTRIBUTION" and vsa not in ("PHÂN PHỐI", "CẠN CẦU"):
            risk = min(risk, 75)  # reserve 80+ for clearer distribution

        risk = float(np.clip(risk, 0, 100))
        if risk < 20:
            action, band = "HOLD", "HEALTHY"
        elif risk < 40:
            action, band = "WATCH", "WATCH"
        elif risk < 55:
            action, band = "WATCH", "DETERIORATING"
        elif risk < 75:
            action, band = "REDUCE", "REDUCE"
        else:
            action, band = "EXIT", "EXIT"

        return {
            "exit_risk": round(risk, 1),
            "exit_band": band,
            "exit_action": action,
            "exit_reasons": reasons,
        }


def calibrate_exit_bands(ds: pd.DataFrame, horizon_col="future_return_10d") -> pd.DataFrame:
    """
    Empirical check: mean forward return by exit_band.
    Healthy should have higher forward ret than EXIT if calibrated.
    """
    if ds is None or "exit_band" not in ds.columns or horizon_col not in ds.columns:
        print("Need exit_band + forward returns on dataset")
        return pd.DataFrame()
    g = ds.groupby("exit_band")[horizon_col].agg(["count", "mean", "median"])
    print("Exit band vs forward return (calibration diagnostic):")
    print(g.to_string())
    return g


print("ExitRiskEngineV2 + calibrate_exit_bands defined.")


ExitRiskEngineV2 + calibrate_exit_bands defined.


In [47]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 38 — SCANNER v3: + Sector + Breadth (Spec §25–26 + §62)           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class DailyScannerV3(DailyScannerV2):
    def __init__(self, store=None, cfg=CFG):
        store = store or MarketDataStore(cfg)
        super().__init__(store, cfg)
        self.sector_eng = SectorEngine(store, cfg)
        self.breadth_eng = MarketBreadthEngine(store, cfg)
        self.exit_v2 = ExitRiskEngineV2(cfg)
        # override rec exit engine
        self.rec.exit_r.engine = self.exit_v2

    def scan(self, symbols, top_n=15, min_avg_val=8e6, portfolio=None):
        portfolio = portfolio or {}
        print("═" * 60)
        print(f"  DAILY SCANNER v3 (+Sector/Breadth) | {len(symbols)} | {datetime.now():%Y-%m-%d %H:%M}")
        print("═" * 60)

        end = datetime.today().strftime("%Y-%m-%d")
        start = (datetime.today() - timedelta(days=400)).strftime("%Y-%m-%d")
        df_vni = self.store.get("VNINDEX", start, end)
        regime = self.regime_eng.detect(df_vni.tail(60) if df_vni is not None else None)

        # Breadth on scan universe (document: universe-breadth)
        print("  Computing breadth...")
        breadth = self.breadth_eng.compute(symbols, end)
        regime = dict(regime)
        regime["breadth"] = breadth.get("breadth_label")
        regime["breadth_detail"] = breadth

        print("  Computing sector stats...")
        sec_tbl = self.sector_eng.compute_sector_stats(end, symbols, df_vni)
        if len(sec_tbl):
            print(sec_tbl.sort_values("sector_strength", ascending=False).head(30).to_string(index=False))

        rows = []
        for i, sym in enumerate(symbols):
            print(f"  [{i+1}/{len(symbols)}] {sym}...", end="\r")
            try:
                f = self._features_one(sym, df_vni)
                if f is None:
                    continue
                if float(f.get("avg_val_20d") or 0) < min_avg_val and sym not in portfolio:
                    continue
                f = self.sector_eng.attach_to_features(f, sym)
                # recompute exit with v2
                ex = self.exit_v2.score(f, breadth=breadth)
                f.update(ex)
                rows.append(f)
            except Exception:
                continue
            time.sleep(self.cfg.delay_sec * 0.3)

        if not rows:
            print("\n  No rows")
            return {"regime": regime, "breadth": breadth, "sectors": sec_tbl}

        df = pd.DataFrame(rows)

        # Boost alpha with sector alignment
        def alpha_with_sector(row):
            base = self.rec.alpha.score_row(row.to_dict(), regime)
            ss = float(row.get("sector_strength") or 50)
            if ss >= 60:
                base["alpha_score"] = min(100, base["alpha_score"] + 3)
            elif ss <= 40:
                base["alpha_score"] = max(0, base["alpha_score"] - 4)
            base["sector"] = row.get("sector")
            base["sector_strength"] = ss
            return base

        new_recs = []
        for _, r in df.iterrows():
            f = r.to_dict()
            # inject sector-boosted alpha into for_new_buyer path
            a = alpha_with_sector(r)
            f.update(a)
            rec = self.rec.for_new_buyer(f, regime)
            rec["symbol"] = r["symbol"]
            rec["close"] = r["close"]
            rec["sector"] = r.get("sector")
            rec["sector_strength"] = r.get("sector_strength")
            # rebuild levels
            lv = self.rec.levels.build(f, mode="new")
            rec.update(lv)
            new_recs.append(rec)
        new_df = pd.DataFrame(new_recs)
        new_df["alpha_pct"] = new_df["alpha_score"].rank(pct=True) * 100
        # buy_zone = new_df[new_df["action"].isin(["NEW_BUY", "WAIT_DIP"])].sort_values("alpha_score", ascending=False)
        buy_zone = new_df.sort_values("alpha_score", ascending=False)

        holder_rows = []
        for _, r in df.iterrows():
            sym = r["symbol"]
            if portfolio:
                if sym not in portfolio and float(r.get("exit_risk") or 0) < 40:
                    continue
            else:
                if float(r.get("exit_risk") or 0) < 40:
                    continue
            h = self.rec.for_holder(r.to_dict(), cost_basis=portfolio.get(sym), regime=regime)
            h["symbol"] = sym
            h["close"] = r["close"]
            h["sector"] = r.get("sector")
            holder_rows.append(h)
        holder_df = pd.DataFrame(holder_rows)
        if len(holder_df):
            holder_df = holder_df.sort_values("exit_risk", ascending=False)

        print(f"\n\n  MARKET STATUS")
        print(f"  {'─'*56}")
        print(f"  Regime      : {regime.get('label')}")
        print(f"  Breadth     : {breadth.get('breadth_label')} "
              f"(MA20 {breadth.get('pct_above_ma20')}% | MA50 {breadth.get('pct_above_ma50')}% | "
              f"MA200 {breadth.get('pct_above_ma200')}% | n={breadth.get('n_universe')})")
        print(f"  Liquidity   : {regime.get('liquidity')} | Risk: {regime.get('risk_appetite')}")
        print(f"  Note        : Breadth = within scan universe (not full exchange)")

        print(f"\n  ══ CHƯA CẦM — NEW_BUY / WAIT_DIP ══")
        cols = [c for c in ["symbol","sector","close","action","setup","alpha_score",
                            "sector_strength","entry_suggest","sl","tp1","tp2","rr_tp1",
                            "exit_risk","reasons"] if c in buy_zone.columns]
        if len(buy_zone):
            print(buy_zone[cols].head(300).to_string(index=False))
        else:
            print("  (empty)")

        print(f"\n  ══ ĐANG CẦM / EXIT MONITOR ══")
        if len(holder_df):
            hcols = [c for c in ["symbol","sector","close","action","exit_risk","exit_band",
                                 "trail_sl","tp1_hold","unrealized_pnl_pct","reasons"] if c in holder_df.columns]
            print(holder_df[hcols].to_string(index=False))
        else:
            print("  (pass portfolio={{...}} or no high exit_risk)")

        print(f"\n  ══ CHI TIẾT TOP MUA ══")
        for _, r in buy_zone.head(10).iterrows():
            print(f"\n  [{r['symbol']}] {r.get('sector')} | {r['action']} | close={r['close']}")
            print(f"    Entry {r.get('entry_suggest')} | SL {r.get('sl')} ({r.get('sl_pct')}%) | "
                  f"TP1 {r.get('tp1')} | TP2 {r.get('tp2')} | R:R {r.get('rr_tp1')}")
            print(f"    Sector strength {r.get('sector_strength')} | {r.get('reasons')}")
            print(f"    Size: {r.get('size_hint')}")

        ts = datetime.now().strftime("%Y%m%d_%H%M")
        buy_zone.to_csv(RESULTS_DIR / f"scan_v3_buy_{ts}.csv", index=False)
        if len(holder_df):
            holder_df.to_csv(RESULTS_DIR / f"scan_v3_holder_{ts}.csv", index=False)
        if len(sec_tbl):
            sec_tbl.to_csv(RESULTS_DIR / f"scan_v3_sectors_{ts}.csv", index=False)
        print(f"\n  ✓ Saved results/scan_v3_*_{ts}.csv")

        return {
            "regime": regime,
            "breadth": breadth,
            "sectors": sec_tbl,
            "new": buy_zone,
            "holder": holder_df,
            "all": df,
        }


def run_daily_scan_v3(symbols=None, portfolio=None, full=False):
    register_vnstock_if_needed()
    if symbols is None:
        symbols = VN100 if full else LIQUID_CORE
    seen = set()
    symbols = [s for s in symbols if not (s in seen or seen.add(s))]
    if portfolio:
        symbols = list(dict.fromkeys(list(symbols) + list(portfolio.keys())))
    return DailyScannerV3(MarketDataStore(CFG), CFG).scan(symbols, portfolio=portfolio or {})


print("run_daily_scan_v3() ready")
print("→ out = run_daily_scan_v3()")
print("→ out = run_daily_scan_v3(portfolio={'HDB':24.5,'PNJ':40.0})")


run_daily_scan_v3() ready
→ out = run_daily_scan_v3()
→ out = run_daily_scan_v3(portfolio={'HDB':24.5,'PNJ':40.0})


---

## Spec progress this step

| Spec | Component | Status |
|------|-----------|--------|
| §25 | **SectorEngine** | ✅ map + ret 5/20/60 + RS vs market + strength |
| §26 | **MarketBreadthEngine** | ✅ % above MA20/50/200 + A/D + label (universe-relative) |
| §27 | Regime + breadth field | ✅ scanner injects breadth into regime dict |
| §35–37 | ExitRisk v2 | 🟡 softer + sector/breadth; calibration helper |
| §62 | Scanner dashboard | ✅ v3 includes sector column + breadth line |

### Still next (priority order)
1. Attach sector/breadth into **ResearchDatasetBuilder** historical rows (for conditional P)
2. Exit band **empirical calibration** on labeled ds (`calibrate_exit_bands`)
3. `pip install scikit-learn` → logistic OOS
4. Portfolio concentration engine (§44)
5. Purged CV / full ablation

### Run
```python
out = run_daily_scan_v3()
# out = run_daily_scan_v3(portfolio={"HDB": 24.5, "VNM": 58.0, "PNJ": 40.0})
```


In [48]:
# out = run_daily_scan_v3()
# out = run_daily_scan_v3(portfolio={'HDB': 24.5, 'VNM': 58.0, 'PNJ': 40.0})


In [49]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 39 — DATASET + SECTOR CONTEXT + EXIT CALIBRATION (Spec §16, §37)  ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def enrich_dataset_sector_exit(ds: pd.DataFrame, store: MarketDataStore = None) -> pd.DataFrame:
    """
    Attach sector fields + ExitRisk v2 to an existing features parquet.
    Sector strength here is STATIC snapshot from SECTOR_MAP membership only
    for speed; full historical sector panel is expensive under community API.
    For conditional stats we use sector label + contemporaneous stock factors.
    """
    store = store or MarketDataStore(CFG)
    se = SectorEngine(store, CFG)
    ee = ExitRiskEngineV2(CFG)
    rows = []
    for _, r in ds.iterrows():
        f = r.to_dict()
        sym = str(f.get("symbol", "")).upper()
        f["sector"] = se.sector_of(sym)
        # placeholder strength until batch sector panel built
        f.setdefault("sector_strength", 50.0)
        f.setdefault("sector_ret_20d", np.nan)
        f.setdefault("sector_rs_20d", np.nan)
        ex = ee.score(f, breadth=None)
        f.update(ex)
        rows.append(f)
    out = pd.DataFrame(rows)
    return out


def run_exit_calibration(ds: pd.DataFrame = None):
    """Spec §37 — validate exit bands vs forward returns."""
    if ds is None:
        files = sorted(DATA_FEATURES.glob("features_v3*.parquet"))
        files = [f for f in files if "atr" not in f.name] or files
        if not files:
            print("No features parquet")
            return None
        ds = pd.read_parquet(files[-1])
        print(f"Loaded {files[-1]} → {len(ds)}")

    if "setup" not in ds.columns:
        ds = enrich_with_setup_exit(ds)

    ds2 = enrich_dataset_sector_exit(ds)
    print("\n[1] ExitRisk v2 band vs future_return_10d")
    cal = calibrate_exit_bands(ds2, "future_return_10d")

    print("\n[2] Same for future_return_5d")
    calibrate_exit_bands(ds2, "future_return_5d")

    # Monotonic check
    if cal is not None and len(cal):
        order = ["HEALTHY", "WATCH", "DETERIORATING", "REDUCE", "EXIT"]
        means = []
        for b in order:
            if b in cal.index:
                means.append((b, cal.loc[b, "mean"]))
        print("\n[3] Monotonicity (HEALTHY should ≥ EXIT mean forward ret):")
        for b, m in means:
            print(f"  {b:15s} mean_fwd={m:+.4f}")
        if len(means) >= 2:
            if means[0][1] > means[-1][1]:
                print("  → Direction OK (HEALTHY > last band)")
            else:
                print("  → STILL INVERTED or weak — do not hard-auto EXIT on band alone")

    path = DATA_FEATURES / "features_v3_sector_exit.parquet"
    ds2.to_parquet(path, index=False)
    print(f"\n✓ Saved {path}")
    return ds2


print("run_exit_calibration() ready")


run_exit_calibration() ready


In [50]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 40 — PORTFOLIO CONCENTRATION (Spec §44)                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class PortfolioRiskEngine:
    """
    Detect concentration risk for a proposed or existing book.
    portfolio: {symbol: weight} weights sum≈1, or {symbol: notional_vnd}
    """

    def __init__(self, sector_map: dict = None):
        self.sector_map = sector_map or SECTOR_MAP

    def _weights(self, portfolio: dict) -> pd.Series:
        s = pd.Series({k.upper(): float(v) for k, v in portfolio.items()})
        if s.sum() > 2.0:  # treat as notional
            s = s / s.sum()
        else:
            # already weights; renormalize
            s = s / s.sum() if s.sum() > 0 else s
        return s

    def analyze(self, portfolio: dict, max_name_pct: float = 0.20,
                max_sector_pct: float = 0.35) -> dict:
        w = self._weights(portfolio)
        if len(w) == 0:
            return {"ok": True, "warnings": ["empty portfolio"]}

        sec = w.index.to_series().map(lambda x: self.sector_map.get(x, "UNKNOWN"))
        by_sec = w.groupby(sec).sum().sort_values(ascending=False)
        top_name = w.max()
        top_sym = w.idxmax()
        hhi = float((w ** 2).sum())  # 1/n → 1

        warnings = []
        if top_name > max_name_pct:
            warnings.append(f"Name concentration: {top_sym}={top_name:.1%} > {max_name_pct:.0%}")
        for sector, sw in by_sec.items():
            if sw > max_sector_pct:
                warnings.append(f"Sector concentration: {sector}={sw:.1%} > {max_sector_pct:.0%}")
        if hhi > 0.25:
            warnings.append(f"HHI={hhi:.3f} high (concentrated book)")

        # correlated clusters (hard-coded VN peer groups)
        peers = {
            "Banks": [s for s in w.index if self.sector_map.get(s) == "Bank"],
            "Securities": [s for s in w.index if self.sector_map.get(s) == "Securities"],
            "RealEstate": [s for s in w.index if self.sector_map.get(s) == "RealEstate"],
        }
        for name, mem in peers.items():
            if len(mem) >= 3:
                cw = float(w[mem].sum())
                if cw > max_sector_pct:
                    warnings.append(f"Peer cluster {name}: {len(mem)} names, weight={cw:.1%}")

        report = {
            "ok": len(warnings) == 0,
            "n_names": len(w),
            "top_name": top_sym,
            "top_weight": round(float(top_name), 4),
            "hhi": round(hhi, 4),
            "sector_weights": {k: round(float(v), 4) for k, v in by_sec.items()},
            "warnings": warnings,
        }
        print("══ PORTFOLIO RISK ══")
        print(f"  names={report['n_names']} top={report['top_name']} ({report['top_weight']:.1%}) HHI={report['hhi']}")
        print("  Sectors:")
        for k, v in report["sector_weights"].items():
            print(f"    {k:12s} {v:.1%}")
        if warnings:
            print("  WARNINGS:")
            for wmsg in warnings:
                print(f"    ⚠ {wmsg}")
        else:
            print("  ✓ within default concentration limits")
        return report


print("PortfolioRiskEngine defined.")
print("→ PortfolioRiskEngine().analyze({'VCB':0.15,'TCB':0.15,'MBB':0.15,'HPG':0.1,'FPT':0.2})")


PortfolioRiskEngine defined.
→ PortfolioRiskEngine().analyze({'VCB':0.15,'TCB':0.15,'MBB':0.15,'HPG':0.1,'FPT':0.2})


In [51]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 41 — LOGISTIC OOS (ensure sklearn) Spec §39–41                    ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def ensure_sklearn():
    global HAS_SK
    try:
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler
        from sklearn.metrics import brier_score_loss
        HAS_SK = True
        return True
    except ImportError:
        print("Installing scikit-learn...")
        import subprocess, sys
        subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-learn", "-q"])
        from sklearn.linear_model import LogisticRegression
        from sklearn.preprocessing import StandardScaler
        from sklearn.metrics import brier_score_loss
        HAS_SK = True
        return True


FEATURE_COLS_V2 = [
    "factor_trend", "factor_momentum", "factor_flow", "factor_rs",
    "factor_volatility", "factor_structure",
    "rsi", "cmf", "volume_ratio", "atr_pct", "ema_pct",
    "rs_20d", "bb_pctb", "ret_5d",
]


def run_logistic_research(ds: pd.DataFrame = None, setups=("PULLBACK", "BREAKOUT")):
    ensure_sklearn()
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import brier_score_loss

    if ds is None:
        # prefer sector-exit enriched
        p = DATA_FEATURES / "features_v3_sector_exit.parquet"
        if p.exists():
            ds = pd.read_parquet(p)
        else:
            files = sorted(DATA_FEATURES.glob("features_v3*.parquet"))
            ds = pd.read_parquet(files[-1])
        print(f"Loaded dataset {len(ds)} rows")

    if "setup" not in ds.columns:
        ds = enrich_with_setup_exit(ds)

    d0 = ds.copy()
    d0["date"] = pd.to_datetime(d0["date"])
    d0 = d0[d0["setup"].isin(list(setups))]

    folds = make_walk_forward_folds(start="2022-01-01", end="2025-12-31")
    print("═" * 60)
    print(f"  LOGISTIC OOS setups={setups}")
    print("═" * 60)

    all_preds = []
    for i, fold in enumerate(folds):
        tr = d0[(d0["date"] >= fold.train_start) & (d0["date"] <= fold.train_end)]
        te = d0[(d0["date"] >= fold.test_start) & (d0["date"] <= fold.test_end)]
        cols = [c for c in FEATURE_COLS_V2 if c in tr.columns]
        tr = tr.dropna(subset=cols + ["tb_label"])
        te = te.dropna(subset=cols + ["tb_label"])
        tr = tr[tr["tb_label"].isin(["TP_FIRST", "SL_FIRST", "TIMEOUT"])]
        te = te[te["tb_label"].isin(["TP_FIRST", "SL_FIRST", "TIMEOUT"])]
        ytr = (tr["tb_label"] == "TP_FIRST").astype(int).values
        yte = (te["tb_label"] == "TP_FIRST").astype(int).values
        if len(tr) < 40 or len(te) < 8 or ytr.sum() < 3:
            print(f"  Fold {i+1}: skip train={len(tr)} test={len(te)} pos={ytr.sum() if len(tr) else 0}")
            continue
        Xtr = tr[cols].astype(float).values
        Xte = te[cols].astype(float).values
        scaler = StandardScaler()
        Xtr_s = scaler.fit_transform(Xtr)
        Xte_s = scaler.transform(Xte)
        clf = LogisticRegression(max_iter=800, C=0.5, class_weight="balanced")
        clf.fit(Xtr_s, ytr)
        proba = clf.predict_proba(Xte_s)[:, 1]
        brier = brier_score_loss(yte, proba)
        te = te.copy()
        te["p_tp"] = proba
        te["y_tp"] = yte
        hi = te[te["p_tp"] >= te["p_tp"].quantile(0.7)]
        lo = te[te["p_tp"] <= te["p_tp"].quantile(0.3)]
        print(f"\n  Fold {i+1}: {fold.test_start}→{fold.test_end}")
        print(f"    n={len(te)} base={yte.mean():.3f} pred_mean={proba.mean():.3f} brier={brier:.4f}")
        print(f"    TOP30% obs_TP={hi['y_tp'].mean():.3f} avg_ret={hi['tb_ret'].mean():.4f} n={len(hi)}")
        print(f"    BOT30% obs_TP={lo['y_tp'].mean():.3f} avg_ret={lo['tb_ret'].mean():.4f} n={len(lo)}")
        coef = pd.Series(clf.coef_[0], index=cols).sort_values(key=abs, ascending=False)
        print("    coef:", dict(coef.head(6).round(3)))
        all_preds.append(te)

    if all_preds:
        cat = pd.concat(all_preds, ignore_index=True)
        path = RESULTS_DIR / "logistic_oos_preds.parquet"
        cat.to_parquet(path, index=False)
        print(f"\n✓ Predictions saved {path}")
        return cat
    return None


print("run_logistic_research() ready")


run_logistic_research() ready


In [52]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 42 — ONE-SHOT: calibration + logistic + portfolio demo            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def run_spec_next_batch():
    print("═" * 60)
    print("  SPEC NEXT BATCH: exit calib + logistic + portfolio demo")
    print("═" * 60)

    ds2 = run_exit_calibration()

    print("\n── Logistic ──")
    try:
        preds = run_logistic_research(ds2)
    except Exception as e:
        print(f"Logistic failed: {type(e).__name__}: {e}")
        preds = None

    print("\n── Portfolio concentration demo ──")
    # example concentrated banks book (should WARN)
    PortfolioRiskEngine().analyze({
        "VCB": 0.18, "TCB": 0.16, "MBB": 0.14, "ACB": 0.12, "HDB": 0.10,
        "HPG": 0.10, "FPT": 0.10, "VNM": 0.10,
    })
    # diversified
    print()
    PortfolioRiskEngine().analyze({
        "FPT": 0.12, "VCB": 0.10, "HPG": 0.10, "MWG": 0.10, "VNM": 0.10,
        "GAS": 0.10, "REE": 0.08, "VJC": 0.08, "SSI": 0.08, "PNJ": 0.07,
        "BMP": 0.07,
    })

    meta = ResearchRunMetadata(
        run_id=make_run_id(),
        timestamp=datetime.now().isoformat(),
        feature_version=CFG.feature_version + "+sector_exit_logit",
        model_version="logistic_v1",
        config_version=CFG.config_version,
        data_start="2022-01-01", data_end="2026-07-31",
        universe=LIQUID_CORE, n_symbols=len(LIQUID_CORE),
        execution_model=CFG.execution_model,
        tp_pct=CFG.tp_pct, sl_pct=CFG.sl_pct,
        max_holding_sessions=CFG.max_holding_sessions,
        fee_buy=CFG.fee_buy, fee_sell=CFG.fee_sell, tax_sell=CFG.tax_sell,
        random_seed=CFG.random_seed,
        notes="exit calibration + logistic OOS + portfolio risk",
    )
    meta.save()
    return {"ds": ds2, "preds": preds}


print("run_spec_next_batch() ready → result = run_spec_next_batch()")


run_spec_next_batch() ready → result = run_spec_next_batch()


---

## This step (Spec)

| Item | Status |
|------|--------|
| §37 Exit band empirical calibration | ✅ `run_exit_calibration` |
| §25 Sector label on dataset rows | ✅ (strength still snapshot/placeholder historically) |
| §39–41 Logistic OOS + Brier + TOP/BOT | ✅ `run_logistic_research` (+ auto pip sklearn) |
| §44 Portfolio concentration HHI/sector/name | ✅ `PortfolioRiskEngine` |

### Historical sector panel note
Full day-by-day sector returns for every research date would re-fetch many symbols × dates (API heavy on community).  
Current approach: **sector label** on each row for conditional stats; **live sector_strength** in Scanner v3 (computed once per scan).  
Upgrade path: precompute sector index parquet offline when sponsor API available.

### Run
```python
result = run_spec_next_batch()
```
Or separately:
```python
ds2 = run_exit_calibration()
preds = run_logistic_research(ds2)
PortfolioRiskEngine().analyze({"VCB": 0.2, "TCB": 0.2, "MBB": 0.15, "FPT": 0.15})
```


In [53]:
# result = run_spec_next_batch()


In [54]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 43 — VSASequence / BreakoutFollowThrough / MarketState (Spec §23,33,34)
# ╚══════════════════════════════════════════════════════════════════════════╝

class VSASequenceEngine:
    """
    Spec §23: sequence of recent VSA states (not only today).
    Input: list of states oldest→newest (len 3–5).
    """
    ACCUM_PATTERNS = [
        ("CẠN VOL", "CẠN VOL", "NỔ VOL"),
        ("CẠN VOL", "CẠN VOL", "CẠN VOL"),
        ("CẠN VOL", "RÚT CHÂN", "NỔ VOL"),
        ("—", "CẠN VOL", "NỔ VOL"),
    ]
    DIST_PATTERNS = [
        ("NỔ VOL", "PHÂN PHỐI", "PHÂN PHỐI"),
        ("—", "PHÂN PHỐI", "CẠN CẦU"),
        ("NỔ VOL", "—", "PHÂN PHỐI"),
        ("PHÂN PHỐI", "PHÂN PHỐI", "PHÂN PHỐI"),
    ]

    def classify(self, states: list) -> dict:
        states = [str(s) if s else "—" for s in states][-5:]
        seq = tuple(states[-3:]) if len(states) >= 3 else tuple(states)
        label = "NEUTRAL"
        if len(seq) == 3:
            if seq in self.ACCUM_PATTERNS or (seq.count("CẠN VOL") >= 2 and "PHÂN PHỐI" not in seq):
                label = "ACCUM_SEQ"
            elif seq in self.DIST_PATTERNS or seq.count("PHÂN PHỐI") >= 2:
                label = "DIST_SEQ"
            elif "NỔ VOL" in seq and "PHÂN PHỐI" in seq:
                label = "DIST_SEQ"
        return {"vsa_seq": "|".join(states), "vsa_seq_label": label}


class BreakoutFollowThroughEngine:
    """
    Spec §33: after a breakout bar, check next sessions hold above level.
    Call with bars AFTER signal (no look-ahead when used at T with only past).
    For research labels, use known future only in label phase.
    """

    def evaluate(self, post_bars: pd.DataFrame, breakout_level: float, n: int = 3) -> dict:
        if post_bars is None or len(post_bars) == 0 or breakout_level <= 0:
            return {"ft_status": "UNKNOWN", "ft_sessions_above": 0, "ft_failed": False}
        sub = post_bars.head(n)
        closes = sub["close"].astype(float).values
        lows = sub["low"].astype(float).values
        above = int(np.sum(closes > breakout_level))
        failed = bool(np.any(lows < breakout_level * 0.98))
        if failed and above == 0:
            status = "FAILED"
        elif above >= max(1, n - 1):
            status = "CONFIRMED"
        else:
            status = "MIXED"
        return {
            "ft_status": status,
            "ft_sessions_above": above,
            "ft_failed": failed,
        }


class MarketStateEngine:
    """
    Spec §34 lifecycle (soft):
    BASE → ACCUMULATION → BREAKOUT → MARKUP → EXHAUSTION → DISTRIBUTION → MARKDOWN
    """
    ORDER = ["BASE", "ACCUMULATION", "BREAKOUT", "MARKUP", "EXHAUSTION", "DISTRIBUTION", "MARKDOWN", "UNKNOWN"]

    def classify(self, f: dict) -> dict:
        setup = str(f.get("setup") or "NONE")
        trend = float(f.get("factor_trend") or 50)
        flow = float(f.get("factor_flow") or 50)
        mom = float(f.get("factor_momentum") or 50)
        exit_r = float(f.get("exit_risk") or 0)
        vsa = str(f.get("vsa_state") or "—")
        above50 = int(f.get("above_sma50") or 0)
        atr = float(f.get("atr_pct") or 2)
        bb = float(f.get("bb_pctb") or 0.5) if f.get("bb_pctb") is not None else 0.5

        state = "UNKNOWN"
        if setup == "DISTRIBUTION" or vsa == "PHÂN PHỐI" or exit_r >= 70:
            state = "DISTRIBUTION"
        elif setup == "ACCUMULATION" or (atr < 1.5 and flow > 55 and trend < 55):
            state = "ACCUMULATION"
        elif setup == "BREAKOUT":
            state = "BREAKOUT"
        elif trend >= 70 and above50 and flow >= 50 and exit_r < 40:
            state = "MARKUP"
        elif trend >= 60 and mom < 40 and flow < 45:
            state = "EXHAUSTION"
        elif trend < 40 and not above50:
            state = "MARKDOWN"
        elif trend < 50 and atr < 2:
            state = "BASE"

        return {"market_state": state}


print("VSASequenceEngine / BreakoutFollowThroughEngine / MarketStateEngine defined.")


VSASequenceEngine / BreakoutFollowThroughEngine / MarketStateEngine defined.


In [55]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 44 — PURGED / EMBARGO WALK-FORWARD (Spec §54)                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

@dataclass
class PurgedFold:
    train_start: str
    train_end: str
    test_start: str
    test_end: str
    embargo_sessions: int


def make_purged_folds(
    start: str = "2022-01-01",
    end: str = "2025-12-31",
    train_years: int = 2,
    test_years: int = 1,
    embargo_sessions: int = 15,  # >= max label horizon
) -> List[PurgedFold]:
    """
    Embargo: gap between train_end and test_start in calendar approx.
    True session embargo applied when slicing dataset by shifting train_end back.
    """
    folds = []
    y0 = int(start[:4])
    y1 = int(end[:4])
    t = y0
    while t + train_years + test_years - 1 <= y1:
        tr_s = f"{t}-01-01"
        tr_e = f"{t + train_years - 1}-12-31"
        te_s = f"{t + train_years}-01-01"
        te_e = f"{t + train_years + test_years - 1}-12-31"
        folds.append(PurgedFold(tr_s, tr_e, te_s, te_e, embargo_sessions))
        t += 1
    return folds


def apply_embargo(train_df: pd.DataFrame, test_start: str, embargo: int = 15) -> pd.DataFrame:
    """Drop train rows within `embargo` calendar days of test_start (proxy for sessions)."""
    if train_df is None or len(train_df) == 0:
        return train_df
    d = train_df.copy()
    d["date"] = pd.to_datetime(d["date"])
    cutoff = pd.Timestamp(test_start) - pd.Timedelta(days=int(embargo * 1.5))  # ~sessions→calendar
    return d[d["date"] <= cutoff]


print(f"Purged folds example: {make_purged_folds()[:3]}")


Purged folds example: [PurgedFold(train_start='2022-01-01', train_end='2023-12-31', test_start='2024-01-01', test_end='2024-12-31', embargo_sessions=15), PurgedFold(train_start='2023-01-01', train_end='2024-12-31', test_start='2025-01-01', test_end='2025-12-31', embargo_sessions=15)]


In [56]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 45 — ABLATION A→H + FEATURE GROUPS (Spec §55–56)                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝

ABLATION_MODELS = {
    "A_trend": ["factor_trend"],
    "B_trend_mom": ["factor_trend", "factor_momentum"],
    "C_tm_flow": ["factor_trend", "factor_momentum", "factor_flow"],
    "D_plus_rs": ["factor_trend", "factor_momentum", "factor_flow", "factor_rs"],
    "E_plus_vol": ["factor_trend", "factor_momentum", "factor_flow", "factor_rs", "factor_volatility"],
    "F_full_factors": ["factor_trend", "factor_momentum", "factor_flow", "factor_rs", "factor_volatility", "factor_structure"],
    "G_plus_raw": [
        "factor_trend", "factor_momentum", "factor_flow", "factor_rs", "factor_volatility", "factor_structure",
        "rsi", "cmf", "volume_ratio", "atr_pct",
    ],
    "H_full": FEATURE_COLS_V2,
}

FEATURE_DROP_TESTS = {
    "full": FEATURE_COLS_V2,
    "no_flow": [c for c in FEATURE_COLS_V2 if c not in ("factor_flow", "cmf", "volume_ratio")],
    "no_rs": [c for c in FEATURE_COLS_V2 if c not in ("factor_rs", "rs_20d")],
    "no_trend": [c for c in FEATURE_COLS_V2 if c != "factor_trend"],
    "no_mom": [c for c in FEATURE_COLS_V2 if c not in ("factor_momentum", "rsi", "ret_5d")],
}


def _logit_fold_eval(tr, te, cols):
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.metrics import brier_score_loss
    cols = [c for c in cols if c in tr.columns and c in te.columns]
    tr = tr.dropna(subset=cols + ["tb_label"])
    te = te.dropna(subset=cols + ["tb_label"])
    tr = tr[tr["tb_label"].isin(["TP_FIRST", "SL_FIRST", "TIMEOUT"])]
    te = te[te["tb_label"].isin(["TP_FIRST", "SL_FIRST", "TIMEOUT"])]
    ytr = (tr["tb_label"] == "TP_FIRST").astype(int).values
    yte = (te["tb_label"] == "TP_FIRST").astype(int).values
    if len(tr) < 40 or len(te) < 8 or ytr.sum() < 3:
        return None
    sc = StandardScaler()
    Xtr = sc.fit_transform(tr[cols].astype(float).values)
    Xte = sc.transform(te[cols].astype(float).values)
    clf = LogisticRegression(max_iter=600, C=0.5, class_weight="balanced")
    clf.fit(Xtr, ytr)
    p = clf.predict_proba(Xte)[:, 1]
    hi = p >= np.quantile(p, 0.7)
    return {
        "n_test": len(te),
        "base_rate": float(yte.mean()),
        "brier": float(brier_score_loss(yte, p)),
        "top30_tp": float(yte[hi].mean()) if hi.sum() else np.nan,
        "top30_ret": float(te.loc[hi, "tb_ret"].mean()) if hi.sum() and "tb_ret" in te.columns else np.nan,
        "pred_mean": float(p.mean()),
    }


def run_ablation_suite(ds: pd.DataFrame = None, setups=("PULLBACK", "BREAKOUT")):
    ensure_sklearn()
    if ds is None:
        p = DATA_FEATURES / "features_v3_sector_exit.parquet"
        ds = pd.read_parquet(p) if p.exists() else pd.read_parquet(
            sorted(DATA_FEATURES.glob("features_v3*.parquet"))[-1]
        )
    d0 = ds.copy()
    d0["date"] = pd.to_datetime(d0["date"])
    if "setup" in d0.columns:
        d0 = d0[d0["setup"].isin(list(setups))]
    folds = make_purged_folds()
    rows = []
    print("═" * 60)
    print("  ABLATION A→H (purged folds, logistic)")
    print("═" * 60)
    for name, cols in ABLATION_MODELS.items():
        metrics = []
        for fold in folds:
            tr = d0[(d0["date"] >= fold.train_start) & (d0["date"] <= fold.train_end)]
            tr = apply_embargo(tr, fold.test_start, fold.embargo_sessions)
            te = d0[(d0["date"] >= fold.test_start) & (d0["date"] <= fold.test_end)]
            m = _logit_fold_eval(tr, te, cols)
            if m:
                metrics.append(m)
        if not metrics:
            print(f"  {name}: no valid folds")
            continue
        avg = {k: float(np.nanmean([m[k] for m in metrics])) for k in metrics[0]}
        avg["model"] = name
        avg["n_folds"] = len(metrics)
        rows.append(avg)
        print(f"  {name:16s} folds={len(metrics)} brier={avg['brier']:.4f} "
              f"top30_tp={avg['top30_tp']:.3f} top30_ret={avg['top30_ret']:.4f}")

    print("\n── Feature drop tests ──")
    for name, cols in FEATURE_DROP_TESTS.items():
        metrics = []
        for fold in folds:
            tr = apply_embargo(
                d0[(d0["date"] >= fold.train_start) & (d0["date"] <= fold.train_end)],
                fold.test_start, fold.embargo_sessions,
            )
            te = d0[(d0["date"] >= fold.test_start) & (d0["date"] <= fold.test_end)]
            m = _logit_fold_eval(tr, te, cols)
            if m:
                metrics.append(m)
        if metrics:
            avg = {k: float(np.nanmean([m[k] for m in metrics])) for k in metrics[0]}
            print(f"  {name:12s} brier={avg['brier']:.4f} top30_tp={avg['top30_tp']:.3f}")
            avg["model"] = name
            rows.append(avg)

    out = pd.DataFrame(rows)
    path = RESULTS_DIR / "ablation_suite.csv"
    out.to_csv(path, index=False)
    print(f"✓ {path}")
    return out


print("run_ablation_suite() ready")


run_ablation_suite() ready


In [57]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 46 — FULL PERFORMANCE METRICS (Spec §50)                          ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def compute_performance_metrics(trades: pd.DataFrame, initial_capital: float = 1e9) -> dict:
    if trades is None or len(trades) == 0:
        return {"n_trades": 0}
    t = trades.copy()
    ret = t["ret_net"].astype(float) if "ret_net" in t.columns else t.get("ret", pd.Series(dtype=float))
    ret = ret.dropna()
    wins = ret[ret > 0]
    losses = ret[ret <= 0]
    pnl = t["pnl_net"].astype(float) if "pnl_net" in t.columns else ret * 0
    equity = initial_capital + pnl.cumsum()
    peak = equity.cummax()
    dd = (equity - peak) / peak.replace(0, np.nan)
    max_dd = float(dd.min()) if len(dd) else 0.0

    # session-based annualization proxy
    n = len(ret)
    avg_sess = float(t["sessions"].mean()) if "sessions" in t.columns else 5.0
    # approx trades per year if ~250 sessions
    trades_per_year = max(1.0, 250.0 / max(avg_sess, 1) * max(n / max((t["exit_date"].nunique() if "exit_date" in t.columns else n), 1), 0.1))
    # simpler: use sum ret and scale
    total_ret = float(pnl.sum() / initial_capital) if len(pnl) else float(ret.sum())
    # CAGR proxy from total_ret over years from dates
    years = 1.0
    if "entry_date" in t.columns and "exit_date" in t.columns:
        d0 = pd.to_datetime(t["entry_date"]).min()
        d1 = pd.to_datetime(t["exit_date"]).max()
        years = max((d1 - d0).days / 365.25, 0.25)
    cagr = (1 + total_ret) ** (1 / years) - 1 if total_ret > -0.999 else -1.0

    vol = float(ret.std() * np.sqrt(max(trades_per_year, 1))) if len(ret) > 2 else np.nan
    mean_r = float(ret.mean())
    sharpe = (mean_r / ret.std() * np.sqrt(max(trades_per_year, 1))) if len(ret) > 2 and ret.std() > 1e-12 else np.nan
    downside = ret[ret < 0]
    sortino = (mean_r / downside.std() * np.sqrt(max(trades_per_year, 1))) if len(downside) > 2 and downside.std() > 1e-12 else np.nan
    calmar = (cagr / abs(max_dd)) if max_dd < 0 else np.nan

    pf = float(wins.sum() / abs(losses.sum())) if len(losses) and losses.sum() != 0 else np.nan
    metrics = {
        "n_trades": n,
        "win_rate_pct": round(100 * len(wins) / n, 2) if n else 0,
        "profit_factor": round(pf, 3) if not np.isnan(pf) else None,
        "expectancy_ret": round(mean_r, 5),
        "avg_win": round(float(wins.mean()), 5) if len(wins) else None,
        "avg_loss": round(float(losses.mean()), 5) if len(losses) else None,
        "total_return": round(total_ret, 4),
        "cagr_proxy": round(cagr, 4),
        "ann_vol_proxy": round(vol, 4) if not np.isnan(vol) else None,
        "sharpe_proxy": round(sharpe, 3) if not np.isnan(sharpe) else None,
        "sortino_proxy": round(sortino, 3) if not np.isnan(sortino) else None,
        "max_dd_pct": round(max_dd * 100, 2),
        "calmar_proxy": round(calmar, 3) if not np.isnan(calmar) else None,
        "avg_sessions": round(avg_sess, 2),
        "largest_win": round(float(ret.max()), 4) if n else None,
        "largest_loss": round(float(ret.min()), 4) if n else None,
        "sum_pnl_net": round(float(pnl.sum()), 0) if len(pnl) else None,
        "total_costs": round(float(t["cost"].sum()), 0) if "cost" in t.columns else None,
    }
    return metrics


def print_full_metrics(trades: pd.DataFrame, title: str = "BACKTEST METRICS"):
    m = compute_performance_metrics(trades)
    print("═" * 60)
    print(f"  {title}")
    print("═" * 60)
    for k, v in m.items():
        print(f"  {k:24s}: {v}")
    return m


print("compute_performance_metrics / print_full_metrics ready")


compute_performance_metrics / print_full_metrics ready


In [58]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 47 — VN LIMITS + SURVIVORSHIP / CORPORATE ACTIONS NOTES (Spec §66–68)
# ╚══════════════════════════════════════════════════════════════════════════╝

# HOSE ±7%, HNX ±10% typical (simplified; board rules can change)
VN_LIMIT_PCT = {
    "HOSE": 0.07,
    "HNX": 0.10,
    "UPCOM": 0.15,
    "DEFAULT": 0.07,
}


def detect_limit_hit(row_open, row_high, row_low, row_close, prev_close, board="HOSE") -> dict:
    if prev_close is None or prev_close <= 0:
        return {"limit_up": False, "limit_down": False}
    lim = VN_LIMIT_PCT.get(board, VN_LIMIT_PCT["DEFAULT"])
    up = prev_close * (1 + lim)
    dn = prev_close * (1 - lim)
    # tolerance for rounding
    limit_up = row_close >= up * 0.998 or row_high >= up * 0.998
    limit_down = row_close <= dn * 1.002 or row_low <= dn * 1.002
    return {"limit_up": bool(limit_up), "limit_down": bool(limit_down), "limit_pct": lim}


def execution_possible(signal_close_day_limit: dict, next_open_limit: dict) -> bool:
    """Spec §66: cannot assume fill if locked limit."""
    if signal_close_day_limit.get("limit_up") and next_open_limit.get("limit_up"):
        return False  # may not buy
    if signal_close_day_limit.get("limit_down") and next_open_limit.get("limit_down"):
        return False
    return True


SURVIVORSHIP_NOTE = """
SURVIVORSHIP BIAS (Spec §67)
----------------------------
This research universe (LIQUID_CORE / VN100 list as of notebook authoring date)
contains stocks that exist TODAY. Delisted / ticker-changed names are absent.
Historical backtests using this list OVERSTATE survivorship quality.
Do NOT claim backtests are free of survivorship bias without a point-in-time
constituent file (not available in community vnstock workflow used here).
"""

CORPORATE_ACTIONS_NOTE = """
CORPORATE ACTIONS (Spec §68)
----------------------------
OHLCV from vnstock is generally ADJUSTED for splits/dividends (source-dependent).
Volume consistency after adjustment is source-dependent.
All returns in this engine assume the series from MarketDataStore is usable as
adjusted total-return proxy — VERIFY for any production capital allocation.
"""

print(SURVIVORSHIP_NOTE)
print(CORPORATE_ACTIONS_NOTE)



SURVIVORSHIP BIAS (Spec §67)
----------------------------
This research universe (LIQUID_CORE / VN100 list as of notebook authoring date)
contains stocks that exist TODAY. Delisted / ticker-changed names are absent.
Historical backtests using this list OVERSTATE survivorship quality.
Do NOT claim backtests are free of survivorship bias without a point-in-time
constituent file (not available in community vnstock workflow used here).


CORPORATE ACTIONS (Spec §68)
----------------------------
OHLCV from vnstock is generally ADJUSTED for splits/dividends (source-dependent).
Volume consistency after adjustment is source-dependent.
All returns in this engine assume the series from MarketDataStore is usable as
adjusted total-return proxy — VERIFY for any production capital allocation.



In [59]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 48 — RESEARCH MODE API (Spec §63)                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝

class ResearchAPI:
    def __init__(self, ds: pd.DataFrame = None):
        self.ds = ds
        self.eng = EmpiricalProbabilityEngine(CFG)

    def load(self, path: str = None):
        if path:
            self.ds = pd.read_parquet(path)
        else:
            p = DATA_FEATURES / "features_v3_sector_exit.parquet"
            if not p.exists():
                p = sorted(DATA_FEATURES.glob("features_v3*.parquet"))[-1]
            self.ds = pd.read_parquet(p)
        print(f"ResearchAPI loaded {len(self.ds)} rows")
        return self

    def feature_analysis(self, col: str, n_bins: int = 5):
        assert self.ds is not None
        return self.eng.by_quantile(self.ds, col, n_bins=n_bins)

    def setup_analysis(self, setup: str = None):
        d = self.ds
        if setup:
            d = d[d["setup"] == setup]
            print(self.eng.summarize(d))
            return d
        return analyze_setups(self.ds, self.eng)

    def regime_analysis(self):
        if "market_trend" not in self.ds.columns:
            print("no market_trend")
            return None
        return self.eng.by_group(self.ds, "market_trend")

    def vsa_analysis(self):
        if "vsa_state" not in self.ds.columns:
            return None
        return self.eng.by_group(self.ds, "vsa_state")

    def exit_calibration(self):
        return calibrate_exit_bands(self.ds)

    def ablation(self):
        return run_ablation_suite(self.ds)

    def parameter_sensitivity(self, store=None):
        store = store or MarketDataStore(CFG)
        return barrier_sensitivity(self.ds, store, "PULLBACK")


research = ResearchAPI()
print("research = ResearchAPI()  # research.load(); research.setup_analysis(); research.ablation()")


research = ResearchAPI()  # research.load(); research.setup_analysis(); research.ablation()


In [60]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 49 — MASTER RUN (one-shot when you're ready)                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def run_all_research_and_scan(
    do_scan: bool = True,
    do_ablation: bool = True,
    do_logistic: bool = True,
    do_calib: bool = True,
    portfolio: dict = None,
):
    """
    Single entry for user: run after all cells defined.
    """
    register_vnstock_if_needed()
    print("═" * 60)
    print("  MASTER RUN — research + optional scan")
    print("═" * 60)
    print(SURVIVORSHIP_NOTE)
    print(CORPORATE_ACTIONS_NOTE)

    results = {}

    if do_calib:
        print("\n>>> EXIT CALIBRATION")
        results["ds"] = run_exit_calibration()
    else:
        p = DATA_FEATURES / "features_v3_sector_exit.parquet"
        results["ds"] = pd.read_parquet(p) if p.exists() else None

    if do_logistic:
        print("\n>>> LOGISTIC")
        try:
            results["preds"] = run_logistic_research(results.get("ds"))
        except Exception as e:
            print("logistic error:", e)
            results["preds"] = None

    if do_ablation:
        print("\n>>> ABLATION")
        try:
            results["ablation"] = run_ablation_suite(results.get("ds"))
        except Exception as e:
            print("ablation error:", e)
            results["ablation"] = None

    # state/vsa seq demo on dataset head
    if results.get("ds") is not None:
        ms = MarketStateEngine()
        vs = VSASequenceEngine()
        sample = results["ds"].head(10)
        print("\n>>> STATE / VSA SEQ sample")
        for _, r in sample.iterrows():
            st = ms.classify(r.to_dict())
            # sequence needs history — single day only shows NEUTRAL often
            print(f"  {r.get('symbol')} {r.get('date')} setup={r.get('setup')} → {st['market_state']}")

    print("\n>>> PORTFOLIO DEMO")
    PortfolioRiskEngine().analyze({
        "VCB": 0.18, "TCB": 0.16, "MBB": 0.14, "ACB": 0.12,
        "HPG": 0.10, "FPT": 0.15, "VNM": 0.15,
    })

    if do_scan:
        print("\n>>> DAILY SCAN v3")
        results["scan"] = run_daily_scan_v3(portfolio=portfolio)

    meta = ResearchRunMetadata(
        run_id=make_run_id(),
        timestamp=datetime.now().isoformat(),
        feature_version=CFG.feature_version + "+master",
        model_version="logistic_v1+ablation",
        config_version=CFG.config_version,
        data_start="2022-01-01", data_end="2026-07-31",
        universe=LIQUID_CORE, n_symbols=len(LIQUID_CORE),
        execution_model=CFG.execution_model,
        tp_pct=CFG.tp_pct, sl_pct=CFG.sl_pct,
        max_holding_sessions=CFG.max_holding_sessions,
        fee_buy=CFG.fee_buy, fee_sell=CFG.fee_sell, tax_sell=CFG.tax_sell,
        random_seed=CFG.random_seed,
        notes="master run: calib+logistic+ablation+scan",
    )
    meta.save()
    print("\n✓ MASTER RUN COMPLETE")
    return results


print("=" * 60)
print("WHEN READY, RUN ONCE:")
print("  results = run_all_research_and_scan(portfolio={'HDB':24.5,'PNJ':40.0})")
print("Or research only (no scan API load):")
print("  results = run_all_research_and_scan(do_scan=False)")
print("=" * 60)


WHEN READY, RUN ONCE:
  results = run_all_research_and_scan(portfolio={'HDB':24.5,'PNJ':40.0})
Or research only (no scan API load):
  results = run_all_research_and_scan(do_scan=False)


---

## Batch implemented this round

| Spec | Item |
|------|------|
| §23 | VSASequenceEngine |
| §33 | BreakoutFollowThroughEngine |
| §34 | MarketStateEngine |
| §50 | Full metrics (Sharpe/Sortino/CAGR/Calmar proxies) |
| §54 | Purged/embargo folds |
| §55–56 | Ablation A→H + feature drop |
| §63 | `ResearchAPI` facade |
| §66 | Limit-up/down helpers |
| §67–68 | Survivorship + corporate actions notes (printed) |
| — | `run_all_research_and_scan()` one-shot |

### Still thinner / future
- Full historical sector panel per date (API-heavy)
- Real session holiday calendar file
- Limit-up integrated into BacktestEngine fills
- LightGBM only if logistic shows structure
- Point-in-time universe for survivorship-free backtest

### One-shot
```python
results = run_all_research_and_scan(
    do_scan=True,
    portfolio={"HDB": 24.5, "VNM": 58.0, "PNJ": 40.0},
)
# or results = run_all_research_and_scan(do_scan=False)  # research only, faster
```


In [61]:
# results = run_all_research_and_scan(do_scan=False)
# results = run_all_research_and_scan(portfolio={'HDB':24.5,'VNM':58.0,'PNJ':40.0})


In [62]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 50 — SETUP REVERSAL + ENRICH STATE/VSA-SEQ (Spec §28, §23, §34)   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# Monkey-patch / extend SetupEngine.classify with REVERSAL
_original_setup_classify = SetupEngine.classify

def _setup_classify_with_reversal(self, f: dict) -> dict:
    base = _original_setup_classify(self, f)
    if base.get("setup") != "NONE":
        return base
    # REVERSAL: oversold-ish + flow turning + not in free-fall structure
    rsi = float(f.get("rsi") or 50)
    cmf = float(f.get("cmf") or 0)
    ret5 = float(f.get("ret_5d") or 0)
    flow = float(f.get("factor_flow") or 50)
    trend = float(f.get("factor_trend") or 50)
    bb = f.get("bb_pctb")
    bb = float(bb) if bb is not None and not (isinstance(bb, float) and np.isnan(bb)) else 0.5
    if rsi <= 35 and cmf > -0.05 and ret5 < -0.03 and flow >= 45 and trend >= 35 and bb < 0.25:
        return {
            "setup": "REVERSAL",
            "setup_score": 55.0,
            "setup_reasons": ["RSI oversold", "flow stabilizing", "near lower BB"],
        }
    return base

SetupEngine.classify = _setup_classify_with_reversal


def enrich_state_vsa_seq(ds: pd.DataFrame) -> pd.DataFrame:
    """Add market_state + vsa_seq_label (seq limited without full history per symbol)."""
    ms = MarketStateEngine()
    vs = VSASequenceEngine()
    rows = []
    # groupby symbol for sequential VSA if available
    if "vsa_state" in ds.columns and "symbol" in ds.columns:
        ds = ds.sort_values(["symbol", "date"])
        for sym, g in ds.groupby("symbol", sort=False):
            hist = []
            for _, r in g.iterrows():
                f = r.to_dict()
                st = ms.classify(f)
                hist.append(str(f.get("vsa_state") or "—"))
                seq = vs.classify(hist[-5:])
                f.update(st)
                f.update(seq)
                rows.append(f)
        return pd.DataFrame(rows)
    # fallback row-wise
    for _, r in ds.iterrows():
        f = r.to_dict()
        f.update(ms.classify(f))
        f.update(vs.classify([str(f.get("vsa_state") or "—")]))
        rows.append(f)
    return pd.DataFrame(rows)


print("REVERSAL setup enabled; enrich_state_vsa_seq() ready")


REVERSAL setup enabled; enrich_state_vsa_seq() ready


In [63]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 51 — SLIPPAGE=f(size/ADV) + LIMIT-AWARE ENTRY (Spec §47, §66)     ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def slippage_bps_from_participation(notional: float, adv_value: float,
                                    base_bps: float = 10.0) -> float:
    """
    participation = notional / ADV_value
    slippage grows with square-root-ish participation (simplified Almgren-like).
    """
    if adv_value is None or adv_value <= 0 or notional <= 0:
        return base_bps
    part = notional / adv_value
    # 0.5% ADV → ~base, 5% ADV → much higher
    extra = 150.0 * np.sqrt(max(part, 0.0))  # bps
    return float(min(base_bps + extra, 80.0))  # cap 80 bps


@dataclass
class BacktestConfigV3(BacktestConfigV2):
    use_adv_slippage: bool = True
    skip_limit_locked: bool = True
    board: str = "HOSE"


class BacktestEngineV3(BacktestEngineV2):
    """
    Extends v2:
      - dynamic slippage from size/ADV
      - skip entries when limit-locked (approx using signal bar vs next open)
    """

    def __init__(self, cfg: ResearchConfig = CFG, bt: BacktestConfigV3 = None):
        self.bt3 = bt or BacktestConfigV3()
        super().__init__(cfg, self.bt3)

    def run_from_dataset(self, ds: pd.DataFrame) -> pd.DataFrame:
        bt = self.bt3
        d = ds.copy()
        d["date"] = pd.to_datetime(d["date"])
        d = d.sort_values(["date", "symbol"])
        if bt.allowed_setups and "setup" in d.columns:
            d = d[d["setup"].isin(bt.allowed_setups)]
        if bt.use_exit_risk_gate and "exit_risk" in d.columns:
            d = d[d["exit_risk"] <= bt.max_exit_risk_to_enter]

        trades = []
        open_positions = []  # (exit_date, symbol)
        capital = bt.initial_capital
        store = MarketDataStore(self.cfg)

        for _, row in d.iterrows():
            sym = row["symbol"]
            sig_date = pd.Timestamp(row["date"]).normalize()
            # free positions
            open_positions = [(ed, s) for ed, s in open_positions if ed > sig_date]
            if len(open_positions) >= bt.max_open_positions:
                continue

            entry = float(row.get("entry_price") or row.get("close") or 0)
            if entry <= 0:
                continue
            # barriers from row if present
            tp = float(row.get("tp_price") or entry * (1 + self.cfg.tp_pct))
            sl = float(row.get("sl_price") or entry * (1 - self.cfg.sl_pct))
            max_h = int(row.get("max_hold") or self.cfg.max_holding_sessions)

            # ADV for slippage
            adv = float(row.get("avg_val_20d") or 0)
            risk_cash = capital * bt.risk_per_trade
            stop_dist = max(entry - sl, entry * 0.015)
            shares = risk_cash / stop_dist
            notional = shares * entry
            max_n = capital * bt.max_position_pct
            if notional > max_n:
                shares = max_n / entry
                notional = max_n

            if bt.use_adv_slippage and adv > 0:
                slip_bps = slippage_bps_from_participation(notional, adv, bt.slippage_bps)
            else:
                slip_bps = bt.slippage_bps

            # Limit check approximate: if ret_1d huge and volume dry — skip
            if bt.skip_limit_locked:
                r1 = row.get("ret_1d")
                if r1 is not None and not (isinstance(r1, float) and np.isnan(r1)):
                    if abs(float(r1)) >= 0.065 and float(row.get("volume_ratio") or 1) < 0.8:
                        continue  # likely limit / illiquid lock proxy

            # Simulate path from store if possible for realistic exit
            try:
                df = store.get_as_of(sym, (sig_date + pd.Timedelta(days=60)).strftime("%Y-%m-%d"),
                                    lookback_bars=80)
            except Exception:
                df = None

            if df is not None and len(df) > 2:
                df = df[df["time"] > sig_date].head(max_h + 1)
                if len(df) < 1:
                    continue
                # entry at first open
                e_px = float(df.iloc[0]["open"]) * (1 + slip_bps / 10000.0)
                exit_px = e_px
                exit_i = len(df) - 1
                reason = "TIMEOUT"
                for i in range(len(df)):
                    hi = float(df.iloc[i]["high"])
                    lo = float(df.iloc[i]["low"])
                    if lo <= sl:
                        exit_px = sl * (1 - slip_bps / 10000.0)
                        exit_i = i
                        reason = "SL"
                        break
                    if hi >= tp:
                        exit_px = tp * (1 - slip_bps / 10000.0)
                        exit_i = i
                        reason = "TP"
                        break
                else:
                    exit_px = float(df.iloc[-1]["close"]) * (1 - slip_bps / 10000.0)
                exit_date = pd.Timestamp(df.iloc[exit_i]["time"])
                sessions = exit_i + 1
            else:
                # fall back to label fields
                e_px = entry * (1 + slip_bps / 10000.0)
                lab = str(row.get("tb_label") or "TIMEOUT")
                if lab == "TP_FIRST":
                    exit_px = tp * (1 - slip_bps / 10000.0)
                    reason = "TP"
                elif lab == "SL_FIRST":
                    exit_px = sl * (1 - slip_bps / 10000.0)
                    reason = "SL"
                else:
                    exit_px = e_px * (1 + float(row.get("tb_ret") or 0))
                    reason = "TIMEOUT"
                sessions = int(row.get("tb_sessions") or max_h)
                exit_date = sig_date + pd.Timedelta(days=int(sessions * 1.5))

            fee = notional * (self.cfg.fee_buy + self.cfg.fee_sell + self.cfg.tax_sell)
            ret_net = (exit_px / e_px - 1.0) - (self.cfg.fee_buy + self.cfg.fee_sell + self.cfg.tax_sell)
            # adjust ret for actual slip already in prices roughly
            pnl = notional * ret_net
            trades.append({
                "symbol": sym,
                "entry_date": sig_date,
                "exit_date": exit_date,
                "setup": row.get("setup"),
                "market_trend": row.get("market_trend"),
                "entry_px": e_px,
                "exit_px": exit_px,
                "shares": shares,
                "notional": notional,
                "ret_net": ret_net,
                "pnl_net": pnl,
                "cost": fee + notional * (2 * slip_bps / 10000.0),
                "sessions": sessions,
                "reason": reason,
                "slip_bps": slip_bps,
            })
            open_positions.append((exit_date, sym))
            capital += pnl

        return pd.DataFrame(trades)


print("BacktestEngineV3 (ADV slippage + limit proxy) ready")


BacktestEngineV3 (ADV slippage + limit proxy) ready


In [64]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 52 — AUTO RESEARCH REPORT (Spec §81)                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def write_research_report(
    ds: pd.DataFrame = None,
    trades: pd.DataFrame = None,
    ablation: pd.DataFrame = None,
    path: str = None,
) -> Path:
    path = Path(path) if path else REPORTS_DIR / f"research_report_{datetime.now():%Y%m%d_%H%M}.md"
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    lines = []
    lines.append("# VN Quant Research Report\n")
    lines.append(f"Generated: {datetime.now().isoformat()}\n")
    lines.append(SURVIVORSHIP_NOTE)
    lines.append(CORPORATE_ACTIONS_NOTE)
    lines.append("\n## 1. Dataset\n")
    if ds is not None and len(ds):
        lines.append(f"- rows: {len(ds)}\n")
        lines.append(f"- symbols: {ds['symbol'].nunique() if 'symbol' in ds.columns else 'n/a'}\n")
        if "date" in ds.columns:
            lines.append(f"- date range: {ds['date'].min()} → {ds['date'].max()}\n")
        if "setup" in ds.columns:
            lines.append("- setups:\n")
            for k, v in ds["setup"].value_counts().items():
                lines.append(f"  - {k}: {v}\n")
        eng = EmpiricalProbabilityEngine(CFG)
        s = eng.summarize(ds)
        lines.append("\n## 2. Overall triple-barrier\n")
        for k, v in s.items():
            lines.append(f"- {k}: {v}\n")
        lines.append("\n## 3. By setup\n")
        st = analyze_setups(ds, eng)
        if st is not None and len(st):
            lines.append(st.to_string(index=False))
            lines.append("\n")
        if "exit_band" in ds.columns and "future_return_10d" in ds.columns:
            lines.append("\n## 4. Exit band calibration\n")
            g = ds.groupby("exit_band")["future_return_10d"].agg(["count", "mean"])
            lines.append(g.to_string())
            lines.append("\n")
    else:
        lines.append("_No dataset provided._\n")

    lines.append("\n## 5. Backtest metrics\n")
    if trades is not None and len(trades):
        m = compute_performance_metrics(trades)
        for k, v in m.items():
            lines.append(f"- {k}: {v}\n")
    else:
        lines.append("_No trades provided._\n")

    lines.append("\n## 6. Ablation\n")
    if ablation is not None and len(ablation):
        lines.append(ablation.to_string(index=False))
        lines.append("\n")
    else:
        lines.append("_Run run_ablation_suite() first._\n")

    lines.append("\n## 7. Philosophy reminder\n")
    lines.append("Decisions should be probability × expected return × risk/reward, ")
    lines.append("not certainty that price will rise.\n")
    lines.append("\n## 8. Known gaps\n")
    lines.append("- Survivorship bias present (today's list)\n")
    lines.append("- Sector strength historically incomplete under community API\n")
    lines.append("- Exit bands need ongoing calibration\n")
    lines.append("- OOS edge after costs not yet robust on PULLBACK alone\n")

    path.write_text("".join(lines), encoding="utf-8")
    print(f"✓ Research report → {path}")
    return path


print("write_research_report() ready")


write_research_report() ready


In [65]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 53 — MASTER RUN v2 (includes report + bt v3)                      ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def run_all_v2(
    do_scan: bool = False,
    do_ablation: bool = True,
    do_logistic: bool = True,
    do_backtest: bool = True,
    portfolio: dict = None,
):
    register_vnstock_if_needed()
    print("═" * 60)
    print("  MASTER RUN v2")
    print("═" * 60)
    print(SURVIVORSHIP_NOTE)
    print(CORPORATE_ACTIONS_NOTE)

    out = {}
    out["ds"] = run_exit_calibration()
    if out["ds"] is not None:
        out["ds"] = enrich_state_vsa_seq(out["ds"])
        out["ds"].to_parquet(DATA_FEATURES / "features_v3_full_enrich.parquet", index=False)
        print("  states:", out["ds"]["market_state"].value_counts().to_dict() if "market_state" in out["ds"].columns else {})
        print("  setups:", out["ds"]["setup"].value_counts().to_dict() if "setup" in out["ds"].columns else {})

    if do_logistic:
        try:
            out["preds"] = run_logistic_research(out["ds"])
        except Exception as e:
            print("logistic:", e)
            out["preds"] = None

    if do_ablation:
        try:
            out["ablation"] = run_ablation_suite(out["ds"])
        except Exception as e:
            print("ablation:", e)
            out["ablation"] = None

    trades = None
    if do_backtest and out.get("ds") is not None:
        print("\n>>> BACKTEST v3 PULLBACK+BREAKOUT")
        sub = out["ds"][out["ds"]["setup"].isin(["PULLBACK", "BREAKOUT", "REVERSAL"])]
        eng = BacktestEngineV3(CFG, BacktestConfigV3(
            allowed_setups=("PULLBACK", "BREAKOUT", "REVERSAL"),
            use_exit_risk_gate=True,
            max_exit_risk_to_enter=45,
            use_adv_slippage=True,
            risk_per_trade=0.005,
            max_open_positions=4,
        ))
        trades = eng.run_from_dataset(sub)
        out["trades"] = trades
        if len(trades):
            print_full_metrics(trades, "BACKTEST v3 (ADV slip + costs)")
            trades.to_parquet(RESULTS_DIR / "backtest_v3_trades.parquet", index=False)
        else:
            print("  no trades")

    print("\n>>> PORTFOLIO")
    PortfolioRiskEngine().analyze(portfolio or {
        "VCB": 0.15, "TCB": 0.12, "MBB": 0.10, "FPT": 0.15,
        "HPG": 0.10, "MWG": 0.10, "VNM": 0.10, "GAS": 0.08, "SSI": 0.10,
    })

    if do_scan:
        out["scan"] = run_daily_scan_v3(portfolio=portfolio)

    out["report"] = write_research_report(
        ds=out.get("ds"), trades=trades, ablation=out.get("ablation"),
    )

    ResearchRunMetadata(
        run_id=make_run_id(), timestamp=datetime.now().isoformat(),
        feature_version=CFG.feature_version + "+v2master",
        model_version="logistic+ablation+btv3",
        config_version=CFG.config_version,
        data_start="2022-01-01", data_end="2026-07-31",
        universe=LIQUID_CORE, n_symbols=len(LIQUID_CORE),
        execution_model=CFG.execution_model,
        tp_pct=CFG.tp_pct, sl_pct=CFG.sl_pct,
        max_holding_sessions=CFG.max_holding_sessions,
        fee_buy=CFG.fee_buy, fee_sell=CFG.fee_sell, tax_sell=CFG.tax_sell,
        random_seed=CFG.random_seed,
        notes="master v2: state/seq/btv3/report",
    ).save()

    print("\n✓ MASTER v2 COMPLETE → see reports/")
    return out


print("run_all_v2() ready")
print("→ results = run_all_v2(do_scan=False)          # research + bt + report")
print("→ results = run_all_v2(do_scan=True, portfolio={...})")


run_all_v2() ready
→ results = run_all_v2(do_scan=False)          # research + bt + report
→ results = run_all_v2(do_scan=True, portfolio={...})


---

## Added this round

| Spec | Item |
|------|------|
| §28 | **REVERSAL** setup |
| §23/34 | `enrich_state_vsa_seq` on dataset |
| §47 | Slippage = f(participation / ADV) |
| §66 | Limit-lock entry skip (proxy) in **BacktestEngineV3** |
| §50 | `print_full_metrics` wired to v3 backtest |
| §81 | `write_research_report()` → `reports/research_report_*.md` |
| — | `run_all_v2()` one-shot |

### Recommended single run
```python
results = run_all_v2(do_scan=False)
# full day including scanner:
# results = run_all_v2(do_scan=True, portfolio={"HDB": 24.5, "PNJ": 40.0})
```

Outputs:
- `data/features/features_v3_full_enrich.parquet`
- `results/backtest_v3_trades.parquet`
- `results/ablation_suite.csv`
- `reports/research_report_*.md`


In [66]:
# results = run_all_v2(do_scan=False)
# results = run_all_v2(do_scan=True, portfolio={'HDB':24.5,'VNM':58.0,'PNJ':40.0})


In [67]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 54 — DISPLAY TABLE + HTML EXPORT (user feature, not in Spec)      ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Reference: legacy v8 display_table + html_report style
# Purpose: notebook-friendly styled table (copy → Excel) + standalone HTML

from pathlib import Path

def _ipy_available():
    try:
        from IPython.display import display as ipy_display
        return True, ipy_display
    except ImportError:
        return False, None


def build_scan_display_df(scan_out: dict) -> pd.DataFrame:
    """
    Flatten scanner v3 output into one table for display / Excel.
    Columns bilingual-ish to match user habit.
    """
    rows = []
    # FULL list: all_new > new
    new_df = scan_out.get("all_new")

    if new_df is None or len(new_df) == 0:
        new_df = scan_out.get("new")

    if new_df is not None and len(new_df):
        for _, r in new_df.iterrows():
            rows.append({
                "Mã": r.get("symbol"),
                "Audience": "NEW",
                "Action": r.get("action"),
                "Setup": r.get("setup"),
                "Sector": r.get("sector"),
                "Giá": r.get("close"),
                "Alpha": r.get("alpha_score"),
                "P_proxy": r.get("p_proxy"),
                "VSA": r.get("vsa_state"),
                "TrendF": r.get("factor_trend"),
                "FlowF": r.get("factor_flow"),
                "RS_F": r.get("factor_rs"),
                "SecStr": r.get("sector_strength"),
                "Entry": r.get("entry_suggest"),
                "SL": r.get("sl"),
                "TP1": r.get("tp1"),
                "TP2": r.get("tp2"),
                "R:R": r.get("rr_tp1"),
                "SL%": r.get("sl_pct"),
                "TP1%": r.get("tp1_pct"),
                "ExitRisk": r.get("exit_risk"),
                "Lý do": r.get("reasons"),
                "Size hint": r.get("size_hint"),
            })
    hold_df = scan_out.get("holder")
    if hold_df is not None and len(hold_df):
        for _, r in hold_df.iterrows():
            rows.append({
                "Mã": r.get("symbol"),
                "Audience": "HOLDER",
                "Action": r.get("action"),
                "Setup": r.get("setup"),
                "Sector": r.get("sector"),
                "Giá": r.get("close"),
                "Alpha": r.get("alpha_score"),
                "P_proxy": r.get("p_proxy"),
                "VSA": r.get("vsa_state"),
                "TrendF": r.get("factor_trend"),
                "FlowF": r.get("factor_flow"),
                "RS_F": r.get("factor_rs"),
                "SecStr": r.get("sector_strength"),
                "Entry": r.get("ref_close"),
                "SL": r.get("trail_sl"),
                "TP1": r.get("tp1_hold"),
                "TP2": r.get("tp2_hold"),
                "R:R": np.nan,
                "SL%": np.nan,
                "TP1%": np.nan,
                "ExitRisk": r.get("exit_risk"),
                "Lý do": r.get("reasons"),
                "Size hint": r.get("unrealized_pnl_pct"),
            })
    return pd.DataFrame(rows)


def display_scan_table(scan_out: dict, title: str = None):
    """Styled table in notebook; falls back to print."""
    in_nb, ipy_display = _ipy_available()
    d = build_scan_display_df(scan_out)
    if d is None or len(d) == 0:
        print("(empty scan table)")
        return d

    regime = scan_out.get("regime") or {}
    breadth = scan_out.get("breadth") or {}
    caption = title or (
        f"<b>VN Quant Research v2</b> | "
        f"Regime: {regime.get('label', regime.get('trend', '?'))} | "
        f"Breadth: {breadth.get('breadth_label', regime.get('breadth', '—'))} | "
        f"{datetime.now():%d/%m/%Y %H:%M}"
    )

    if not in_nb:
        pd.set_option("display.max_columns", None)
        pd.set_option("display.width", 220)
        print(caption.replace("<b>", "").replace("</b>", ""))
        print(d.to_string(index=False))
        return d

    def row_bg(row):
        a = str(row.get("Action", ""))
        aud = str(row.get("Audience", ""))
        if a in ("NEW_BUY", "HOLD_OR_SCALE_IN", "GIA TĂNG"):
            bg = "#0d2e16"
        elif a in ("WAIT_DIP", "HOLD", "WATCH"):
            bg = "#1a2332"
        elif a in ("REDUCE", "AVOID"):
            bg = "#2e2410"
        elif a in ("EXIT",):
            bg = "#2e1a1a"
        elif aud == "HOLDER":
            bg = "#1a1f2e"
        else:
            bg = "#1a1f2e"
        return [f"background-color:{bg}" for _ in row]

    def c_action(v):
        s = str(v)
        if s in ("NEW_BUY", "HOLD_OR_SCALE_IN"):
            return "color:#4ade80;font-weight:bold"
        if s in ("WAIT_DIP", "HOLD"):
            return "color:#60a5fa;font-weight:bold"
        if s == "WATCH":
            return "color:#facc15"
        if s in ("REDUCE", "AVOID"):
            return "color:#f59e0b;font-weight:bold"
        if s == "EXIT":
            return "color:#f87171;font-weight:bold"
        return "color:#94a3b8"

    def c_alpha(v):
        try:
            f = float(v)
            if f >= 75: return "color:#4ade80;font-weight:bold"
            if f >= 60: return "color:#facc15"
            return "color:#f87171"
        except Exception:
            return ""

    def c_rr(v):
        try:
            f = float(v)
            if f >= 2: return "color:#4ade80;font-weight:bold"
            if f >= 1.5: return "color:#facc15"
            return "color:#f87171"
        except Exception:
            return ""

    def c_exit(v):
        try:
            f = float(v)
            if f >= 70: return "color:#f87171;font-weight:bold"
            if f >= 40: return "color:#f59e0b"
            return "color:#4ade80"
        except Exception:
            return ""

    def sub(*cols):
        return [c for c in cols if c in d.columns]

    styled = (
        d.style
        .apply(row_bg, axis=1)
        .map(c_action, subset=sub("Action"))
        .map(c_alpha, subset=sub("Alpha"))
        .map(c_rr, subset=sub("R:R"))
        .map(c_exit, subset=sub("ExitRisk"))
        .set_table_styles([
            {"selector": "table", "props": [
                ("border-collapse", "collapse"), ("width", "100%"),
                ("font-size", "11px"), ("font-family", "'Segoe UI',monospace"),
            ]},
            {"selector": "thead tr", "props": [("background-color", "#1e293b")]},
            {"selector": "th", "props": [
                ("color", "#e2e8f0"), ("padding", "5px 7px"),
                ("text-align", "center"), ("font-size", "10px"),
                ("white-space", "nowrap"), ("border-bottom", "2px solid #334155"),
            ]},
            {"selector": "td", "props": [
                ("padding", "4px 7px"), ("text-align", "center"),
                ("border-bottom", "1px solid #1e293b"), ("white-space", "nowrap"),
                ("color", "#e2e8f0"),
            ]},
            {"selector": "tr:hover td", "props": [("background-color", "rgba(148,163,184,0.08)")]},
            {"selector": "caption", "props": [
                ("caption-side", "top"), ("color", "#94a3b8"),
                ("font-size", "12px"), ("padding", "8px"), ("text-align", "left"),
            ]},
        ])
        .set_caption(caption)
        .format({
            "Giá": "{:,.2f}", "Alpha": "{:.1f}", "P_proxy": "{:.3f}",
            "TrendF": "{:.0f}", "FlowF": "{:.0f}", "RS_F": "{:.0f}", "SecStr": "{:.0f}",
            "Entry": "{:,.2f}", "SL": "{:,.2f}", "TP1": "{:,.2f}", "TP2": "{:,.2f}",
            "R:R": "{:.2f}", "SL%": "{:.1f}", "TP1%": "{:.1f}", "ExitRisk": "{:.0f}",
        }, na_rep="—")
    )
    ipy_display(styled)
    return d


def df_to_excel_friendly_html_table(df: pd.DataFrame) -> str:
    """Plain HTML table that pastes cleanly into Excel."""
    if df is None or len(df) == 0:
        return "<p>(empty)</p>"
    return df.to_html(index=False, border=1, justify="center", float_format=lambda x: f"{x:.4g}")


def write_html_report(
    scan_out: dict = None,
    trades: pd.DataFrame = None,
    metrics: dict = None,
    ablation: pd.DataFrame = None,
    path: str = None,
    title: str = "VN Quant Research Engine v2 — Report",
) -> str:
    """
    Standalone dark-theme HTML (open in browser / share).
    Also embeds a clean table block for copy-paste into Excel.
    """
    path = path or str(REPORTS_DIR / f"dashboard_{datetime.now():%Y%m%d_%H%M}.html")
    Path(path).parent.mkdir(parents=True, exist_ok=True)

    regime = (scan_out or {}).get("regime") or {}
    breadth = (scan_out or {}).get("breadth") or {}
    sec = (scan_out or {}).get("sectors")
    disp = build_scan_display_df(scan_out) if scan_out else pd.DataFrame()

    # metrics chips
    metrics = metrics or {}
    if trades is not None and len(trades) and not metrics:
        metrics = compute_performance_metrics(trades)

    chips = []
    for k, label in [
        ("n_trades", "Trades"), ("win_rate_pct", "Win%"), ("profit_factor", "PF"),
        ("expectancy_ret", "Expectancy"), ("cagr_proxy", "CAGR"), ("sharpe_proxy", "Sharpe"),
        ("max_dd_pct", "MaxDD%"), ("total_return", "TotalRet"),
    ]:
        if k in metrics and metrics[k] is not None:
            chips.append(f"<span>{label}: <b>{metrics[k]}</b></span>")
    chips_html = "\n".join(chips) if chips else "<span>No backtest metrics</span>"

    # scan table
    if len(disp):
        # color pnl-like columns not applicable; plain html for excel
        scan_table = disp.to_html(index=False, classes="data", border=0, float_format=lambda x: f"{x:.4g}")
    else:
        scan_table = "<p>No scan rows</p>"

    # trades last 50
    trades_html = ""
    if trades is not None and len(trades):
        t = trades.tail(50).copy()
        rows = []
        for _, r in t.iterrows():
            pnl = float(r.get("pnl_net") or r.get("pnl") or 0)
            ret = float(r.get("ret_net") or 0)
            color = "#16a34a" if pnl >= 0 else "#dc2626"
            rows.append(
                f"<tr>"
                f"<td>{r.get('symbol','')}</td>"
                f"<td>{r.get('entry_date','')}</td><td>{r.get('exit_date','')}</td>"
                f"<td>{float(r.get('entry_px') or r.get('entry_price') or 0):.2f}</td>"
                f"<td>{float(r.get('exit_px') or r.get('exit_price') or 0):.2f}</td>"
                f"<td style='color:{color}'>{pnl:,.0f}</td>"
                f"<td style='color:{color}'>{ret:+.2%}</td>"
                f"<td>{r.get('reason') or r.get('exit_reason') or ''}</td>"
                f"<td>{r.get('sessions') or r.get('holding_days') or ''}</td>"
                f"<td>{r.get('setup','')}</td>"
                f"</tr>"
            )
        trades_html = "\n".join(rows)

    sec_html = ""
    if sec is not None and len(sec):
        sec_html = sec.head(300).to_html(index=False, float_format=lambda x: f"{x:.4g}")

    ab_html = ""
    if ablation is not None and len(ablation):
        ab_html = ablation.to_html(index=False, float_format=lambda x: f"{x:.4g}")

    html = f"""<!DOCTYPE html>
<html><head><meta charset="utf-8"><title>{title}</title>
<style>
body{{font-family:system-ui,-apple-system,sans-serif;margin:24px;background:#0f172a;color:#e2e8f0}}
h1,h2{{color:#38bdf8}}
.card{{background:#1e293b;border-radius:12px;padding:16px 20px;margin:12px 0}}
.metrics span{{display:inline-block;margin:6px 16px 6px 0;padding:8px 12px;background:#334155;border-radius:8px}}
table{{border-collapse:collapse;width:100%;font-size:12px}}
th,td{{border:1px solid #334155;padding:6px 8px;text-align:right}}
th{{background:#334155;text-align:center;color:#e2e8f0}}
td:first-child,th:first-child{{text-align:left}}
tr:nth-child(even){{background:#1a2332}}
.note{{color:#94a3b8;font-size:13px}}
a{{color:#38bdf8}}
</style></head><body>
<h1>{title}</h1>
<p class="note">Generated {datetime.now():%Y-%m-%d %H:%M} · Research decision-support, not guaranteed alpha</p>

<div class="card">
<h2>Market status</h2>
<p>
  Regime: <b>{regime.get('label', regime.get('trend', '—'))}</b> ·
  Breadth: <b>{breadth.get('breadth_label', regime.get('breadth', '—'))}</b>
  (MA20 {breadth.get('pct_above_ma20','—')}% / MA50 {breadth.get('pct_above_ma50','—')}% / n={breadth.get('n_universe','—')}) ·
  Liquidity: {regime.get('liquidity','—')} · Risk: {regime.get('risk_appetite','—')}
</p>
</div>

<div class="card metrics">
<h2>Backtest metrics</h2>
{chips_html}
</div>

<div class="card">
<h2>Scan — NEW + HOLDER (copy table into Excel)</h2>
{scan_table}
</div>
"""
    if sec_html:
        html += f'<div class="card"><h2>Sectors</h2>{sec_html}</div>\n'
    if ab_html:
        html += f'<div class="card"><h2>Ablation</h2>{ab_html}</div>\n'
    if trades_html:
        html += f"""
<div class="card"><h2>Trades (last 50)</h2>
<table>
<thead><tr>
<th>Symbol</th><th>Entry</th><th>Exit</th><th>Entry Px</th><th>Exit Px</th>
<th>PnL</th><th>Ret</th><th>Reason</th><th>Sessions</th><th>Setup</th>
</tr></thead>
<tbody>
{trades_html}
</tbody></table></div>
"""
    html += f"""
<div class="card note">
<p>{SURVIVORSHIP_NOTE.replace(chr(10), '<br>')}</p>
</div>
</body></html>
"""
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"✓ HTML report → {path}")
    return path


def export_scan_excel_csv(scan_out: dict, stem: str = None) -> dict:
    """CSV + HTML for Excel workflow."""
    stem = stem or f"scan_export_{datetime.now():%Y%m%d_%H%M}"
    d = build_scan_display_df(scan_out)
    paths = {}
    if len(d):
        p_csv = RESULTS_DIR / f"{stem}.csv"
        d.to_csv(p_csv, index=False)
        paths["csv"] = str(p_csv)
        print(f"✓ CSV → {p_csv}")
    p_html = write_html_report(scan_out=scan_out, path=str(REPORTS_DIR / f"{stem}.html"))
    paths["html"] = p_html
    return paths


print("display_scan_table / write_html_report / export_scan_excel_csv ready")


display_scan_table / write_html_report / export_scan_excel_csv ready


In [68]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 55 — CONVENIENCE: scan → display → HTML                           ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def run_scan_and_export(portfolio: dict = None, full: bool = False, show: bool = True):
    """
    One call: scan v3 → styled table → HTML + CSV.
    """
    out = run_daily_scan_v3(portfolio=portfolio, full=full)
    if show:
        display_scan_table(out)
    paths = export_scan_excel_csv(out)
    out["export_paths"] = paths
    return out


def export_backtest_html(trades: pd.DataFrame, path: str = None, title: str = "Backtest v3 Report"):
    m = compute_performance_metrics(trades) if trades is not None and len(trades) else {}
    return write_html_report(trades=trades, metrics=m, path=path, title=title)


# Patch run_all_v2 to always write HTML at end if scan/trades exist
_run_all_v2_orig = run_all_v2

def run_all_v2(*args, **kwargs):
    out = _run_all_v2_orig(*args, **kwargs)
    try:
        html_path = write_html_report(
            scan_out=out.get("scan"),
            trades=out.get("trades"),
            ablation=out.get("ablation"),
            title="VN Quant Research v2 — Master Report",
        )
        out["html"] = html_path
        if out.get("scan"):
            display_scan_table(out["scan"])
    except Exception as e:
        print("HTML export skip:", e)
    return out


print("run_scan_and_export() ready")
print("→ out = run_scan_and_export(portfolio={'HDB':24.5})")
print("→ export_backtest_html(trades)")
print("→ results = run_all_v2(do_scan=True)  # includes HTML")


run_scan_and_export() ready
→ out = run_scan_and_export(portfolio={'HDB':24.5})
→ export_backtest_html(trades)
→ results = run_all_v2(do_scan=True)  # includes HTML


---

## Display + HTML export (ngoài Spec — theo request)

### Notebook (copy Excel)
```python
out = run_scan_and_export(portfolio={"HDB": 24.5, "PNJ": 40.0})
# hoặc sau scan:
# display_scan_table(out)
```

### File
- `results/scan_export_*.csv` — mở Excel trực tiếp  
- `reports/scan_export_*.html` / `dashboard_*.html` — mở browser, copy bảng  

### Backtest only
```python
export_backtest_html(trades)
```

### Master
```python
results = run_all_v2(do_scan=True, portfolio={...})
# → reports/*.html + styled table nếu có scan
```

Style: dark theme tương tự v8 (`#0f172a` / cards / metrics chips).


In [69]:
# out = run_scan_and_export()
# out = run_scan_and_export(portfolio={'HDB':24.5,'VNM':58.0,'PNJ':40.0})


In [70]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 56 — VERBOSE SCAN (fixed: no full-universe breadth upfront)       ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# FIX: breadth/sector trên toàn VN100 trước vòng lặp → timeout API, không in mã nào.
# Giờ: in từng mã NGAY; breadth tính sau trên subset đã fetch thành công.

import logging
import warnings

def _quiet_logs():
    """Giảm spam Google Drive / vnstock / urllib3."""
    warnings.filterwarnings("ignore")
    for name in [
        "vnstock", "vnstock.core", "vnstock.core.utils", "vnstock.core.utils.client",
        "vnai", "urllib3", "google", "googleapiclient",
    ]:
        logging.getLogger(name).setLevel(logging.ERROR)
    # root noise
    logging.getLogger().setLevel(logging.ERROR)


def _fmt_progress_line(i: int, n: int, sym: str, f: dict, rec_new: dict = None,
                       rec_hold: dict = None, is_portfolio: bool = False) -> str:
    close = f.get("close")
    setup = str(f.get("setup") or "—")
    vsa = str(f.get("vsa_state") or "—")
    trend = f.get("factor_trend")
    flow = f.get("factor_flow")
    alpha = (rec_new or {}).get("alpha_score", f.get("alpha_score"))
    action = (rec_hold or {}).get("action") if is_portfolio else (rec_new or {}).get("action")
    exit_r = f.get("exit_risk")
    sector = f.get("sector") or ""

    act = str(action or "—")
    if act in ("NEW_BUY", "HOLD_OR_SCALE_IN"):
        mark = "✅"
    elif act in ("WAIT_DIP", "HOLD"):
        mark = "·"
    elif act in ("WATCH",):
        mark = "👀"
    elif act in ("REDUCE", "AVOID"):
        mark = "⚠"
    elif act in ("EXIT",):
        mark = "🚫"
    else:
        mark = "·"

    try:
        t = float(trend or 50)
        if t >= 70: t_lbl = "🟢 Up"
        elif t >= 55: t_lbl = "🔵 WeakUp"
        elif t >= 45: t_lbl = "🟡 Side"
        elif t >= 35: t_lbl = "🟠 WeakDn"
        else: t_lbl = "🔴 Down"
    except Exception:
        t_lbl = "—"

    try:
        fl = float(flow or 50)
        flow_lbl = "💰 Flow+" if fl >= 60 else ("📤 Flow-" if fl < 40 else "🔄 Flow~")
    except Exception:
        flow_lbl = ""

    tp1 = (rec_new or {}).get("tp1_pct")
    tp_s = f"TP1:+{tp1:.1f}%" if isinstance(tp1, (int, float)) and tp1 == tp1 else ""
    al_s = f"{float(alpha):.1f}" if alpha is not None and alpha == alpha else "—"
    er_s = f"XR:{float(exit_r):.0f}" if exit_r is not None else ""
    px = f"{float(close):.2f}" if close is not None else "—"
    port = " [PF]" if is_portfolio else ""

    return (
        f"  [{i}/{n}] {sym:6s}{port}  {mark} {al_s:>5s}  {act:<16s}  "
        f"px:{px}  {setup:<12s}  {t_lbl}  {flow_lbl}  VSA:{vsa}  {tp_s}  {er_s}  {sector}"
    ).rstrip()


def scan_vn100_verbose(
    symbols,
    portfolio: dict = None,
    min_avg_val: float = 1e6,
    sleep_sec: float = 2.0,
    top_n: int = 300,
    max_retries: int = 2,
):
    """
    Full scan with LIVE per-symbol print.
    - Không compute breadth full-universe trước (gây timeout im lặng).
    - Retry nhẹ khi API timeout.
    - Sleep sau mỗi mã.
    """
    _quiet_logs()
    portfolio = portfolio or {}
    register_vnstock_if_needed()
    store = MarketDataStore(CFG)
    scanner = DailyScannerV3(store, CFG)
    rec_eng = RecommendationEngine(CFG)

    n = len(symbols)
    print("═" * 60)
    print(f"  VERBOSE SCAN | {n} symbols | sleep={sleep_sec}s | {datetime.now():%Y-%m-%d %H:%M}")
    print("═" * 60)

    # Regime from VNINDEX only (1 request) — nhanh
    end = datetime.today().strftime("%Y-%m-%d")
    start = (datetime.today() - timedelta(days=400)).strftime("%Y-%m-%d")
    print("  Loading VNINDEX for regime...")
    df_vni = None
    for attempt in range(max_retries + 1):
        try:
            df_vni = store.get("VNINDEX", start, end)
            if df_vni is not None and len(df_vni):
                break
        except Exception as e:
            print(f"  VNINDEX attempt {attempt+1} failed: {e}")
            time.sleep(3)
    regime = scanner.regime_eng.detect(df_vni.tail(60) if df_vni is not None else None)
    print(f"  Market regime: {regime.get('label')} | VNI={regime.get('vnindex')}")
    print("  (Breadth/sector: tính sau trên mã fetch OK — không block upfront)\n")

    # Placeholder breadth — update at end
    breadth = {
        "n_universe": 0, "pct_above_ma20": np.nan, "pct_above_ma50": np.nan,
        "pct_above_ma200": np.nan, "advance_decline": np.nan, "breadth_label": "PENDING",
    }
    regime = dict(regime)
    regime["breadth"] = "PENDING"

    rows = []
    n_ok = n_skip = n_err = 0
    # for post-hoc breadth
    above20 = above50 = above200 = 0
    up = down = 0

    for i, sym in enumerate(symbols, 1):
        is_pf = sym in portfolio
        f = None
        last_err = None
        for attempt in range(max_retries + 1):
            try:
                f = scanner._features_one(sym, df_vni)
                last_err = None
                break
            except Exception as e:
                last_err = e
                msg = str(e).lower()
                if "timeout" in msg or "timed out" in msg or "connection" in msg:
                    print(f"  [{i}/{n}] {sym:6s}  ⏳ timeout retry {attempt+1}/{max_retries}...")
                    time.sleep(sleep_sec + 1)
                    continue
                break

        if f is None:
            if last_err:
                print(f"  [{i}/{n}] {sym:6s}  — ERROR {type(last_err).__name__}: {str(last_err)[:60]}")
                n_err += 1
            else:
                print(f"  [{i}/{n}] {sym:6s}  — skip (no data)")
                n_skip += 1
            time.sleep(sleep_sec)
            continue

        try:
            if float(f.get("avg_val_20d") or 0) < min_avg_val and not is_pf:
                print(f"  [{i}/{n}] {sym:6s}  — skip (low ADV)")
                n_skip += 1
                time.sleep(sleep_sec)
                continue

            f = scanner.sector_eng.attach_to_features(f, sym)
            try:
                f.update(MarketStateEngine().classify(f))
            except Exception:
                pass
            ex = scanner.exit_v2.score(f, breadth=None)  # no full breadth yet
            f.update(ex)

            rn = rec_eng.for_new_buyer(f, regime)
            rh = rec_eng.for_holder(f, cost_basis=portfolio.get(sym), regime=regime) if is_pf else None
            print(_fmt_progress_line(i, n, sym, f, rn, rh, is_portfolio=is_pf))

            f.update(rn)
            f["symbol"] = sym
            rows.append(f)
            n_ok += 1

            # accumulate breadth stats from already-computed features
            try:
                if int(f.get("above_sma50") or 0) == 1:
                    above50 += 1
                # approximate MA20 via ema/close relation if present
                ema_pct = f.get("ema_pct")
                if ema_pct is not None and float(ema_pct) > 0:
                    above20 += 1
                r1 = f.get("ret_1d")
                if r1 is not None:
                    if float(r1) > 0: up += 1
                    elif float(r1) < 0: down += 1
            except Exception:
                pass

        except Exception as e:
            print(f"  [{i}/{n}] {sym:6s}  — ERROR {type(e).__name__}: {str(e)[:80]}")
            n_err += 1

        time.sleep(sleep_sec)

    # finalize breadth from scanned set
    if n_ok > 0:
        p50 = above50 / n_ok
        p20 = above20 / n_ok
        if p50 >= 0.65: bl = "HEALTHY"
        elif p50 >= 0.45: bl = "MIXED"
        elif p50 >= 0.30: bl = "WEAK"
        else: bl = "POOR"
        breadth = {
            "n_universe": n_ok,
            "pct_above_ma20": round(p20 * 100, 1),
            "pct_above_ma50": round(p50 * 100, 1),
            "pct_above_ma200": np.nan,
            "advance_decline": round(up / max(down, 1), 2),
            "breadth_label": bl,
        }
        regime["breadth"] = bl
        regime["breadth_detail"] = breadth

    # light sector table from successful rows only (no extra API)
    sec_tbl = pd.DataFrame()
    if rows:
        tmp = pd.DataFrame(rows)
        if "sector" in tmp.columns and "factor_trend" in tmp.columns:
            sec_tbl = (tmp.groupby("sector")
                       .agg(n=("symbol", "count"),
                            avg_trend=("factor_trend", "mean"),
                            avg_flow=("factor_flow", "mean") if "factor_flow" in tmp.columns else ("factor_trend", "mean"))
                       .reset_index())
            sec_tbl["sector_strength"] = sec_tbl["avg_trend"].clip(0, 100).round(1)

    print(f"\n  Done. ok={n_ok} skip={n_skip} err={n_err}")
    print(f"  Breadth (from scanned): {breadth.get('breadth_label')} | "
          f"MA50~{breadth.get('pct_above_ma50')}% | n={breadth.get('n_universe')}")

    if not rows:
        return {"regime": regime, "breadth": breadth, "sectors": sec_tbl,
                "new": pd.DataFrame(), "holder": pd.DataFrame(), "all": pd.DataFrame()}

    df = pd.DataFrame(rows)
    new_recs = []
    for _, r in df.iterrows():
        f = r.to_dict()
        rec = rec_eng.for_new_buyer(f, regime)
        rec["symbol"] = r["symbol"]
        rec["close"] = r["close"]
        rec["sector"] = r.get("sector")
        rec["sector_strength"] = r.get("sector_strength")
        lv = rec_eng.levels.build(f, mode="new")
        rec.update(lv)
        new_recs.append(rec)
    new_df = pd.DataFrame(new_recs)
    if len(new_df):
        new_df["alpha_pct"] = new_df["alpha_score"].rank(pct=True) * 100
    # buy_zone = new_df[new_df["action"].isin(["NEW_BUY", "WAIT_DIP"])].sort_values("alpha_score", ascending=False) if len(new_df) else new_df
    buy_zone = new_df.sort_values("alpha_score", ascending=False) if len(new_df) else new_df

    holder_rows = []
    for _, r in df.iterrows():
        sym = r["symbol"]
        if portfolio:
            if sym not in portfolio and float(r.get("exit_risk") or 0) < 40:
                continue
        else:
            if float(r.get("exit_risk") or 0) < 40:
                continue
        h = rec_eng.for_holder(r.to_dict(), cost_basis=portfolio.get(sym), regime=regime)
        h["symbol"] = sym
        h["close"] = r["close"]
        h["sector"] = r.get("sector")
        holder_rows.append(h)
    holder_df = pd.DataFrame(holder_rows)
    if len(holder_df):
        holder_df = holder_df.sort_values("exit_risk", ascending=False)

    print("\n" + "═" * 60)
    print("  MARKET STATUS")
    print("═" * 60)
    print(f"  Regime  : {regime.get('label')}")
    print(f"  Breadth : {breadth.get('breadth_label')} "
          f"(proxy MA20 {breadth.get('pct_above_ma20')}% | MA50 {breadth.get('pct_above_ma50')}%)")

    print("\n  TOP NEW_BUY / WAIT_DIP")
    cols = [c for c in ["symbol", "sector", "close", "action", "setup", "alpha_score",
                        "entry_suggest", "sl", "tp1", "tp2", "rr_tp1", "exit_risk", "reasons"]
            if c in buy_zone.columns]
    if len(buy_zone):
        print(buy_zone[cols].to_string(index=False))
    else:
        print("  (none)")

    print("\n  PORTFOLIO / EXIT MONITOR")
    if len(holder_df):
        hcols = [c for c in ["symbol", "close", "action", "exit_risk", "exit_band",
                             "trail_sl", "tp1_hold", "unrealized_pnl_pct", "reasons"]
                 if c in holder_df.columns]
        print(holder_df[hcols].to_string(index=False))
    else:
        print("  (none)")

    return {
        "regime": regime,
        "breadth": breadth,
        "sectors": sec_tbl,
        "new": buy_zone,
        "holder": holder_df,
        "all": df,
        "all_new": new_df,
    }


print("scan_vn100_verbose() READY (fixed — prints each symbol immediately)")


scan_vn100_verbose() READY (fixed — prints each symbol immediately)


---

## ★ Full VN100 — chạy 1 cell (có log từng mã)

1. **Run All** các cell phía trên (load engine)
2. Chạy cell **ONE-CLICK** bên dưới

Trong lúc chạy sẽ thấy dạng:
```
  [12/243] FPT     ✅  78.2  NEW_BUY           px:95.50  PULLBACK      🟢 Up  💰 Flow+  VSA:CẠN VOL  TP1:+6.2%  XR:12  Technology
  [13/243] TCB [PF] ·  55.1  HOLD              px:32.40  NONE          🟡 Side  🔄 Flow~  VSA:—  XR:48  Bank
```

`[PF]` = mã trong portfolio của bạn.


In [71]:
pd.set_option("display.max_rows", 300)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

## 🚀 ENHANCEMENT — RSI Divergence + Ichimoku Trịnh Phát / Dao Găm + Top-5 Backtest

Batch này mở rộng **Trading Decision World** theo một hypothesis mới:

1. **RSI divergence**
   - Phân kỳ dương thường: Price Lower Low + RSI Higher Low.
   - Phân kỳ âm thường: Price Higher High + RSI Lower High.
   - Đồng thời giữ hidden bullish / hidden bearish; hidden bearish bám ý tưởng Pine Script user đưa: *Price Lower High + RSI Higher High*.
   - Pivot chỉ được dùng sau khi đủ `right bars` xác nhận → tránh look-ahead.

2. **Ichimoku Trịnh Phát / Dao Găm**
   - Dao Găm 65 và Dao Găm 129 được tính theo midpoint `(Highest High + Lowest Low) / 2`.
   - Theo tài liệu công khai về Ichimoku Trịnh Phát, hệ thống dùng bộ 9–17–26 và Dao Găm 65/129. citeturn0search5turn0search2
   - Scanner báo trạng thái Dao Găm ở **Daily / Weekly / Monthly**.
   - Nếu giá đang gần/chạm Dao Găm, output sẽ ghi rõ timeframe và level.

3. **Strategy backtest mới**
   - Entry: bullish RSI divergence còn hiệu lực + giá chạm/gần Dao Găm.
   - Exit: bearish RSI divergence, mất Dao Găm, hoặc ATR stop / take-profit / max holding.
   - Hỗ trợ DG ngày/tuần/tháng.
   - Entry mặc định tại **T+1 Open** để thống nhất với research engine hiện tại.

4. **Top-5 Trading Decision World**
   - Sau khi scan, lấy 5 mã đầu bảng theo `alpha_score`.
   - Backtest riêng chiến thuật RSI + Dao Găm cho 5 mã đó.
   - Kết quả trả về trong `SCAN_OUT["top5_backtest"]`.

> Đây là hypothesis để kiểm chứng bằng dữ liệu lịch sử, không mặc định rằng RSI divergence hay Dao Găm có alpha. Top-5 hiện tại cũng là một conditional test: các mã được chọn bằng dữ liệu hiện tại rồi đem kiểm tra lịch sử, nên không được coi là một backtest OOS của toàn universe.


In [72]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ENHANCEMENT 1 — RSI DIVERGENCE ENGINE                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Ý tưởng lấy từ Pine Script user cung cấp:
#   Hidden Bearish = Price Lower High + RSI Higher High tại các pivot.
#
# Quy tắc chống look-ahead:
#   Pivot tại i chỉ được xem là "confirmed" tại i + right_bars.
#   Vì vậy backtest chỉ phát tín hiệu từ ngày xác nhận trở đi.
#
# Các loại divergence:
#   REG_BULL = Price Lower Low  + RSI Higher Low
#   REG_BEAR = Price Higher High + RSI Lower High
#   HID_BULL = Price Higher Low + RSI Lower Low
#   HID_BEAR = Price Lower High + RSI Higher High
#
# "positive/negative RSI divergence" trong Trading Decision World sẽ ưu tiên
# REG_BULL / REG_BEAR; hidden divergence được đưa thêm vào cột riêng.

def _rsi_series_wilder_like(close: pd.Series, period: int = 14) -> pd.Series:
    """RSI dùng cùng tinh thần với feature engine hiện tại, nhưng trả về full series."""
    c = pd.to_numeric(close, errors="coerce").astype(float)
    delta = c.diff()
    gain = delta.clip(lower=0)
    loss = (-delta.clip(upper=0))
    avg_gain = gain.rolling(period, min_periods=period).mean()
    avg_loss = loss.rolling(period, min_periods=period).mean()
    rs = avg_gain / avg_loss.replace(0, np.nan)
    rsi = 100 - 100 / (1 + rs)
    # Nếu loss=0, RSI có ý nghĩa là 100 thay vì NaN.
    rsi = rsi.where(~((avg_loss == 0) & (avg_gain > 0)), 100.0)
    return rsi


def _pivot_low_confirmed(series: pd.Series, left: int, right: int) -> list:
    """Trả về (pivot_idx, confirmation_idx) cho pivot low."""
    x = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    out = []
    for i in range(left, len(x) - right):
        w = x[i-left:i+right+1]
        if np.isfinite(x[i]) and np.isfinite(w).all() and x[i] <= np.min(w):
            # Tie-break: pivot phải thấp hơn ít nhất một điểm lân cận.
            if x[i] < np.min(np.r_[x[i-left:i], x[i+1:i+right+1]]):
                out.append((i, i + right))
    return out


def _pivot_high_confirmed(series: pd.Series, left: int, right: int) -> list:
    """Trả về (pivot_idx, confirmation_idx) cho pivot high."""
    x = pd.to_numeric(series, errors="coerce").to_numpy(dtype=float)
    out = []
    for i in range(left, len(x) - right):
        w = x[i-left:i+right+1]
        if np.isfinite(x[i]) and np.isfinite(w).all() and x[i] >= np.max(w):
            if x[i] > np.max(np.r_[x[i-left:i], x[i+1:i+right+1]]):
                out.append((i, i + right))
    return out


def compute_rsi_divergence_series(df: pd.DataFrame, cfg: ResearchConfig = CFG) -> pd.DataFrame:
    """
    Compute divergence events for the entire history once.

    This is the efficient version used by backtest. Pivot confirmation is
    still strictly causal: event_date = pivot2 + right_bars.
    """
    d = df.sort_values("time").reset_index(drop=True).copy()
    rsi = _rsi_series_wilder_like(d["close"], cfg.rsi_period)

    out = pd.DataFrame(index=d.index)
    out["rsi"] = rsi
    out["rsi_divergence"] = "NONE"
    out["rsi_div_bull"] = False
    out["rsi_div_bear"] = False
    out["rsi_hidden_bull"] = False
    out["rsi_hidden_bear"] = False
    out["rsi_div_age"] = np.nan
    out["rsi_div_strength"] = np.nan
    out["rsi_div_signal_date"] = pd.NaT

    events = []

    low = pd.to_numeric(d["low"], errors="coerce")
    high = pd.to_numeric(d["high"], errors="coerce")

    pl = _pivot_low_confirmed(low, cfg.rsi_div_left, cfg.rsi_div_right)
    for (i1, c1), (i2, c2) in zip(pl[:-1], pl[1:]):
        if i2 - i1 > cfg.rsi_div_max_bars:
            continue
        r1, r2 = rsi.iloc[i1], rsi.iloc[i2]
        if pd.isna(r1) or pd.isna(r2):
            continue
        p1, p2 = low.iloc[i1], low.iloc[i2]
        if p2 < p1 and r2 > r1 + cfg.rsi_div_min_rsi:
            events.append(("REG_BULL", c2, float(abs(r2-r1)/max(abs(r1),1))))
        if p2 > p1 and r2 < r1 - cfg.rsi_div_min_rsi:
            events.append(("HID_BULL", c2, float(abs(r2-r1)/max(abs(r1),1))))

    ph = _pivot_high_confirmed(high, cfg.rsi_div_left, cfg.rsi_div_right)
    for (i1, c1), (i2, c2) in zip(ph[:-1], ph[1:]):
        if i2 - i1 > cfg.rsi_div_max_bars:
            continue
        r1, r2 = rsi.iloc[i1], rsi.iloc[i2]
        if pd.isna(r1) or pd.isna(r2):
            continue
        p1, p2 = high.iloc[i1], high.iloc[i2]
        if p2 > p1 and r2 < r1 - cfg.rsi_div_min_rsi:
            events.append(("REG_BEAR", c2, float(abs(r2-r1)/max(abs(r1),1))))
        if p2 < p1 and r2 > r1 + cfg.rsi_div_min_rsi:
            events.append(("HID_BEAR", c2, float(abs(r2-r1)/max(abs(r1),1))))

    # Keep the latest confirmed event active for N bars.
    events.sort(key=lambda x: x[1])
    for typ, conf_idx, strength in events:
        end_idx = min(len(d)-1, conf_idx + cfg.rsi_div_active_bars)
        # Latest event overwrites an older overlapping event, which is the
        # same "latest confirmed signal wins" convention used by the live wrapper.
        idx = np.arange(conf_idx, end_idx + 1)
        out.loc[idx, "rsi_divergence"] = typ
        out.loc[idx, "rsi_div_bull"] = typ in ("REG_BULL","HID_BULL")
        out.loc[idx, "rsi_div_bear"] = typ in ("REG_BEAR","HID_BEAR")
        out.loc[idx, "rsi_hidden_bull"] = typ == "HID_BULL"
        out.loc[idx, "rsi_hidden_bear"] = typ == "HID_BEAR"
        out.loc[idx, "rsi_div_age"] = np.arange(len(idx))
        out.loc[idx, "rsi_div_strength"] = strength
        out.loc[idx, "rsi_div_signal_date"] = d["time"].iloc[conf_idx]

    return out


def compute_rsi_divergence(df: pd.DataFrame, cfg: ResearchConfig = CFG) -> dict:
    """Live wrapper: return only the last causal divergence state."""
    if df is None or len(df) == 0:
        return {
            "rsi": np.nan, "rsi_divergence":"NONE",
            "rsi_div_bull":False, "rsi_div_bear":False,
            "rsi_hidden_bull":False, "rsi_hidden_bear":False,
            "rsi_div_age":np.nan, "rsi_div_strength":np.nan,
            "rsi_div_signal_date":pd.NaT,
        }
    s = compute_rsi_divergence_series(df, cfg)
    return s.iloc[-1].to_dict()




print("RSI divergence engine ready — regular + hidden, pivot-confirmed / no look-ahead.")


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ENHANCEMENT 2 — ICHIMOKU TRỊNH PHÁT / DAO GĂM ENGINE                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Public descriptions of Ichimoku Trịnh Phát describe:
#   Tenkan = midpoint(HH/LL, 9)
#   Kijun  = midpoint(HH/LL, 17)
#   Kumo   from the 9/17/26 setup
#   Dao Găm 65 and Dao Găm 129 = midpoint(HH/LL, 65/129)
#
# Here we implement the observable numerical levels only.
# We do NOT invent proprietary labels/rules that are not documented.

def _midpoint_hl(high: pd.Series, low: pd.Series, n: int) -> pd.Series:
    hh = high.rolling(n, min_periods=n).max()
    ll = low.rolling(n, min_periods=n).min()
    return (hh + ll) / 2.0


def compute_ichimoku_trinh_phat(df: pd.DataFrame) -> dict:
    """Daily Ichimoku TP levels on the last available bar."""
    if df is None or len(df) < 30:
        return {}
    d = df.sort_values("time").reset_index(drop=True)
    h, l, c = d["high"].astype(float), d["low"].astype(float), d["close"].astype(float)
    tenkan = _midpoint_hl(h, l, 9)
    kijun17 = _midpoint_hl(h, l, 17)
    dg65 = _midpoint_hl(h, l, 65)
    dg129 = _midpoint_hl(h, l, 129)
    return {
        "ichi_tenkan_9": float(tenkan.iloc[-1]) if pd.notna(tenkan.iloc[-1]) else np.nan,
        "ichi_kijun_17": float(kijun17.iloc[-1]) if pd.notna(kijun17.iloc[-1]) else np.nan,
        "dg65": float(dg65.iloc[-1]) if pd.notna(dg65.iloc[-1]) else np.nan,
        "dg129": float(dg129.iloc[-1]) if pd.notna(dg129.iloc[-1]) else np.nan,
        "close": float(c.iloc[-1]),
    }


def _period_dg_at_daily_dates(df_daily: pd.DataFrame, freq: str, periods=(65,129)) -> pd.DataFrame:
    """
    Vectorized current-period DG for every daily date.

    For each daily date:
      - current week/month high/low = information seen up to that date;
      - previous periods = completed period bars only.

    This implementation is O(N) rather than recalculating every rolling
    window inside every daily row. That matters for 10+ years of data.
    """
    d = df_daily.sort_values("time").reset_index(drop=True).copy()
    if d.empty:
        return pd.DataFrame(index=d.index)

    period = d["time"].dt.to_period(freq)
    g = d.groupby(period, sort=True)

    # Current-period running high/low, using only data <= each daily row.
    cur_high = g["high"].cummax().to_numpy(dtype=float)
    cur_low = g["low"].cummin().to_numpy(dtype=float)

    # One row per completed/current period.
    agg = g.agg(high=("high", "max"), low=("low", "min"))
    # Position of each daily row in the period table.
    pos = period.map({p:i for i, p in enumerate(agg.index)}).to_numpy(dtype=int)

    out = pd.DataFrame(index=d.index)
    for n in periods:
        # Previous n-1 completed periods only.
        prior_high = agg["high"].shift(1).rolling(
            n-1, min_periods=n-1
        ).max().to_numpy(dtype=float)
        prior_low = agg["low"].shift(1).rolling(
            n-1, min_periods=n-1
        ).min().to_numpy(dtype=float)

        ph = prior_high[pos]
        pl = prior_low[pos]

        hh = np.maximum(cur_high, ph)
        ll = np.minimum(cur_low, pl)
        valid = np.isfinite(hh) & np.isfinite(ll)
        vals = np.full(len(d), np.nan, dtype=float)
        vals[valid] = (hh[valid] + ll[valid]) / 2.0
        out[f"dg{n}"] = vals

    return out



def compute_dg_multi_timeframe(df_daily: pd.DataFrame, cfg: ResearchConfig = CFG) -> dict:
    """
    Return Daily / Weekly / Monthly Dao Găm values for the LAST daily bar.

    Monthly DG129 needs ~129 months of history. If history is insufficient,
    the function returns NaN and an explicit status instead of fabricating it.
    """
    if df_daily is None or len(df_daily) < 65:
        return {}

    d = df_daily.sort_values("time").reset_index(drop=True)
    close = float(d["close"].iloc[-1])
    out = {"close": close}

    # Daily DG: straightforward rolling.
    dg_d = {
        65: _midpoint_hl(d["high"].astype(float), d["low"].astype(float), 65).iloc[-1],
        129: _midpoint_hl(d["high"].astype(float), d["low"].astype(float), 129).iloc[-1],
    }

    # Higher timeframes: current partial period is allowed, but only data <= T.
    dg_w = _period_dg_at_daily_dates(d, "W-FRI", periods=(65,129))
    dg_m = _period_dg_at_daily_dates(d, "M", periods=(65,129))

    for tf, vals in [("D", dg_d),
                     ("W", {65: dg_w["dg65"].iloc[-1], 129: dg_w["dg129"].iloc[-1]}),
                     ("M", {65: dg_m["dg65"].iloc[-1], 129: dg_m["dg129"].iloc[-1]})]:
        for n, value in vals.items():
            out[f"DG_{tf}_{n}"] = float(value) if pd.notna(value) else np.nan

    # Status is based on the current close and the nearest valid DG line.
    for tf in ("D","W","M"):
        vals = [out.get(f"DG_{tf}_65"), out.get(f"DG_{tf}_129")]
        vals = [float(v) for v in vals if v is not None and pd.notna(v) and float(v) > 0]
        if not vals:
            out[f"DG_{tf}_status"] = "INSUFFICIENT_HISTORY"
            out[f"DG_{tf}_near"] = False
            out[f"DG_{tf}_distance_pct"] = np.nan
            continue
        distances = [abs(close-v)/v for v in vals]
        nearest = min(distances)
        out[f"DG_{tf}_distance_pct"] = round(nearest*100, 3)
        out[f"DG_{tf}_near"] = bool(nearest <= cfg.dg_near_pct)
        if nearest <= cfg.dg_near_pct:
            out[f"DG_{tf}_status"] = "AT_DG"
        elif close > max(vals):
            out[f"DG_{tf}_status"] = "ABOVE_DG"
        elif close < min(vals):
            out[f"DG_{tf}_status"] = "BELOW_DG"
        else:
            out[f"DG_{tf}_status"] = "BETWEEN_DG"
    return out


def _dg_signal_summary(dg: dict, cfg: ResearchConfig = CFG) -> str:
    """Human-readable summary for the scanner table."""
    hits = []
    for tf, label in (("D","DG_D"),("W","DG_W"),("M","DG_M")):
        if dg.get(f"DG_{tf}_near"):
            hits.append(label)
    return " + ".join(hits) if hits else "NONE"


print("Ichimoku Trịnh Phát / Dao Găm engine ready — DG65/DG129 on D/W/M.")


# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ENHANCEMENT 3 — COMBINED RSI + DG FEATURE BLOCK                       ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def compute_rsi_dg_features(df: pd.DataFrame, cfg: ResearchConfig = CFG) -> dict:
    """
    Single entry point used by scanner and by the dedicated strategy.

    All values are calculated from df ending at T.
    """
    out = {}
    out.update(compute_rsi_divergence(df, cfg))
    dg = compute_dg_multi_timeframe(df, cfg)
    out.update(dg)

    # Compact trading context.
    div = out.get("rsi_divergence", "NONE")
    dg_summary = _dg_signal_summary(out, cfg)
    if div in ("REG_BULL","HID_BULL") and dg_summary != "NONE":
        ctx = "BULL_DIV_AT_DG"
    elif div in ("REG_BEAR","HID_BEAR") and dg_summary != "NONE":
        ctx = "BEAR_DIV_AT_DG"
    elif div in ("REG_BULL","HID_BULL"):
        ctx = "BULL_DIV"
    elif div in ("REG_BEAR","HID_BEAR"):
        ctx = "BEAR_DIV"
    elif dg_summary != "NONE":
        ctx = "AT_DG"
    else:
        ctx = "NEUTRAL"
    out["dg_signal"] = dg_summary
    out["rsi_dg_context"] = ctx
    return out


print("Combined RSI + DG feature block ready.")


RSI divergence engine ready — regular + hidden, pivot-confirmed / no look-ahead.
Ichimoku Trịnh Phát / Dao Găm engine ready — DG65/DG129 on D/W/M.
Combined RSI + DG feature block ready.


In [73]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ENHANCEMENT 4 — RSI DIVERGENCE + DAO GĂM BACKTEST                     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# Strategy hypothesis:
#   BUY:
#       confirmed bullish RSI divergence is active
#       AND price is near/touching at least one DG (D/W/M).
#
#   SELL:
#       confirmed bearish RSI divergence is active
#       OR price closes below the DG level used for entry by a configurable
#          buffer.
#
# Risk exits:
#       ATR stop, ATR take-profit, or maximum holding sessions.
#
# Execution:
#       signal at T close -> next available session open.
#
# This is deliberately a separate strategy engine rather than modifying
# BacktestEngineV3, because the old engine is label-driven and this new
# strategy must make decisions from raw historical OHLCV at each T.

@dataclass
class RSIDGBacktestConfig:
    initial_capital: float = 100_000_000
    risk_per_trade: float = 0.005
    max_position_pct: float = 0.20
    max_open_positions: int = 1
    fee_buy: float = 0.0015
    fee_sell: float = 0.0015
    tax_sell: float = 0.001
    slippage_bps: float = 8.0

    # Signal parameters.
    near_dg_pct: float = 0.015
    divergence_active_bars: int = 8
    require_regular_divergence: bool = True

    # Risk / exit parameters.
    atr_period: int = 14
    stop_atr: float = 1.5
    target_atr: float = 3.0
    max_hold_sessions: int = 30
    dg_break_pct: float = 0.01
    bear_div_requires_dg: bool = True

    # Which DG timeframes can trigger an entry.
    enabled_timeframes: tuple = ("D","W","M")


def _first_next_session_index(df: pd.DataFrame, i: int) -> int:
    return i + 1 if i + 1 < len(df) else -1


def _atr_series(df: pd.DataFrame, period: int = 14) -> pd.Series:
    h = df["high"].astype(float)
    l = df["low"].astype(float)
    c = df["close"].astype(float)
    tr = pd.concat([h-l, (h-c.shift()).abs(), (l-c.shift()).abs()], axis=1).max(axis=1)
    return tr.rolling(period, min_periods=period).mean()


def _nearest_dg_level(row: dict, enabled=("D","W","M"), near_pct=0.015):
    """Return nearest DG level among all valid 65/129 lines."""
    c = float(row.get("close") or np.nan)
    candidates = []
    for tf in enabled:
        for n in (65,129):
            v = row.get(f"DG_{tf}_{n}")
            if v is None or pd.isna(v) or float(v) <= 0:
                continue
            v = float(v)
            dist = abs(c-v)/v
            candidates.append((dist, tf, n, v))
    if not candidates:
        return None
    candidates.sort(key=lambda x: x[0])
    best = candidates[0]
    return {
        "distance_pct": best[0],
        "tf": best[1],
        "period": best[2],
        "level": best[3],
        "near": best[0] <= near_pct,
    }


def _enrich_daily_history_with_rsi_dg(df: pd.DataFrame, cfg=CFG) -> pd.DataFrame:
    """
    Build the entire daily signal table once.

    RSI divergence is calculated once for the full history.
    Daily/weekly/monthly DG are calculated once for the full history.
    No future data is used by the signal columns at any row.
    """
    d = df.sort_values("time").reset_index(drop=True).copy()
    d["atr_abs"] = _atr_series(d, 14)

    # Causal RSI divergence series.
    div = compute_rsi_divergence_series(d, cfg)

    # Daily/weekly/monthly DG series.
    dg_d = pd.DataFrame({
        "DG_D_65": _midpoint_hl(d["high"].astype(float), d["low"].astype(float), 65),
        "DG_D_129": _midpoint_hl(d["high"].astype(float), d["low"].astype(float), 129),
    })
    dg_w = _period_dg_at_daily_dates(d, "W-FRI", periods=(65,129))
    dg_m = _period_dg_at_daily_dates(d, "M", periods=(65,129))

    out = d[["time","open","high","low","close","volume","atr_abs"]].copy()
    for block in [div, dg_d, dg_w.rename(columns={"dg65":"DG_W_65","dg129":"DG_W_129"}),
                  dg_m.rename(columns={"dg65":"DG_M_65","dg129":"DG_M_129"})]:
        for c in block.columns:
            if c not in out.columns:
                out[c] = block[c].values

    # Timeframe statuses and nearest DG summary.
    for tf in ("D","W","M"):
        vals65 = pd.to_numeric(out[f"DG_{tf}_65"], errors="coerce")
        vals129 = pd.to_numeric(out[f"DG_{tf}_129"], errors="coerce")
        valid = pd.concat([vals65, vals129], axis=1)
        distance = pd.concat([
            (out["close"]-vals65).abs()/vals65,
            (out["close"]-vals129).abs()/vals129
        ], axis=1).min(axis=1)
        out[f"DG_{tf}_distance_pct"] = distance * 100
        out[f"DG_{tf}_near"] = distance <= cfg.dg_near_pct
        out[f"DG_{tf}_status"] = np.select(
            [
                distance.isna(),
                distance <= cfg.dg_near_pct,
                out["close"] > pd.concat([vals65,vals129],axis=1).max(axis=1),
                out["close"] < pd.concat([vals65,vals129],axis=1).min(axis=1),
            ],
            ["INSUFFICIENT_HISTORY","AT_DG","ABOVE_DG","BELOW_DG"],
            default="BETWEEN_DG",
        )

    # Compact context.
    out["dg_signal"] = out.apply(
        lambda r: " + ".join([
            f"DG_{tf}" for tf in ("D","W","M") if bool(r.get(f"DG_{tf}_near",False))
        ]) or "NONE", axis=1
    )
    def ctx(row):
        divv = str(row.get("rsi_divergence") or "NONE")
        dg = str(row.get("dg_signal") or "NONE")
        if divv in ("REG_BULL","HID_BULL") and dg != "NONE": return "BULL_DIV_AT_DG"
        if divv in ("REG_BEAR","HID_BEAR") and dg != "NONE": return "BEAR_DIV_AT_DG"
        if divv in ("REG_BULL","HID_BULL"): return "BULL_DIV"
        if divv in ("REG_BEAR","HID_BEAR"): return "BEAR_DIV"
        if dg != "NONE": return "AT_DG"
        return "NEUTRAL"
    out["rsi_dg_context"] = out.apply(ctx, axis=1)
    return out




def backtest_rsi_dg_symbol(
    symbol: str,
    df: pd.DataFrame,
    bt_cfg: RSIDGBacktestConfig = None,
) -> dict:
    """
    Dedicated event-driven backtest for one symbol.

    A trade is opened only when:
      REG_BULL/HID_BULL + near DG (D/W/M).

    For the conservative/default mode, regular bullish divergence is required.
    Hidden bullish can be enabled by setting require_regular_divergence=False.
    """
    bt = bt_cfg or RSIDGBacktestConfig()
    if df is None or len(df) < 250:
        return {"symbol": symbol, "trades": pd.DataFrame(), "metrics": {"n_trades": 0}}

    sig = _enrich_daily_history_with_rsi_dg(df)
    if sig.empty:
        return {"symbol": symbol, "trades": pd.DataFrame(), "metrics": {"n_trades": 0}}

    capital = bt.initial_capital
    trades = []
    i = 0

    while i < len(sig) - 1:
        row = sig.iloc[i].to_dict()

        div = str(row.get("rsi_divergence") or "NONE")
        bullish_ok = div == "REG_BULL" if bt.require_regular_divergence else div in ("REG_BULL","HID_BULL")
        dg = _nearest_dg_level(row, bt.enabled_timeframes, bt.near_dg_pct)

        if not bullish_ok or dg is None or not dg["near"]:
            i += 1
            continue

        entry_i = i + 1
        entry_row = sig.iloc[entry_i]
        entry_raw = float(entry_row["open"])
        if entry_raw <= 0:
            i += 1
            continue

        atr = float(row.get("atr_abs") or np.nan)
        if not np.isfinite(atr) or atr <= 0:
            i += 1
            continue

        # Position sizing by risk budget, capped by max position.
        stop_raw = max(entry_raw - bt.stop_atr * atr, entry_raw * 0.90)
        target_raw = entry_raw + bt.target_atr * atr
        risk_per_share = max(entry_raw - stop_raw, entry_raw * 0.005)
        risk_cash = capital * bt.risk_per_trade
        shares = int(risk_cash / risk_per_share / 100) * 100
        max_shares = int((capital * bt.max_position_pct) / entry_raw / 100) * 100
        shares = min(shares, max_shares)
        if shares <= 0:
            i += 1
            continue

        slip = bt.slippage_bps / 10000.0
        entry_px = entry_raw * (1 + slip)
        dg_level = float(dg["level"])

        exit_reason = "TIMEOUT"
        exit_i = min(len(sig)-1, entry_i + bt.max_hold_sessions)
        exit_raw = float(sig.iloc[exit_i]["close"])

        # Path-wise simulation. If TP and SL happen on same candle, choose SL first
        # to stay conservative, matching the notebook's risk-first convention.
        for j in range(entry_i, min(len(sig), entry_i + bt.max_hold_sessions + 1)):
            rr = sig.iloc[j]
            hi, lo, cl = float(rr["high"]), float(rr["low"]), float(rr["close"])

            # First: hard stop / target.
            if lo <= stop_raw:
                exit_raw = stop_raw
                exit_i = j
                exit_reason = "ATR_SL"
                break
            if hi >= target_raw:
                exit_raw = target_raw
                exit_i = j
                exit_reason = "ATR_TP"
                break

            # Sell when bearish divergence is confirmed.
            # Default requires the bearish divergence to occur near one of
            # the D/W/M Dao Găm levels, keeping the strategy symmetric with
            # the entry hypothesis.
            bear_div = str(rr.get("rsi_divergence") or "NONE") in ("REG_BEAR","HID_BEAR")
            rr_dg = _nearest_dg_level(rr.to_dict(), bt.enabled_timeframes, bt.near_dg_pct)
            if bear_div and ((not bt.bear_div_requires_dg) or (rr_dg is not None and rr_dg["near"])):
                exit_raw = cl
                exit_i = j
                exit_reason = "RSI_BEAR_DIV_AT_DG"
                break

            # Sell when price loses the DG level that triggered the entry.
            if cl < dg_level * (1 - bt.dg_break_pct):
                exit_raw = cl
                exit_i = j
                exit_reason = "DG_BREAK"
                break

        exit_px = exit_raw * (1 - slip)
        notional_in = entry_px * shares
        notional_out = exit_px * shares
        costs = (
            notional_in * bt.fee_buy
            + notional_out * bt.fee_sell
            + notional_out * bt.tax_sell
            + notional_in * 2 * slip
        )
        pnl = (exit_px - entry_px) * shares - costs
        ret = pnl / notional_in if notional_in > 0 else 0.0

        trades.append({
            "symbol": symbol,
            "signal_date": row["time"],
            "entry_date": entry_row["time"],
            "exit_date": sig.iloc[exit_i]["time"],
            "entry_px": entry_px,
            "exit_px": exit_px,
            "shares": shares,
            "notional": notional_in,
            "ret_net": ret,
            "pnl_net": pnl,
            "cost": costs,
            "sessions": exit_i - entry_i + 1,
            "reason": exit_reason,
            "rsi_divergence": div,
            "rsi_div_age": row.get("rsi_div_age"),
            "dg_tf": dg["tf"],
            "dg_period": dg["period"],
            "dg_level": dg_level,
            "dg_distance_pct": dg["distance_pct"],
            "entry_dg_status": row.get(f"DG_{dg['tf']}_status"),
            "rsi_at_signal": row.get("rsi"),
        })

        capital += pnl
        # One position per symbol. Move to the bar after exit.
        i = exit_i + 1

    trades_df = pd.DataFrame(trades)
    metrics = compute_performance_metrics(trades_df, initial_capital=bt.initial_capital) if len(trades_df) else {"n_trades": 0}
    metrics["symbol"] = symbol
    return {"symbol": symbol, "trades": trades_df, "metrics": metrics}


def backtest_top5_trading_decision_world(
    scan_out: dict,
    store: MarketDataStore = None,
    top_n: int = 5,
    start: str = "2012-01-01",
    end: str = None,
    bt_cfg: RSIDGBacktestConfig = None,
) -> dict:
    """
    Lấy Top-N từ Trading Decision World rồi backtest chiến thuật RSI + DG.

    TOP-N được lấy từ all_new/new theo alpha_score hiện tại.
    Đây là conditional historical test, KHÔNG phải OOS selection test.
    """
    if scan_out is None:
        return {"top_symbols": [], "summary": pd.DataFrame(), "trades": pd.DataFrame()}

    world = scan_out.get("all_new")
    if world is None or len(world) == 0:
        world = scan_out.get("new")
    if world is None or len(world) == 0:
        return {"top_symbols": [], "summary": pd.DataFrame(), "trades": pd.DataFrame()}

    top = world.copy()
    top["alpha_score"] = pd.to_numeric(top["alpha_score"], errors="coerce")
    top = top.sort_values("alpha_score", ascending=False).drop_duplicates("symbol").head(top_n)
    symbols = top["symbol"].astype(str).tolist()
    print("\n" + "═"*70)
    print(f"  TOP-{len(symbols)} TRADING DECISION WORLD → RSI + DAO GĂM BACKTEST")
    print("═"*70)
    print("  Symbols:", ", ".join(symbols))

    store = store or MarketDataStore(CFG)
    end = end or datetime.today().strftime("%Y-%m-%d")
    all_trades = []
    summaries = []

    for rank, sym in enumerate(symbols, 1):
        print(f"\n  [{rank}/{len(symbols)}] Backtest {sym}...")
        try:
            df = store.get(sym, start, end, use_cache=True, force_refresh=False)
            # Monthly DG129 requires long history. If the cache is too short,
            # force one refresh so the result is explicit rather than silently
            # falling back to a short sample.
            if df is None or len(df) < 2500:
                df = store.get(sym, start, end, use_cache=False, force_refresh=True)
            result = backtest_rsi_dg_symbol(sym, df, bt_cfg=bt_cfg)
            m = dict(result["metrics"])
            m["rank_current_alpha"] = rank
            m["current_alpha_score"] = float(top.loc[top["symbol"].eq(sym), "alpha_score"].iloc[0])
            m["current_action"] = top.loc[top["symbol"].eq(sym), "action"].iloc[0] if "action" in top.columns else ""
            m["current_rsi_divergence"] = top.loc[top["symbol"].eq(sym), "rsi_divergence"].iloc[0] if "rsi_divergence" in top.columns else ""
            m["current_dg_signal"] = top.loc[top["symbol"].eq(sym), "dg_signal"].iloc[0] if "dg_signal" in top.columns else ""
            summaries.append(m)
            if len(result["trades"]):
                all_trades.append(result["trades"])
            print(f"    trades={m.get('n_trades',0)} | win={m.get('win_rate_pct','—')}% | "
                  f"PF={m.get('profit_factor','—')} | total={m.get('total_return','—')}")
        except Exception as e:
            print(f"    ERROR {type(e).__name__}: {str(e)[:120]}")
            summaries.append({
                "symbol": sym, "rank_current_alpha": rank,
                "current_alpha_score": float(top.loc[top["symbol"].eq(sym), "alpha_score"].iloc[0]),
                "error": f"{type(e).__name__}: {e}",
            })

    summary_df = pd.DataFrame(summaries)
    trades_df = pd.concat(all_trades, ignore_index=True) if all_trades else pd.DataFrame()

    ts = datetime.now().strftime("%Y%m%d_%H%M")
    summary_df.to_csv(RESULTS_DIR / f"top5_rsi_dg_summary_{ts}.csv", index=False)
    if len(trades_df):
        trades_df.to_parquet(RESULTS_DIR / f"top5_rsi_dg_trades_{ts}.parquet", index=False)

    print(f"\n✓ Top-{len(symbols)} summary → {RESULTS_DIR / f'top5_rsi_dg_summary_{ts}.csv'}")
    return {"top_symbols": symbols, "summary": summary_df, "trades": trades_df}


print("RSI + Dao Găm backtest engine ready.")


RSI + Dao Găm backtest engine ready.


In [74]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ENHANCEMENT 5 — INTEGRATE INTO TRADING DECISION WORLD / OUTPUT        ║
# ╚══════════════════════════════════════════════════════════════════════════╝
#
# We wrap the existing verbose scanner instead of rewriting its 100+ lines.
# This keeps the original scanner behavior intact and adds a post-processing
# layer for RSI divergence + DG multi-timeframe context.

_original_scan_vn100_verbose = scan_vn100_verbose


def _get_long_ichimoku_features(symbol: str, store: MarketDataStore = None) -> dict:
    """Fetch enough history for daily/weekly/monthly DG calculations."""
    store = store or MarketDataStore(CFG)
    end = datetime.today().strftime("%Y-%m-%d")
    start = (pd.Timestamp(end) - pd.Timedelta(days=CFG.dg_history_days)).strftime("%Y-%m-%d")

    df = store.get(symbol, start, end, use_cache=True, force_refresh=False)
    # If existing cache is too short for monthly DG129, refresh explicitly.
    if df is None or len(df) < 2500:
        df = store.get(symbol, start, end, use_cache=False, force_refresh=True)

    if df is None or len(df) < 150:
        return {}

    return compute_rsi_dg_features(df, CFG)


def _attach_rsi_dg_to_scan(scan_out: dict, store: MarketDataStore = None) -> dict:
    """Attach the new feature block to all scanner tables."""
    if not scan_out:
        return scan_out

    store = store or MarketDataStore(CFG)
    all_df = scan_out.get("all")
    if all_df is None or len(all_df) == 0:
        return scan_out

    feature_rows = []
    symbols = list(dict.fromkeys(all_df["symbol"].astype(str).tolist()))
    for i, sym in enumerate(symbols, 1):
        try:
            f = _get_long_ichimoku_features(sym, store)
            if not f:
                continue
            f["symbol"] = sym
            feature_rows.append(f)
        except Exception as e:
            print(f"  ⚠ RSI/DG {sym}: {type(e).__name__}: {str(e)[:80]}")

    if not feature_rows:
        return scan_out

    feat_df = pd.DataFrame(feature_rows)
    feat_map = feat_df.set_index("symbol").to_dict("index")

    def merge_features(df):
        if df is None or len(df) == 0:
            return df
        out = df.copy()
        for k in [
            "rsi","rsi_divergence","rsi_div_bull","rsi_div_bear",
            "rsi_hidden_bull","rsi_hidden_bear","rsi_div_age","rsi_div_strength",
            "rsi_div_signal_date","dg_signal","rsi_dg_context",
            "DG_D_65","DG_D_129","DG_D_status","DG_D_near","DG_D_distance_pct",
            "DG_W_65","DG_W_129","DG_W_status","DG_W_near","DG_W_distance_pct",
            "DG_M_65","DG_M_129","DG_M_status","DG_M_near","DG_M_distance_pct",
        ]:
            out[k] = out["symbol"].map(lambda s: feat_map.get(str(s), {}).get(k, np.nan))
        return out

    scan_out["all"] = merge_features(scan_out.get("all"))
    scan_out["new"] = merge_features(scan_out.get("new"))
    scan_out["all_new"] = merge_features(scan_out.get("all_new"))
    scan_out["holder"] = merge_features(scan_out.get("holder"))

    # Add compact reasons to NEW/HOLDER recommendations.
    for key in ("new","all_new","holder"):
        d = scan_out.get(key)
        if d is None or len(d) == 0:
            continue
        d = d.copy()
        if "reasons" in d.columns:
            def add_reason(row):
                parts = [str(row.get("reasons") or "")]
                div = str(row.get("rsi_divergence") or "NONE")
                dg = str(row.get("dg_signal") or "NONE")
                if div != "NONE":
                    parts.append(f"RSI {div}")
                if dg != "NONE":
                    parts.append(f"Gần {dg}")
                return "; ".join([p for p in parts if p and p != "nan"])
            d["reasons"] = d.apply(add_reason, axis=1)
        scan_out[key] = d

    return scan_out



def _apply_rsi_dg_recommendation_overlay(scan_out: dict) -> dict:
    """
    Add RSI/DG evidence to the existing recommendation without replacing the
    original factor + exit-risk decision.

    Labels added:
      NEW_BUY_RSI_DG  : original NEW_BUY + bullish divergence near DG.
      WATCH_BUY_DG    : bullish divergence near DG, but original action was
                        WAIT / WAIT_DIP.
      SELL_WATCH_DG   : bearish divergence near DG; review flag for a holder
                        or a new position.
      HOLD_RSI_DG     : bullish divergence near DG while holder action is HOLD.
    """
    for key in ("new", "all_new"):
        d=scan_out.get(key)
        if d is None or len(d)==0:
            continue
        d=d.copy()
        actions=[]
        for _,r in d.iterrows():
            action=str(r.get("action") or "WAIT")
            div=str(r.get("rsi_divergence") or "NONE")
            dg=str(r.get("dg_signal") or "NONE")
            bull=div in ("REG_BULL","HID_BULL") and dg!="NONE"
            bear=div in ("REG_BEAR","HID_BEAR") and dg!="NONE"
            if bull and action=="NEW_BUY":
                action="NEW_BUY_RSI_DG"
            elif bull and action in ("WAIT","WAIT_DIP"):
                action="WATCH_BUY_DG"
            elif bear and action not in ("AVOID","EXIT"):
                action="SELL_WATCH_DG"
            actions.append(action)
        d["action"]=actions
        scan_out[key]=d

    holder=scan_out.get("holder")
    if holder is not None and len(holder):
        holder=holder.copy()
        actions=[]
        for _,r in holder.iterrows():
            action=str(r.get("action") or "HOLD")
            div=str(r.get("rsi_divergence") or "NONE")
            dg=str(r.get("dg_signal") or "NONE")
            if div in ("REG_BEAR","HID_BEAR") and dg!="NONE" and action not in ("EXIT","REDUCE"):
                action="SELL_WATCH_DG"
            elif div in ("REG_BULL","HID_BULL") and dg!="NONE" and action=="HOLD":
                action="HOLD_RSI_DG"
            actions.append(action)
        holder["action"]=actions
        scan_out["holder"]=holder
    return scan_out


def scan_vn100_verbose(
    symbols,
    portfolio: dict = None,
    min_avg_val: float = 1e6,
    sleep_sec: float = 2.0,
    top_n: int = 300,
    max_retries: int = 2,
    add_rsi_dg: bool = True,
    run_top5_backtest: bool = False,
):
    """
    Enhanced wrapper around the original scanner.

    add_rsi_dg=True:
        Adds RSI divergence + DG D/W/M to Trading Decision World.

    run_top5_backtest=True:
        Immediately backtests the current Top-5 Trading Decision World.
        Tắt nếu chỉ muốn scan nhanh.
    """
    out = _original_scan_vn100_verbose(
        symbols=symbols,
        portfolio=portfolio,
        min_avg_val=min_avg_val,
        sleep_sec=sleep_sec,
        top_n=top_n,
        max_retries=max_retries,
    )

    if add_rsi_dg:
        print("\n>>> ENHANCING TRADING DECISION WORLD: RSI DIVERGENCE + DAO GĂM D/W/M")
        out = _attach_rsi_dg_to_scan(out)
        out = _apply_rsi_dg_recommendation_overlay(out)

    if run_top5_backtest:
        out["top5_backtest"] = backtest_top5_trading_decision_world(out)
    else:
        out["top5_backtest"] = None

    return out


# Override display flattening so the NEW columns appear in CSV/HTML/table output.
_original_build_scan_display_df = build_scan_display_df

def build_scan_display_df(scan_out: dict) -> pd.DataFrame:
    d = _original_build_scan_display_df(scan_out)
    if d is None or len(d) == 0:
        return d

    # Rebuild by symbol from the richer scanner rows.
    source = scan_out.get("all_new")
    if source is None or len(source) == 0:
        source = scan_out.get("new")
    if source is None or len(source) == 0:
        return d

    feature_cols = [
        "rsi","rsi_divergence","rsi_div_bull","rsi_div_bear",
        "rsi_hidden_bull","rsi_hidden_bear","rsi_div_age","rsi_div_strength",
        "rsi_div_signal_date","dg_signal","rsi_dg_context",
        "DG_D_65","DG_D_129","DG_D_status","DG_D_near","DG_D_distance_pct",
        "DG_W_65","DG_W_129","DG_W_status","DG_W_near","DG_W_distance_pct",
        "DG_M_65","DG_M_129","DG_M_status","DG_M_near","DG_M_distance_pct",
    ]
    available = [c for c in feature_cols if c in source.columns]
    if not available:
        return d

    fmap = source.drop_duplicates("symbol").set_index("symbol")
    for col in available:
        d[col] = d["Mã"].map(fmap[col].to_dict())
    return d


# Enhance HTML/reporting: if Top-5 backtest exists, append its summary to the HTML.
_original_write_html_report = write_html_report

def write_html_report(*args, **kwargs):
    path = _original_write_html_report(*args, **kwargs)
    scan_out = kwargs.get("scan_out")
    if scan_out is not None and scan_out.get("top5_backtest") is not None:
        bt = scan_out["top5_backtest"]
        summary = bt.get("summary", pd.DataFrame())
        if len(summary):
            p = Path(path)
            html = p.read_text(encoding="utf-8")
            block = (
                '<div class="card"><h2>Top-5 Trading Decision World — RSI + Dao Găm backtest</h2>'
                + summary.to_html(index=False, float_format=lambda x: f"{x:.4g}")
                + '</div>'
            )
            html = html.replace("</body></html>", block + "</body></html>")
            p.write_text(html, encoding="utf-8")
    return path


print("Trading Decision World integration ready.")


Trading Decision World integration ready.


In [75]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ENHANCEMENT 6 — ONE-CLICK ENHANCED SCAN + TOP-5 BACKTEST             ║
# ╚══════════════════════════════════════════════════════════════════════════╝

def run_enhanced_trading_decision_world(
    symbols=None,
    portfolio=None,
    sleep_sec: float = 1.6,
    run_top5_backtest: bool = True,
):
    """
    One-click workflow:
      1) scan Trading Decision World
      2) attach RSI divergence
      3) attach DG Daily/Weekly/Monthly
      4) backtest current Top-5
      5) export CSV/HTML
    """
    symbols = symbols or VN100
    out = scan_vn100_verbose(
        symbols=symbols,
        portfolio=portfolio or {},
        min_avg_val=1e6,
        sleep_sec=sleep_sec,
        top_n=300,
        add_rsi_dg=True,
        run_top5_backtest=run_top5_backtest,
    )

    print("\n>>> ENHANCED DISPLAY")
    display_scan_table(out)

    stem = f"vn100_enhanced_rsi_dg_{datetime.now():%Y%m%d_%H%M}"
    paths = export_scan_excel_csv(out, stem=stem)
    out["export_paths"] = paths

    if out.get("top5_backtest") is not None:
        summary = out["top5_backtest"].get("summary")
        if summary is not None and len(summary):
            print("\n>>> TOP-5 RSI + DAO GĂM BACKTEST SUMMARY")
            print(summary.to_string(index=False))

    print("\n✓ Enhanced Trading Decision World complete.")
    return out


print("run_enhanced_trading_decision_world() READY")
print("→ out = run_enhanced_trading_decision_world(run_top5_backtest=True)")


run_enhanced_trading_decision_world() READY
→ out = run_enhanced_trading_decision_world(run_top5_backtest=True)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  ★ ONE-CLICK: FULL VN100 SCAN + PORTFOLIO + EXPORT (verbose progress)   ║
# ║  Chạy SAU KHI đã load các cell định nghĩa (Run All phần trên)            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

MY_PORTFOLIO = {
    "TCB": 32.25,
    "NVL": 15.3,
    "GVR": 34.0,
    "MBS": 19.21,
    'AAS': 8.725,
    "VHM": 68.4,
}

SLEEP_SEC = 1.6  # rate-limit community API

_universe = list(dict.fromkeys(list(VN100) + list(MY_PORTFOLIO.keys())))
print(f"Full VN100 scan: {len(_universe)} symbols | sleep={SLEEP_SEC}s")
print(f"Portfolio: {MY_PORTFOLIO}\n")

SCAN_OUT = scan_vn100_verbose(
    symbols=_universe,
    portfolio=MY_PORTFOLIO,
    min_avg_val=1e6,
    sleep_sec=SLEEP_SEC,
    top_n=300,
    add_rsi_dg=True,
    run_top5_backtest=True,
)

print("\n>>> STYLED TABLE")
DISPLAY_DF = display_scan_table(SCAN_OUT)

print("\n>>> EXPORT CSV + HTML")
_stem = f"vn100_full_{datetime.now():%Y%m%d_%H%M}"
EXPORT_PATHS = export_scan_excel_csv(SCAN_OUT, stem=_stem)
HTML_PATH = write_html_report(
    scan_out=SCAN_OUT,
    path=str(REPORTS_DIR / f"{_stem}_dashboard.html"),
    title=f"VN100 Full Scan — {datetime.now():%Y-%m-%d %H:%M}",
)
_full = SCAN_OUT.get("all_new")
if _full is None or len(_full) == 0:
    _full = SCAN_OUT.get("new")
if _full is not None and len(_full):
    _full.to_csv(RESULTS_DIR / f"{_stem}_all_symbols.csv", index=False)
    print(f"✓ ALL symbols → {RESULTS_DIR / f'{_stem}_all_symbols.csv'} ({len(_full)} rows)")
if SCAN_OUT.get("holder") is not None and len(SCAN_OUT["holder"]):
    SCAN_OUT["holder"].to_csv(RESULTS_DIR / f"{_stem}_holder.csv", index=False)

print("\n>>> PORTFOLIO FOCUS")
if SCAN_OUT.get("holder") is not None and len(SCAN_OUT["holder"]):
    h = SCAN_OUT["holder"]
    h2 = h[h["symbol"].isin(MY_PORTFOLIO.keys())]
    if len(h2):
        cols = [c for c in ["symbol", "close", "action", "exit_risk", "trail_sl",
                            "tp1_hold", "unrealized_pnl_pct", "reasons"] if c in h2.columns]
        print(h2[cols].to_string(index=False))

PortfolioRiskEngine().analyze({k: 1.0 for k in MY_PORTFOLIO})

print("\n" + "=" * 60)
print("XONG FULL VN100")
print(f"  HTML: {HTML_PATH}")
print(f"  CSV/HTML: {EXPORT_PATHS}")
print("=" * 60)


Full VN100 scan: 243 symbols | sleep=1.6s
Portfolio: {'TCB': 32.25, 'NVL': 15.3, 'GVR': 34.0, 'MBS': 19.21, 'AAS': 8.725, 'VHM': 68.4}

✓ API key đã được lưu thành công! (API key saved successfully!)
Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
(You are using Community version - 60 requests/minute)

Để tham gia gói thành viên tài trợ (To join sponsor membership):
  Truy cập: https://vnstocks.com/insiders-program
✓ API key đã được lưu thành công! (✓ API key saved successfully! vnst***f922
✓ Bạn đang sử dụng gói Cộng đồng (✓ You are using Community Edition - 60 requests/min)
✓ vnstock API key registered (value hidden)
════════════════════════════════════════════════════════════
  VERBOSE SCAN | 243 symbols | sleep=1.6s | 2026-09-20 01:31
════════════════════════════════════════════════════════════
  Loading VNINDEX for regime...



[vnstock] Thư viện gửi số liệu đo lường tuỳ chọn (tên hàm, thời gian chạy, lỗi) để cải thiện sản phẩm.
          Không thu thập giá trị tham số bạn truyền vào, gồm mã chứng khoán và khoảng thời gian tra cứu.
          Tắt: đặt VNSTOCK_TELEMETRY=off hoặc gọi vnai.disable_telemetry()
          Chi tiết: https://vnstocks.com/onboard/chinh-sach-quyen-rieng-tu



📋 Connecting Google Drive account
to save project settings.

Mounted at /content/drive
  Market regime: BULL / normal vol / risk_on | VNI=1815.66
  (Breadth/sector: tính sau trên mã fetch OK — không block upfront)

  [1/243] FPT     ⚠  55.5  AVOID             px:71.70  DISTRIBUTION  🟢 Up  💰 Flow+  VSA:PHÂN PHỐI  TP1:+4.3%  XR:34  Technology

📋 Kết nối tài khoản Google Drive để lưu các thiết lập của dự án.
Dữ liệu phiên làm việc với Colab của bạn sẽ bị xóa nếu không lưu trữ vào Google Drive.

  [2/243] VCB     ⚠  58.1  AVOID             px:59.90  DISTRIBUTION  🟢 Up  🔄 Flow~  VSA:PHÂN PHỐI  TP1:+3.5%  XR:38  Bank
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
  [3/243] HPG     ·  54.9  WAIT              px:21.55  NONE          🔵 WeakUp  🔄 Flow~  VSA:—  TP1:+3.8%  XR:32  Materials
  [4/243] TCB    [PF]  ✅  60.0  HOLD_OR_SCALE_IN  px:31.60  NONE          🟢 Up  💰 Flow+  VSA:—  TP1:+5.5%  XR:18  Bank
  [5/243]